
# Low-SNR membrane-transporter pipeline — distilled from `zll_260130` (2.95 Å)

**v2, revised 2026-08-23 after its first generalisation test.** The notebook was run end-to-end on a
*second* dataset (P5/W23, GLUT5 transporter, 8,567 mics @ 0.73 Å/px) and produced **2.97 Å**
(`J1525`, 234,925 particles, `vis_res` CONFIRMED) against a manual baseline of **5.57 Å** on the
same data and the history bank's best on that specimen of 3.24 Å (which needed a ~10-session merge).
The pipeline works. This revision is the 14 things that made it harder than it needed to be —
see the **"What v2 changed"** section below, and `recipe.json` → `revisions` for the audit trail.

**Reference sample.** CryoSPARC project **P5**, sample **transporter**
(`particle_character: "membrane protein, symmetry: C1, sample_name: transporter"`).

**The protein's character — this is what the whole recipe is shaped around.**
A **~40–50 Å ordered TM-helix bundle sitting inside a ~105 Å detergent micelle**, i.e. the micelle
is roughly **90 % of the particle volume and carries essentially no structure**. Particle
~**100 Å**, symmetry **C1**, **very low SNR / junky** data. Input is **motion-corrected
micrographs**, not movies — so the pipeline starts at `import_micrographs`, and there is no
motion-correction or gain-flip stage.

**The reconstruction path that worked:**

> `import_micrographs` → `patch_ctf_estimation_multi` → **two independent picking branches**
> (blob over the full 0.90–1.50 · D band, Topaz trained on blob's *own* FINAL tight select), **each
> with its own 2D clean loop** → **merge + `remove_duplicate_particles`** → **one finer high-K 2D
> pass** (`max_res` 4) → **loose** RELION-class-ranker select → `homo_abinit` K = 4, C1 →
> **refine each ab-initio class ALONE and compare against the merged pool** → **un-binned full-box
> re-extract** → NU-refine at `refine_res_init` 10 → **texture-derived focused mask** →
> `new_local_refine` → validate on FSC **and** volume coverage.

**What this notebook does.** It runs that straightened path end-to-end on a *new* dataset of the
same character, without needing an agent: it owns its own job creation, queueing,
**poll-to-completion *or* queue-and-return**, error surfacing and resume-from-UID, and every
judgement call is a **code decision cell** driven by the same tested tools the reference used
(RELION `class_ranker` for 2D selection, `diagnose2d_utils` as the pre-3D gate, `vis_res_utils` for
the resolution lie-detector). The tools are **vendored, pinned copies** in `lib/`, and `controls/`
carries **vendored labelled images of a known-correct and a known-wrong outcome** so the vision
calls compare against a control rather than a prior. If a tool is missing the notebook **raises** —
it never silently skips a decision.

**What this notebook does NOT do.** It does not pin a random seed, it does not copy the reference's
paths or GPU/SSD settings (those are re-filled in cell 2 from *your* environment), it does not
**schedule** `class2D_K` or `particle_threshold` (see below), and it does not claim any resolution
that `vis_res` has not backed.

---

## What v2 changed (all 14 findings of the generalisation test)

**Two human-directed repairs.**
1. **Two blob branches → ONE blob picker over the merged band** (`0.90 · D → 1.50 · D`; 90/150 Å on
   the test dataset). The two bands overlap over 1.00–1.30 D by construction and contributed only
   **11.9 %** and **19.9 %** of the pre-3D gate feed at 35.5 % / 45.6 % survivorship, against
   Topaz's **68.2 % at 77.0 %** — the second band roughly doubled the blob arm's 2D cost for a small,
   largely redundant addition. **Breadth is still the principle**, now expressed as
   *blob-range-breadth + a complementary ML picker*. Per-branch 2D cleaning and the merge-time dedup
   are untouched.
2. **`PHYSICAL_SIGNATURE` moved into the user-parameter cell** (cell 2; [v2.2] no
   `base_datainfo.yaml` fallback — all sample information now lives in cell 2) and is injected **verbatim into every vision call** — the 2D selection guard, the
   ab-initio/3D class screen, the pre-3D gate, and `vis_res`.

**Fixed in the machinery.** `WORK_DIR` is project-scoped and the state file asserts its own scope
(#3) · `run_job` has a queue-and-return mode and a complete resume whitelist (#4) · the extract guard
predicts the **geometric** first-extract loss and flags only shortfalls **below** it (#5) · pools
over 1 M are split into concurrent subgroups under one global threshold (#6) · the `n_collapsed`
K-down-trigger needs an absolute particles-per-class corroboration (#7) · the 2D stop condition
compares only same-select-mode rounds and allows a soft floor on a big rise (#8/#9) ·
`topaz_extract particle_threshold` became an adaptive knob with its own decision function and the
raw-log-odds-vs-`ncc_score` trap written into it (#10) · the memory guard now takes a **live** free-
memory reading on the specific GPUs immediately before queueing (#11) · the pre-3D gate adds a
**UID-composition/survivorship** check because per-branch tracing cannot see through a pooled
post-merge 2D (#12) · the class screen gets the physical signature **and vendored controls**, and a
metric/vision disagreement refines **both** arms (#13) · and the deliverable check adds a
**connectivity-sustained-volume** comparison, with the surface render size raised to clear the
10 px/Å legibility floor (#14).

**Deliberately unchanged, because the test vindicated them.** The un-bin gate at ~1.3× binned
Nyquist (it deferred at 2.11×, where un-binning alone cost +0.20 Å, and fired at 1.09× for −0.37 Å;
agentM overrode it once and was wrong) · LOOP E's stop condition · `decide_k`'s rule and its
pool-origin round-1 band · the J112 finer-pass parameter block and the `class2D_force_max` /
`prepare_window_*` / unpinned-seed exclusions · the pre-3D gate itself.

---

## The two ADAPTIVE knobs — decided at run time, never scheduled

### `class2D_K`

*(human directive, 2026-08-20: “the K of 2D class should be decided based on run results, not
fixed”. Full rule + sources: `K_DECISION_SPEC.md` next to this notebook.)*

The reference happened to run K 100 → 75 → 50 per branch and K = 300 post-merge. **Copying that
sequence would reproduce the reference's dataset, not respond to yours** — a 18,259-job / 163-project
audit of the history bank found **0** occurrences of `100 → 75 → 50` in 74,623 three-step lineage
triples. So the ladder is gone. `decide_k()` runs after **every** completed `class_2D → select_2D`
round:

* **Evidence** — every round's `n_in`/`n_out`/kept fraction, the K used, per-class particle counts
  and resolutions, the `class_ranker` score histogram, and **the class-average montage** (the one
  *documented* reason to raise K — junk **inside** an otherwise-clean class — is visual).
* **The rule it applies** (it applies it; it does not invent one) — `hold` is the default. `raise`
  requires `junk_in_clean_class`, `select_fraction_high` (0.6–0.8) or `keep_rate_very_low` (< 0.2).
  **`lower` is admissible on exactly one trigger, `n_collapsed`** — and [v2] that now needs **both**
  a ≥ 8× fall since the current K was set **and** particles-per-class below ~400, because the ratio
  alone measures *shrinkage from the origin*: on the test, two branches in near-identical condition
  (1,263 vs 1,382 ppc) landed on opposite sides of the 8× bar purely because their round-1 pools
  differed by 22 %.
* **Bounds** — `K_MIN` 20, `K_CEILING` 400, a **K × box²** memory guard, and [v2] a **live per-GPU
  free-memory admission check** immediately before `.queue()`. Every clamp is printed.
* **Fallback** — HOLD, and say so. It never descends.
* The **round count** is decoupled: the loop ends only on `stop_2d_loop()`.

### `topaz_extract particle_threshold` [v2]

It is the **only** gate that limits what `topaz_extract` writes out, and the right value belongs to
the **trained model**, not the sample: on the same micrographs one model at the default `0` gave
144.5 picks/mic, another 5.2/mic, and ours would have given 271.94/mic. The default can return ~1
particle per micrograph while the job reports `completed`. So: one **diagnostic** pass with the
filter disabled (`particle_threshold = threshold = −6`), then `decide_topaz_threshold()` reads the
raw score column and picks the value that lands on the target picks/mic.

> **The sub-trap that cost a failed job.** `particle_threshold` gates on **Topaz's raw log-odds
> score** (`topaz_particles_prediction.txt`, here −6.000 … +9.826), **not** on CryoSPARC's
> `pick_stats/ncc_score` (positive, to ~158 on the same model). A threshold from the wrong column
> was a plausible number on the wrong axis, and the job "completed" with **zero** particles.
> *The score a picker reports on its output and the score its own gating parameter filters on can
> be different quantities.*

**How the LLM is called (no API account needed).** Every LLM decision — `decide_k`, the pre-3D
gate's visual read, the ab-initio class screen, the `vis_res` read — goes through one transport,
`_llm_json()` in cell 3, which shells out to the **Claude Code CLI in headless mode**
(`claude -p '<prompt>' --output-format json`, plus `--allowedTools Read` when images are attached by
absolute path). It authenticates from your existing `~/.claude/.credentials.json` — a **Claude
subscription is enough**. Cell 3 prints whether LLM decisions are **ENABLED** or whether the notebook
will run on **fallbacks**; a missing CLI never blocks the import. On the test, 11 `decide_k` calls
were all served by the CLI and none fell back.

---

## Straightened workflow

```mermaid
flowchart TD
    %% ---------- stage 0 ----------
    A["<b>import_micrographs</b><br/>psize_A, accel_kv 300, cs_mm 2.7<br/>total_dose 50, skip_header_check=False<br/><i>blob_paths = ENV</i>"]
    B["<b>patch_ctf_estimation_multi</b><br/><i>(no science knobs; compute_num_gpus = ENV)</i>"]
    A --> B

    %% ---------- stage 1: TWO picking branches ----------
    subgraph PICK["STAGE 1 — picker BREADTH, 2 independent branches. Breadth by RANGE + a complementary ML picker, never a looser threshold."]
        P1["<b>blob_picker_gpu</b> — ONE branch, MERGED band<br/>diameter 90 / diameter_max 150<br/><i>= 0.90·D / 1.50·D (v2: the union of the<br/>reference's two overlapping bands)</i>"]
        E1["<b>extract_micrographs_multi</b><br/>box_size_pix 336, bin_size_pix 160<br/><i>GUARD (v2): loss vs the GEOMETRIC prediction<br/>1−((W−box)/W)((H−box)/H); flag only BELOW it</i>"]
        P1 --> E1
    end

    B --> P1
    B -.->|"exposures → micrographs<br/>(required on EVERY extract — easy to omit)"| E1

    %% ---------- stage 2: per-branch 2D clean loops (DO NOT HOIST) ----------
    subgraph L1["LOOP A1 — blob 2D clean loop (per-branch; NEVER hoisted past the merge)"]
        SP1{{"<b>particle_sets</b> split — ONLY if n &gt; 1 M<br/><i>(v2) concurrent subgroups on DISJOINT gpus,<br/>ONE global ranker threshold, merged after</i>"}}
        C1{{"<b>class_2D_new</b> · <b>K decided at run time</b><br/>round 1 = pool-ORIGIN band: 200 default,<br/>300–400 for a pool ≥ 1.5 M, 100 for a pool ≪ 200 k<br/><i>no ladder, no schedule, no round count</i>"}}
        S1["<b>DECISION</b> · relion_rank_select<br/>+ guard_adjust(<b>loose</b> on round 1 → <b>tight</b> on the FINAL round)<br/>+ <b>PHYSICAL_SIGNATURE</b> (v2) → <b>select_2D</b>"]
        KD1{"<b>DECISION</b> · decide_k() — LLM on the round's own evidence<br/>(counts, kept fraction, selected-class fraction, per-class sizes,<br/>ranker score histogram, <b>the class-average montage</b>)<br/><b>hold</b> = default · <b>raise</b> on junk_in_clean_class /<br/>select_fraction_high / keep_rate_very_low · <b>lower</b> ONLY on<br/>n_collapsed (v2: ratio ≥ 8× <b>AND</b> ppc &lt; 400)<br/>clamped to [20, 400] + K×box² + <b>LIVE per-GPU free memory</b><br/>fallback = <b>HOLD</b>"}
        SP1 --> C1 --> S1 --> KD1
        KD1 -->|"STOP CONDITION not met<br/>(kept frac &lt; plateau AND selected-class frac not rising<br/>[v2: compared only WITHIN the same select mode]<br/>AND n ≥ MIN_PARTICLES_FOR_3D AND rounds &lt; MAX_2D_ROUNDS_SAFETY)"| SP1
    end
    subgraph L3["LOOP A2 — topaz 2D clean loop (identical machinery)"]
        C3{{"<b>class_2D_new</b> · <b>K decided at run time</b>"}}
        S3["<b>DECISION</b> · relion_rank_select + guard + signature<br/>→ <b>select_2D</b>"]
        KD3{"<b>DECISION</b> · decide_k() — same rule, same bounds<br/>fallback = HOLD"}
        C3 --> S3 --> KD3
        KD3 -->|"stop condition not met"| C3
    end

    E1 --> SP1

    %% ---------- topaz branch, bootstrapped off blob's OWN FINAL tight select ----------
    S1 -.->|"blob <b>FINAL tight</b> select<br/>(no external model, no reference)"| T1
    T1["<b>topaz_train</b><br/>par_diam 100 (=D), num_particles 300<br/>num_distribute 4 · <i>exec_path = ENV</i>"]
    TD["<b>topaz_extract</b> — DIAGNOSTIC pass (v2)<br/>threshold −6, <b>particle_threshold −6 = NO-OP</b><br/><i>writes every candidate WITH its raw score</i>"]
    TK{"<b>DECISION</b> · decide_topaz_threshold()<br/>reads the RAW log-odds column of<br/>topaz_particles_prediction.txt (NOT ncc_score)<br/>→ threshold hitting the target picks/mic<br/>bounds: inside [score_min, score_max],<br/>≥ MIN_PICKS_PER_MIC, ≤ 400/mic<br/>fallback = RAISE and ask (never the default 0)"}
    T2["<b>topaz_extract</b> — FINAL<br/>par_diam 100, num_distribute 4<br/><b>particle_threshold = decided</b><br/><i>verify delivered vs predicted (±5%)</i>"]
    E3["<b>extract_micrographs_multi</b><br/>box_size_pix 336, bin_size_pix 160"]
    B --> T1
    B -.->|"exposures → micrographs"| E3
    T1 -->|"topaz_model → model"| TD --> TK --> T2
    T2 --> E3 --> C3

    %% ---------- stage 3: merge + dedup ----------
    M1(["<b>MERGE + DEDUP</b> (2 branches)<br/><b>remove_duplicate_particles</b><br/>min_dist_A 60 (= 0.6·D)"])
    S1 -->|"FINAL tight select"| M1
    S3 -->|"FINAL tight select"| M1
    B -.->|"micrographs"| M1

    %% ---------- stage 4: the pivot ----------
    subgraph L4["LOOP B — post-merge FINER high-K 2D (the pivot). Stop when the averages are clean; ref 1 round, test 2."]
        F1["<b>class_2D_new</b> — <b>J112 params</b><br/><b>class2D_K decided at run time</b> — round 1 from the pool-ORIGIN band<br/>class2D_max_res 4 · class2D_max_res_align 4<br/>class2D_num_full_iter_batch 40<br/>class2D_num_full_iter_batchsize_per_class 200<br/><i>NO force_max, NO window radii, NO pinned seed</i><br/><i>(v2) subgroup-split over 1 M, same as LOOP A</i>"]
        F2["<b>DECISION</b> · relion_rank_select(min_score 0.2)<br/>+ guard_adjust(<b>loose</b>) → <b>select_2D</b><br/><i>ref: 259/300 classes, 92.7 % of particles</i>"]
        KD4{"<b>DECISION</b> · decide_k() — same rule, same bounds<br/>fallback = HOLD"}
        F1 --> F2 --> KD4
        KD4 -->|"stop condition not met (averages not yet clean)"| F1
    end
    M1 --> F1

    X1["<i>OPTIONAL diagnostic, NOT wired downstream</i><br/><b>reference_select_2D</b><br/>alpha_min 0.6, max_res 4, window_inner_A ≈ 0.8·box"]
    F1 -.-> X1

    %% ---------- gate ----------
    G{"<b>PRE-3D GATE</b> — diagnose2d_utils.diagnose_2d + assess_2d<br/>on the job FEEDING the ab-initio<br/>+ <b>(v2) branch_survivorship()</b>: UID intersection of the feed<br/>against each branch's final select — the per-branch answer that<br/>class-average tracing cannot give after a POOLED post-merge 2D<br/>HEALTHY / MINOR / MORE_2D_CLEANING / SERIOUS / ASK_HUMAN"}
    F2 --> G
    G -->|"MORE_2D_CLEANING"| F1
    G -->|"SERIOUS / ASK_HUMAN"| STOP["<b>HALT</b> — fix the branch, do not spend GPU"]

    %% ---------- stage 5: abinit + the branch-alone comparison ----------
    AB["<b>homo_abinit</b><br/>abinit_K 4, abinit_symmetry C1<br/>abinit_center False, abinit_init_res 20<br/>abinit_max_res 4<br/>abinit_minisize_init 300, abinit_minisize 1000<br/>abinit_minisize_epsilon 0.005"]
    G -->|"HEALTHY / MINOR"| AB

    SCR{"<b>DECISION</b> · screen the K classes<br/>PRIMARY: vis_res.compare_volumes count-robust metrics at<br/>FIXED enclosed volume (n_frag, frac_in_largest)<br/>IDENTITY (v2): <b>PHYSICAL_SIGNATURE</b> + <b>vendored controls</b><br/>(reference final map · TARGET abinit class · ARTIFACT class)<br/>discriminate on a NON-SHARED feature (isotropy, not hollowness)<br/><b>METRIC/VISION DISAGREEMENT → refine BOTH arms</b>"}
    AB --> SCR

    subgraph L5["LOOP C — BRANCH-ALONE COMPARISON (do-not-hoist: keep BOTH arms)"]
        R1["<b>nonuniform_refine_new</b> — class_k <b>ALONE</b><br/>refine_symmetry C1, refine_res_init 7<br/>default dynamic mask<br/><i>ref class_0 alone: 351,748 → <b>3.64 Å</b>; test class_3: 3.34 Å</i>"]
        MG(["<b>remove_duplicate_particles</b><br/>(union tool — particle_sets has no union)"])
        R2["<b>nonuniform_refine_new</b> — <b>MERGED</b> good classes<br/>refine_symmetry C1, refine_res_init 7<br/><i>ref 0+1+3: 1,087,189 → 6.54 Å</i>"]
        MG --> R2
    end
    SCR -->|"each candidate class (incl. the vision-best one)"| R1
    SCR -->|"all good classes"| MG

    PICKW{"<b>DECISION</b> · FSC-FIRST<br/>reported masked+corrected FSC (fsc_noisesub)<br/>arms &lt; 0.2 Å apart = TIE → count-robust real-space metrics<br/>+ <b>(v2) connectivity-sustained VOLUME</b>, never cFAR"}
    R1 --> PICKW
    R2 --> PICKW

    %% ---------- stage 6: un-bin ----------
    UNB{"<b>DECISION</b> · un-bin gate — DO NOT OVERRIDE<br/>reported res ≤ <b>1.3 ×</b> binned Nyquist?<br/><i>ref un-binned at 1.10× (−0.58 Å); test deferred at<br/>2.11× (un-binning alone cost +0.20 Å) and fired at 1.09× (−0.37 Å)</i>"}
    PICKW -->|"winner (branch alone, NOT merged upward)"| UNB
    UNB -->|"no — still ≫ Nyquist"| R1

    subgraph L6["LOOP D — un-bin / re-refine"]
        U1["<b>extract_micrographs_multi</b> — UNBINNED<br/>box_size_pix 336, <b>bin_size_pix OMITTED</b><br/>update_location / update_alignments2D / update_alignments3D = True<br/><i>GUARD: RE-extract tolerance ~1 % (already centred)</i>"]
        U2["<b>nonuniform_refine_new</b> — unbinned<br/>refine_symmetry C1, <b>refine_res_init 10</b><br/>seeded with the BINNED volume<br/><i>ref: <b>3.06 Å</b> (−0.58); test: <b>2.97 Å</b> (−0.37)</i>"]
        U1 --> U2
        U2 -->|"still within 1.3× of the new Nyquist"| U1
    end
    UNB -->|"yes"| U1

    %% ---------- stage 7: focused local refine ----------
    subgraph L7["LOOP E — focused local refinement (micelle-specific). Stop: a round must improve FSC AND n_frag/frac_in_largest AND volume coverage."]
        K1["<b>build_focus_mask.py</b> (vendored)<br/>TEXTURE-derived core mask — <b>NOT</b> a density threshold<br/>radius from the ordered-voxel radial MINIMUM (ref r = 40 Å)"]
        K2["<b>import_volumes</b><br/>volume_out_name 'mask', volume_psize = native"]
        K3["<b>volume_tools</b><br/>volume_in_type 'mask', volume_out_type 'mask'<br/><b>threshold 0.5</b> (required or it fails 'mask max != 1')<br/>dilator 3, <b>pad 5</b>"]
        K4["<b>new_local_refine</b><br/>refine_symmetry C1<br/>refine_res_init ≈ 1.5 × current res<br/>init_r_extent 10, init_s_extent 7<br/>use_alignment_prior True <i>(sigmas at DEFAULT)</i>"]
        K1 --> K2 --> K3 --> K4
        K4 -->|"improved on ALL THREE axes → tighten and repeat"| K1
    end
    U2 --> K1
    U2 -->|"particles + volume"| K4
    K3 -->|"mask"| K4

    %% ---------- validation ----------
    V{"<b>DECISION</b> · vis_res — the lie-detector, not a second FSC<br/>render_map_slices (native) + paired SURFACE at <b>≥ 10 px/Å</b> (v2)<br/>→ assess(): CONSISTENT / SUSPICIOUS_RESOLUTION /<br/>NOT_INTERPRETABLE / NEEDS_SURFACE_CHECK"}
    CV{"<b>DECISION (v2)</b> · connectivity_volume / compare_connectivity<br/>sustained enclosed volume + frags per kÅ³ at the connectivity knee<br/><i>a DIFFERENT axis from assess(): it returned CONSISTENT for a map<br/>holding 18–31 % of its expected protein volume and half its own mask</i>"}
    K4 --> V --> CV
    U2 --> CV
    CV --> OUT["<b>DELIVERABLE</b><br/>the REPORTED masked+corrected FSC,<br/>validated by vis_res AND corroborated on volume coverage"]

    OD["<i>OPTIONAL diagnostic</i> · <b>orientation_diagnostics</b><br/><b>CAVEAT (retracted verdict in the reference):</b> cFAR comes from the<br/>ASSIGNED POSES — on an unconverged low-SNR map a low cFAR can mean<br/>alignment failure, not narrow coverage. NEVER a gate here."]
    U2 -.-> OD

    classDef merge fill:#fde8c8,stroke:#c07000,stroke-width:2px
    classDef decision fill:#e6f0ff,stroke:#2255aa,stroke-width:2px
    classDef halt fill:#ffd8d8,stroke:#aa2222,stroke-width:2px
    classDef optional fill:#f0f0f0,stroke:#888,stroke-dasharray: 4 3
    class M1,MG merge
    class S1,S3,F2,KD1,KD3,KD4,SCR,PICKW,UNB,V,CV,G,TK decision
    class STOP halt
    class X1,OD optional
```

---

## Loops, merge/dedup points, and decision points

| kind | where | detail |
|---|---|---|
| **MERGE + DEDUP** | after the **2** picker branches | `remove_duplicate_particles`, `min_dist_A = 0.6·D` (ref 60 Å). Ref removed **41.5 %** across three branches; the test removed 22.4 % — the (deliberate) consequence of training Topaz on the *tight* select. |
| **MERGE** | inside a split 2D round [v2] | subgroup selections are merged with `remove_duplicate_particles` as the union tool; the split is **disjoint**, so ~0 duplicates are expected. |
| **MERGE + DEDUP** | inside the branch-alone comparison | the union tool for the good ab-initio classes (`particle_sets` has no union op). |
| **LOOP A ×2** | per-branch 2D clean | `class2D_K` **decided at run time**; select `loose` on round 1 → `tight` on the final round. **Stop (the only thing that ends the loop):** kept fraction ≥ `PLATEAU_FRAC`, **or** the selected-class fraction rose **within the same select mode** and cleared its floor (0.60, or 0.55 on a big rise), **or** the `MIN_PARTICLES_FOR_3D` floor, **or** `MAX_2D_ROUNDS_SAFETY`. Ref: 3 rounds each; test: 5. Neither is copied. **Never hoisted past the merge** — a shared post-merge clean could not have told you blob keeps 42–49 % of its picks at round 1 while Topaz keeps 85 %. |
| **LOOP B** | post-merge finer high-K 2D | Same `decide_k()`, same stop condition. Ref: 1 round at K = 300 on 1.41 M. Test: 2 rounds (K 200 loose → 300 tight, the raise triggered by `junk_in_clean_class`). |
| **LOOP C** | ab-initio **branch-alone comparison** | Refine each candidate class ALONE **and** the merged pool. [v2] a metric/vision disagreement adds an arm rather than being argued out. **Stop:** all arms refined → pick on reported FSC. |
| **LOOP D** | un-bin / re-refine | **Stop:** reported res no longer within `UNBIN_TRIGGER_FACTOR` (1.3×) of the current binned Nyquist. Ref −0.58 Å; test −0.37 Å. |
| **LOOP E** | focused local refine | **Stop:** a round fails to improve the reported FSC **or** `n_frag`/`frac_in_largest` **or** [v2] the connectivity-sustained volume. Ref: 1 round, −0.11 Å. Test: the local refine was a **negative** result despite a better headline number. |
| **DECISION** | every 2D select | `select2d_utils.relion_rank_select` + `guard_adjust`, with `PHYSICAL_SIGNATURE` on the visual guard. |
| **DECISION** | after every 2D round — `class2D_K` | `decide_k()`. **Fallback: HOLD** (printed), never descend. |
| **DECISION** | topaz `particle_threshold` [v2] | `decide_topaz_threshold()` on the diagnostic pass's **raw log-odds** column. **Fallback: raise and ask** — never the default 0. |
| **DECISION** | pre-3D gate | `diagnose2d_utils` — **hard gate** — plus `branch_survivorship()`. |
| **DECISION** | ab-initio class screen | `compare_volumes` metrics (primary) + signature + vendored controls (identity); disagreement → both arms. |
| **DECISION** | pick the winning arm | FSC-first on the **reported masked+corrected** FSC; `< 0.2 Å` = tie → real-space metrics **and volume coverage**. |
| **DECISION** | final validation | `vis_res_utils.assess` on slices **and** the paired surface at ≥ 10 px/Å, **and** `compare_connectivity`. |

## Seven things that actually mattered (carry these, not just the numbers)

1. **Picker breadth, not looser thresholds** — [v2] breadth by *range* plus a complementary ML
   picker, each branch with its own full 2D loop, then merge + dedup.
2. **Selection tightness staged loose → tight.**
3. **One pass of finer, resolution-capped, high-K 2D beat many ordinary passes** —
   `max_res`/`max_res_align = 4`, 40 full iters, batchsize 200/class, at a **high K for the size of
   the merged pool**. This step is entirely absent from the manual run the test beat by 2.60 Å.
4. **Select LOOSELY at that stage** — 92.7 %, not 81.3 %.
5. **Take the best ab-initio class ALONE; do not merge visually similar classes.** Merging
   decorrelated the half-maps (merged FSC fell 0.77 @ 7 Å → 0.02 @ 6 Å — not natural decay).
6. **Un-bin when the reported resolution approaches the binned Nyquist** (~1.3×), and seed the
   un-binned refine from the binned volume at `refine_res_init 10`.
7. **A focused mask on the ordered core helps a micelle-dominated particle — modestly, and not
   always.** It must be **texture-derived**, and [v2] it must be judged on volume coverage as well
   as on the masked FSC.

## Six ways the numbers lied — do not repeat these

1. **cFAR is computed from the assigned poses.** On an unconverged low-SNR map a low cFAR can mean
   *alignment failure*, not narrow coverage. Never a gate here.
2. **Use CryoSPARC's reported masked+corrected FSC (`fsc_noisesub`) as the first reference** — for
   local refinement too. A hand-rolled mask-free FSC is biased *against* a focused refine.
3. **`vis_res` is the lie-detector, not a second FSC** — and [v2] it tests **legibility at a point**,
   not **volume coverage**. It returned `CONSISTENT` for a map sustaining 18–31 % of its expected
   protein volume.
4. **Never rank two maps on a difference below ~0.2 Å.**
5. **"How many helices separate at a global contour" is not a resolution criterion** in a
   micelle-dominated map.
6. **[v2] The score a picker reports on its output is not always the score its own gating parameter
   filters on** (`ncc_score` vs Topaz raw log-odds) — and a mismatched threshold "completes" with
   zero particles.

---

## Parameters you fill in — see cell 2

Every parameter in cell 2 carries its **reference value and unit** as a comment, plus the
derivation where it is dimensional. Four groups:

* **the physical signature** — one paragraph describing YOUR specimen's density. Not a comment: it
  is a scientific input, and it feeds every vision call.
* **sample / optics** (`PIXEL_SIZE_A`, `PARTICLE_SIZE_A`, `SYMMETRY`, …) — re-derive, don't copy.
  The test derived box 336 / bin 160 at 0.73 Å/px (reference: 300/150 at 0.824), and the pre-3D
  gate then *measured* the particle and confirmed it.
* **environment** (`MICROGRAPHS_GLOB`, lane/host/GPUs, SSD, tool paths) — **never** copied from the
  reference; these crash a job if stale.
* **science knobs carried verbatim from the reference** — change one only with proof of improvement
  in a comparable situation.

In [ ]:
import os
# ============================================================================
# CELL 2 — PARAMETERS.  Fill these in for the NEW dataset.
# Every value shows the zll_260130 REFERENCE value + unit. Nothing here is copied
# blindly: dimensional knobs are DERIVED (see cell 3 `derive_dimensions`),
# environment fields are re-filled from YOUR environment, and the ADAPTIVE knobs
# (class2D_K, topaz particle_threshold) are NOT here at all — they are DECIDED AT
# RUN TIME (see `decide_k` / `decide_topaz_threshold` in cell 3 and
# K_DECISION_SPEC.md); only their hard BOUNDS live here.
#
# REVISED 2026-08-23 after the first generalisation test (P5/W23, GLUT5, 2.97 A).
# New/changed parameters are marked  [v2].
# ============================================================================
import shutil
from pathlib import Path

# ---- 0. where this notebook lives and where its tools come from ------------
# The notebook runs on the EM-Claw CryoSPARC profile's own helpers (2d_select/, vis_res/,
# build_model_mask/, cs_connect.py), so it uses the same tested deciders as the agents.
# Order: PIPELINE_DIR env var -> the dir holding this notebook (Jupyter sets cwd there).
# Run it in a Python with cryosparc-tools 4.7.1, numpy 1.26.4, scipy, mrcfile, matplotlib,
# pillow and pyyaml, and with the CRYOSPARC_* variables of profiles/cryosparc/.env exported
# (for example: `set -a; . profiles/cryosparc/.env; set +a` before starting the kernel).
PIPELINE_DIR = Path(os.environ.get('PIPELINE_DIR') or Path.cwd()).resolve()
PROFILE_DIR  = PIPELINE_DIR.parent.parent    # profiles/cryosparc
assert (PROFILE_DIR / '2d_select' / 'select2d_utils.py').exists(), (
    f'PIPELINE_DIR={PIPELINE_DIR} is not profiles/cryosparc/pipelines/<name> — set the '
    'PIPELINE_DIR env var or run this notebook from its own directory.')
LIB_DIRS     = [PROFILE_DIR, PROFILE_DIR / '2d_select', PROFILE_DIR / 'vis_res',
                PROFILE_DIR / 'build_model_mask']
CONTROLS_DIR = PIPELINE_DIR / 'controls'     # [v2] optional POSITIVE/NEGATIVE visual controls
# WORK_DIR is [v2] PROJECT-SCOPED and is built at the BOTTOM of this cell, after
# PROJECT_UID / WORKSPACE_TITLE / DATASET_TAG are known. Finding #3: a shared
# PIPELINE_DIR/'runs' let a second dataset silently resume into the first one's state.

# ---- 1. SAMPLE INFORMATION — inline, NO external base_datainfo.yaml ----------
# [v2.2 2026-10-09] This notebook no longer reads a project base_datainfo.yaml. Every sample
# fact the vendored deciders used to pull from that file (optics, particle size, particle
# character, symmetry, physical signature, ChimeraX path) is set IN THIS CELL and collected into
# SAMPLE_INFO at the bottom of the cell. select2d_utils / diagnose2d_utils / vis_res_utils still
# take a *path*, so cell 3 serialises SAMPLE_INFO into WORK_DIR/sample_info.generated.yaml on
# every run — an OUTPUT of this cell, overwritten each time, never an input to edit.
SAMPLE_NAME        = 'transporter'     # ref sample name (used only in particle_character below)
PARTICLE_CHARACTER = 'membrane protein in detergent micelle, symmetry: C1, sample_name: transporter'
                                       # free text read by the pre-3D gate (diagnose2d). ref P5:
                                       # "membrane protein, symmetry: C1, sample_name: transporter"

# ---- 1b. [v2] THE PHYSICAL SIGNATURE — what YOUR specimen's density looks like
# =============================================================================
# THIS IS A SCIENTIFIC INPUT, NOT A COMMENT. It is passed VERBATIM into EVERY vision
# judgement the notebook makes: the 2D selection visual guard, the ab-initio / 3D class
# screen, the pre-3D gate's visual read, and vis_res.
#
# WHY IT IS HERE (finding #13 of the generalisation test): the notebook told the model HOW to
# screen but never WHAT this specimen should look like. With no physical description available,
# the screen invented an accept/reject rule from a LINGUISTIC association — "transporter ->
# channel -> hollow" — rejected the real target (the largest class, and the best on BOTH
# count-robust metrics), ran the whole 3D stage on an ab-initio spherical-shell artifact that
# plateaued at 6.46 A, and additionally used the real target as a junk-sink decoy that drained
# 103,807 genuine particles. The human's correction recovered 3.34 A -> 2.97 A. ONE PARAGRAPH
# HERE WOULD HAVE PREVENTED ALL OF IT.
#
# WRITE FOUR THINGS:
#   (a) what the target is MADE OF and at what SCALE (sizes in A);
#   (b) what a CORRECT map / class average shows;
#   (c) what the known ARTIFACT looks like;
#   (d) THE DISCRIMINATOR — a feature that is NOT SHARED between (b) and (c). A shared feature
#       ("hollow vs filled") cannot decide anything; a directional one ("isotropy") can.
#
# The text below is the WORKED EXAMPLE from the test dataset (a micelle-embedded transporter),
# kept as the default. REPLACE IT for a different specimen — do not leave someone else's
# signature in place. If it is left as '' the notebook prints a loud warning and every vision prompt carries
# "physical_signature NOT SET — the criteria below are the model's own inference".
PHYSICAL_SIGNATURE = None

# [v2] Vendored labelled controls attached to the class screen and the final read.
# They are images of a KNOWN-CORRECT and a KNOWN-WRONG outcome on this sample class, so the
# model compares against a control instead of against its own prior. See controls/README.md.
CONTROL_FINAL_MAP     = CONTROLS_DIR / 'reference_final_map_J1077_2.95A_slices.png'
CONTROL_ABINIT_TARGET = CONTROLS_DIR / 'abinit_TARGET_J1458_class_3.png'
CONTROL_ABINIT_ARTIFACT = CONTROLS_DIR / 'abinit_ARTIFACT_J1458_class_0.png'
USE_VISUAL_CONTROLS = True     # set False ONLY if your specimen is a different sample class
                               # (then REPLACE the controls, do not just switch them off)

# ---- 2. sample + optics (the NEW sample; reference values shown) ------------
MICROGRAPHS_GLOB = 'YOUR_DATA_PATH/MotionCorr/job002/Movies/*.mrc'          # ENV FIELD, REQUIRED. Motion-corrected micrographs.
PIXEL_SIZE_A     = 0.824       # A/pix          (ref 0.824; test dataset 0.73)
ACCEL_KV         = 300.0       # kV             (ref 300)
CS_MM            = 2.7         # mm             (ref 2.7)
TOTAL_DOSE       = 50.0        # e-/A^2         (ref 50)
PARTICLE_SIZE_A  = 100.0       # A, longest dimension of the WHOLE particle incl. micelle (ref 100)
SYMMETRY         = 'C1'        # ref C1. Unknown symmetry -> stay C1 (memory: symmetry_handling_protocol)
EXPECTED_CORE_MASS_KDA = 60.0  # kDa of the ORDERED core ONLY (exclude the micelle).
                               # ref 53 kDa: a sphere r=25 A = 65,000 A^3 at 0.81 Da/A^3.
                               # Used for the fixed-enclosed-volume surface render, the mask
                               # sanity check, and [v2] the connectivity-volume check.
MIC_SHAPE_PX     = None        # [v2] (H, W) of a micrograph in px. None = read it LIVE off the
                               # CTF job. Used ONLY by the extract guard, to predict the
                               # geometric first-extract loss. (test: 4096 x 4096)

# ---- 3. CryoSPARC placement + scheduler (ENV — never copied) ----------------
PROJECT_UID     = 'P5'                     # existing project
WORKSPACE_TITLE = 'transporter_pipeline'   # a NEW workspace is created if this title is absent
DATASET_TAG     = ''                       # [v2] optional extra scope for WORK_DIR / the state
                                           #   file, e.g. 'glut5_0p73'. Leave '' to scope by
                                           #   PROJECT_UID + WORKSPACE_TITLE alone.
LANE            = 'YOUR_CRYOSPARC_LANE'    # CryoSPARC scheduler lane
HOSTNAME        = None                     # pin a host so a busy GPU can be dodged; None = lane decides
GPUS_MULTI      = [0, 1, 2, 3]             # for multi-GPU jobs (2D, extract, CTF)
GPUS_SINGLE     = [0]                      # refines are single-GPU (nonuniform_refine_new has NO
                                           #   compute_num_gpus param)
COMPUTE_USE_SSD = True                     # ENV. The reference forced False ONLY because the SHARED
                                           #   /gpu_temp cache was 98-100% full (it FAILED J89/J98/
                                           #   J113/J581 with cache exhaustion). Default here is True
                                           #   (memory: user_prefers_ssd_cache — GPUs at 0% util are
                                           #   I/O starved). Flip to False if the cache is full.
POLL_SECONDS    = 60                       # job poll interval
JOB_TIMEOUT_H   = 96                       # give up (and SAY SO) after this many hours

# ---- 3b. [v2] BLOCKING vs QUEUE-AND-RETURN (finding #4) ---------------------
# run_job() used to always poll to completion — up to JOB_TIMEOUT_H = 96 h. That is right for a
# human at a notebook and unusable from an agent turn, so subC had to bolt its own
# `run_job_queue_only` on (and its hand-copied resume whitelist then created a duplicate job,
# J1431/J1432, two jobs on the same GPUs). Queue-and-return is now a first-class mode.
#   True  = classic behaviour: create -> queue -> POLL to completion (a human at a notebook).
#   False = create -> queue -> confirm it launched -> RETURN the job object immediately.
#           Re-execute the same cell later: RESUME picks the job back up (the resume whitelist
#           covers EVERY non-terminal status, including 'started' and 'waiting'), so a re-run
#           never duplicates. Downstream cells will refuse to use a job that is not completed.
WAIT_FOR_JOBS   = True
QUEUE_CONFIRM_S = 20           # seconds to wait before reading back the status when not polling

# ---- 4. external tool paths (ENV FIELDS — re-filled, never copied) ----------
TOPAZ_EXE           = os.environ.get('EMCLAW_TOPAZ_EXE', 'YOUR_TOPAZ_PATH')
RELION_CLASS_RANKER = str(PROFILE_DIR / '2d_select' / 'relion5' / 'relion_class_ranker')  # profile launcher
RELION_PYTHON       = os.environ.get('EMCLAW_RELION_PYTHON', 'YOUR_RELION5_PYTHON_PATH')
CHIMERAX_EXE        = os.environ.get('EMCLAW_CHIMERAX', shutil.which('chimerax') or '')   # [v2.2] was `model_mask_chimerax` in
                                            #   base_datainfo.yaml. Needed for the vis_res SURFACE look.
CHIMERAX_RENDER_HOST    = os.environ.get('VIS_RES_RENDER_HOST')       # None = render locally
CHIMERAX_RENDER_LD_PATH = os.environ.get('VIS_RES_RENDER_LD_PATH', '')
SSH_CMD             = 'ssh'                 # [v2] used ONLY to read live GPU memory on a lane
# ---- 4b. the LLM decision transport: the CLAUDE CODE CLI in headless mode ----
# ONE transport for every LLM decision in this notebook (decide_k, the pre-3D gate's visual read,
# the ab-initio class screen, the vis_res read) -> `_llm_json()` in cell 3.
#   claude -p '<prompt>' --output-format json           (+ --allowedTools Read for images)
# It authenticates from ~/.claude/.credentials.json, so a CLAUDE SUBSCRIPTION IS ENOUGH: no
# ANTHROPIC_API_KEY, no API account. Images are attached by ABSOLUTE path inside the prompt
# (a relative path is NOT resolved from the CWD). Verified on this server: claude 2.1.236,
# a `claude` CLI on PATH, vision read of a real map-slice PNG OK.
# COST: every call spends your subscription usage (~$0.14-equivalent per short call in that test,
# mostly prompt-caching overhead). decide_k() fires once per completed 2D round.
CLAUDE_CLI          = os.environ.get('EMCLAW_CLAUDE_BIN') or shutil.which('claude') or ''
CLAUDE_CLI_TIMEOUT_S = 300       # per call; a timeout falls back, it never hangs the notebook

# ---- 5. RESUME ---------------------------------------------------------------
# The notebook records every job UID in WORK_DIR/pipeline_state.json and REUSES a completed job
# instead of re-running it. To resume a half-finished run, just re-execute the cells in order.
# To force a step to re-run, delete its key from the state file (or call forget('<key>')).
# [v2] The state file also records the project / workspace / micrographs it was written for, and
# assert_state_scope() REFUSES to resume a state written for a different dataset (finding #3).
RESUME = True

# ============================================================================
# SCIENCE KNOBS — carried verbatim from the reference unless marked DERIVED
# ============================================================================

# ---- picking (DERIVED from PARTICLE_SIZE_A; reference values in the comment) --
# [v2] ONE blob branch, spanning the UNION of the reference's two bands (human directive
# 2026-08-23). The reference ran band 1 (1.00, 1.50) and band 2 (0.90, 1.30) as two independent
# branches, each with a full 2D clean loop. On the generalisation test those two branches
# contributed only 11.9% and 19.9% of the pre-3D gate feed, at 35.5% / 45.6% survivorship through
# the post-merge pass, while the single topaz branch contributed 68.2% at 77.0% — i.e. the second
# blob band roughly DOUBLED the 2D cost of the blob arm (5 rounds, ~3h50m for round 1 alone on
# ~3M particles) for a small and largely redundant addition; the bands overlap over 1.00-1.30 D
# by construction. BREADTH IS STILL THE PRINCIPLE — it is now blob-RANGE-breadth plus a
# complementary ML picker, rather than two overlapping blob bands.
BLOB_BAND_FRAC = (0.90, 1.50)    # -> (90, 150) A at D=100.  ref J84 (100/150) U J85 (90/130)
TOPAZ_PAR_DIAM_FRAC = 1.00       # -> 100 A.                  ref J96/J100
TOPAZ_NUM_PARTICLES = 300        # ref J96 (expected picks per micrograph for training)
TOPAZ_NUM_DISTRIBUTE = 4         # ref J96/J100
TOPAZ_TRAIN_ON_ROUND1_LOOSE = False  # HUMAN DIRECTIVE 2026-08-21: train Topaz on the blob branch's
#   FINAL (tight) select, NOT its round-1 loose select. The reference did the latter (its J96
#   trained on J91, the r1 loose set) and subP copied it faithfully — this is a deliberate
#   IMPROVEMENT on the precedent, not a distillation fix. A loose select is ~50% junk on this
#   sample class, and Topaz learns whatever positives it is given, so a dirty training set
#   teaches the picker to reproduce the contamination. The final tight select is small but
#   clean, which is the trade Topaz actually wants. [v2] The test CONFIRMED this on three
#   independent measurements: topaz round-1 keep rate 85.0% (vs blob 41.9-49.0%), merge overlap
#   22.4% (vs the reference's 41.5%), and LOOP-B survivorship 77.0% (vs blob 35.5-45.6%).
MIN_PICKS_PER_MIC = 10           # health floor. ref blob 729/881 per mic, topaz 190/mic.

# ---- [v2] topaz particle_threshold: ADAPTIVE KNOB — bounds + target only -----
# FINDING #10. `particle_threshold` is the ONLY gate that limits what topaz_extract writes out,
# and the right value is a property of the TRAINED MODEL, not of the sample: on the SAME
# micrographs, W2's J63 at the default 0 gave 144.5 picks/mic while our model at the default 0
# would have given 271.94/mic, and another model on the same project gave 5.2/mic — all
# nominally "default". Worse, the default can return ~1 particle per micrograph while the job
# still reports "completed". So it is decided at run time by decide_topaz_threshold().
#
# THE SUB-TRAP THAT COST A FAILED JOB (J1423): particle_threshold gates on TOPAZ'S RAW LOG-ODDS
# SCORE (in <job_dir>/topaz_particles_prediction.txt; here -6.000 ... +9.826), NOT on CryoSPARC's
# pick_stats/ncc_score (positive, to ~158 on the same model). A threshold derived from the wrong
# column (78.4) was a plausible mid-range number on the wrong axis, and the job "completed" with
# ZERO particles. The decision function reads the RAW file and refuses anything outside its range.
TOPAZ_PARTICLE_THRESHOLD  = None   # None = DECIDE IT (runs one cheap diagnostic pass first).
                                   # Set a float ONLY if you already know this model's scale.
TOPAZ_DIAG_FLOOR          = -6     # the candidate floor `threshold`; the diagnostic pass sets
                                   # particle_threshold to the SAME value = a no-op filter, so
                                   # every NMS candidate is written out WITH ITS SCORE.
TOPAZ_TARGET_PICKS_PER_MIC = None  # None = anchor on the blob branch's own picks/mic on THIS
                                   # dataset (clamped into the bounds below). ref topaz 190/mic;
                                   # test chose 1.843 -> 144.5/mic (predicted 1,238,156, delivered
                                   # 1,238,405, +0.02%). Both are printed as comparisons only.
TOPAZ_MAX_PICKS_PER_MIC   = 400    # hard bound: above this the picker is proposing noise

# ---- boxes (DERIVED; see derive_dimensions) ---------------------------------
BOX_TARGET_MULTIPLE = 2.5        # box ~ 2.5 x particle. ref 300 px x 0.824 = 247.2 A = 2.47 x 100 A
                                 # [v2] test: 336 px x 0.73 = 245.3 A = 2.45x — and the pre-3D gate
                                 # MEASURED the particle at 65.7-111.1 A in that box (2.21x, flat
                                 # radial profile by ~55-60 A), the first time the DATA confirmed
                                 # this derivation instead of arithmetic.
BIN_FACTOR          = 2          # bin 2x for ALL of 2D and the FIRST 3D. ref 300 -> 150

# ---- [v2] extract guards (finding #5) ---------------------------------------
# A FIRST extract from raw picks loses a large, BOX-DEPENDENT fraction at the frame edge:
# 1 - ((W-box)/W)*((H-box)/H). On the test (box 336, 4096x4096) that predicts 15.73%, and the two
# blob first-extracts lost 15.96% / 15.72% — within 0.3% of geometry. A flat 5% tolerance
# false-alarms on every large-box first extract, and the noise trains you to wave through the
# real thing. But the prediction is an UPPER BOUND and is PICKER-DEPENDENT: Topaz self-excludes
# an edge margin (its coordinates spanned 110-3960 in a 4096 frame) and kept 99.47%.
# GUARD RULE: flag only a count BELOW the prediction by more than EXTRACT_SHORTFALL_TOL.
# A yield ABOVE the prediction is not a fault — silent truncation loses particles, never adds them.
EXTRACT_SHORTFALL_TOL = 0.01     # 1% below the geometric prediction (FIRST extract)
REEXTRACT_TOL         = 0.02     # a RE-extract of already-centred particles: the tight tolerance
                                 # (ref measured 0.80%; test 0.7-0.8%)

# ---- 2D clean loops (LOOP A per branch, LOOP B post-merge) -------------------
# THE STOP CONDITION — and ONLY the stop condition — decides how many rounds run.
PLATEAU_FRAC      = 0.85                 # stop once a round's KEPT FRACTION reaches this
CLASS_FRAC_CLEAN_MIN = 0.60              # ...or once the SELECTED-CLASS fraction (kept classes / K)
CLASS_FRAC_RISE_EPS  = 0.02              #    RISES round-over-round by > EPS and is >= the floor.
                                         #    Bank signal for "the branch is clean, stop cleaning":
                                         #    GLUT5 ran 93/200 -> 147/200 -> 145/200 at CONSTANT K.
                                         #    A rising select ratio is NEVER a reason to shrink K.
CLASS_FRAC_CLEAN_MIN_SOFT = 0.55         # [v2] finding #8: blob1's round 3 came in at 0.570 with a
CLASS_FRAC_BIG_RISE       = 0.15         #    +0.257 rise — 3 points short of the uniform 0.60 floor,
                                         #    so it ran another round for nothing. A BIG rise
                                         #    (> CLASS_FRAC_BIG_RISE) qualifies at the SOFT floor.
                                         #    blob2 (0.477) is under either floor, so nothing changes
                                         #    for a branch that is genuinely still dirty.
# [v2] finding #9: class_frac is SELECT-MODE DEPENDENT (loose / none / tight apply different cuts),
# so a tight round is NOT comparable to a loose one — round 5's drop to 0.460 / 0.545 on the test
# was an artefact of the stricter tight rule, not a quality regression. stop_2d_loop() therefore
# compares a round only against the most recent round run in the SAME mode.
MAX_2D_ROUNDS_SAFETY = 8                 # RUNAWAY GUARD ONLY — not the reference count (which was 3;
                                         # the test's data-driven stop took 5 on both blob branches)
MIN_PARTICLES_FOR_3D = 100_000           # floor; ref carried 1.30M into abinit and 352k into the
                                         # winning refine (memory: 3d_purification_is_the_engine —
                                         # ~100k clean particles is plenty)
RANKER_MIN_SCORE  = 0.2                  # ref J595 (LOOSE). guard_adjust loose=0.15 / tight=0.25

# ---- [v2] >1M subgroup splitting (finding #6) --------------------------------
# Project convention (memory: split_large_2d_into_subgroups): any 2D stack over ~1M particles is
# SPLIT into subgroups that classify concurrently on free GPUs, under ONE global ranker threshold,
# with the selections merged afterwards (no dedup — the split is disjoint by construction). The
# distilled pipeline had no size-aware logic except the memory guard, so the test ran ~3M as a
# single job per branch: round 1 alone took ~3h50m.
SPLIT_2D_OVER_N   = 1_000_000    # split when the round's input exceeds this
SPLIT_2D_MAX_JOBS = 4            # at most this many concurrent class_2D jobs per round
SPLIT_2D_TARGET_N = 900_000      # aim for ~this many particles per subgroup

# ---- class2D_K: ADAPTIVE KNOB — bounds only, never a value or a ladder -------
# There is deliberately no K_LADDER here. `decide_k()` (cell 3) chooses K after every completed
# round from that round's own evidence; the reference's K 100/75/50 (per branch) and K 300
# (post-merge) survive only as printed COMPARISON numbers. Rule + sources: K_DECISION_SPEC.md.
K_MIN            = 20            # below this CryoSPARC silently disables Force-max = a DIFFERENT
                                 # algorithm. Hard floor.
K_CEILING        = 400           # iteration time balloons above it and OOM is reported at K=400
                                 # even on 24 GB cards. The notebook RAISES rather than queueing.
# Round 1 has no previous round, so K comes from the pool's ORIGIN (fresh picks, or the round right
# after a merge/re-pick + dedup) — NOT from a particles-per-class target. Bank: median input for
# K >= 200 is 918k with a median of 1 upstream 2D round; K=400 sits at median input 1.65M. Guide:
# "50-200 classes" for a pool in the hundreds of thousands. The intersection:
K_ROUND1_DEFAULT = 200           # 43% of 20,029 bank jobs; the default for a fresh pool
K_ROUND1_SMALL   = 100           # pool well under K_POOL_SMALL_N (the other 38% of the bank)
K_ROUND1_LARGE   = 300           # pool at/above K_POOL_LARGE_N
K_ROUND1_HUGE    = 400           # top of the band — taken ONLY when K_ROUND1_LARGE would still
                                 # leave more than PPC_CEIL_FIRST_PASS particles per class
                                 # (== K_CEILING; K=400's median bank input is 1.65M)
K_POOL_SMALL_N   = 200_000
K_POOL_LARGE_N   = 1_500_000
K_BIMODAL_PRIOR  = (100, 200)    # 81% of all bank jobs. A proposal near neither, without an
                                 # explicit trigger, is PRINTED as deserving a second look.
# The single admissible reason to LOWER K (bank's best map, 1.97 A: 369k -> 39k, K 200 -> 100):
N_COLLAPSE_FACTOR = 8.0          # "roughly an order of magnitude" — the measured case was 9.5x
# [v2] finding #7: the ratio alone measures SHRINKAGE-FROM-ORIGIN, not current state. On the test,
# blob1 (7.77x -> HOLD) and blob2 (8.67x -> LOWER) straddled this bar while their pools were in
# nearly identical condition (378,983 vs 414,505 particles; 1,263 vs 1,382 particles/class at
# K=300) — the split came from a 22% difference in their ROUND-1 ORIGIN pools, an artefact of the
# different picker bands. The down-trigger now needs an ABSOLUTE corroboration too: the branch must
# ALSO have fallen below the particles-per-class floor where the one measured precedent actually
# cut K (369k -> 39k at K=200 is ppc 195).
N_COLLAPSE_PPC_MAX = 400         # == PPC_FLOOR_HIGH; both conditions must hold, and both print
# Up-triggers with a numeric precondition the caller VERIFIES before accepting them:
SELECT_FRAC_HIGH_BAND = (0.60, 0.80)   # selected classes / K -> precedents raised 3x more often
                                       # than they lowered (30% vs 10%)
KEEP_RATE_VERY_LOW    = 0.20           # particle keep-rate below this also skews up (31% vs 17%)
# Soft guard-rails: PRINTED, never enforced silently (particles per class = n_in / K)
PPC_FLOOR_LOW    = 200           # below ~200-400 ppc the branch is OVER-cleaned: stop or re-merge,
PPC_FLOOR_HIGH   = 400           #   do NOT just cut K
PPC_CEIL_FIRST_PASS = 25_000     # round-1 p90; above it K is too low for the pool
# Hard memory guard, checked BEFORE queueing (memory: small_card_2d_oom).
KBOX2_MAX_SMALL_CARD = 7_000_000   # K x box_px^2. MEASURED: K=164 @ box 208 = 7.10e6 OOM'd on the
                                   # 11-12 GB cards; the reference's K=300 @ box 150 = 6.75e6 ran.
SMALL_CARD_LANES = ()                     # lanes whose GPUs have only 11-12 GB, e.g. ('lane_a',)
BIG_K_LANE       = 'YOUR_BIG_GPU_LANE'    # lane with large GPUs (e.g. A40) — where big-K 2D belongs

# ---- [v2] LIVE GPU admission control (finding #11) --------------------------
# The K x box^2 guard models the CARD CLASS. The binding constraint is the FREE MEMORY ON THE
# GPUs ACTUALLY ASSIGNED: J1437 (K=250 @ box 160 = 6.40e6, UNDER the small-card ceiling and under
# a K x box^2 that had run fine) died with CUDA_ERROR_OUT_OF_MEMORY because a small-card node gave it GPU0/1,
# which carried ~8 GB of foreign load the CryoSPARC scheduler cannot see. The identical job at the
# identical K then succeeded on an A40 node. So the memory check is now taken LIVE, on the specific
# candidate ids, IMMEDIATELY BEFORE .queue() — foreign load flips inside a single turn.
MIB_PER_KBOX2_UNIT = 11_000.0 / 7.0e6   # MiB per unit of K x box^2, factored out of the measured
                                        # small-card ceiling (7.0e6 filled an ~11 GB card)
GPU_FREE_MARGIN_MIB = 6_000             # required headroom on top of the estimate (the retry that
                                        # worked used a flat 6 GB floor)
GPU_UTIL_WARN_PCT   = 50                # foreign utilisation above this is printed as a
                                        # wall-clock warning: an A40 GPU cleared the memory floor
                                        # at 100% foreign util and cost ~65% extra wall-clock
                                        # (2h11m vs 1h19m for the same job)
GPU_ADMISSION_REROUTE = True            # pick the freest ids on the lane instead of refusing

# ---- merge + dedup ----------------------------------------------------------
DEDUP_MIN_DIST_FRAC = 0.60       # -> 60 A at D=100.  ref J111 min_dist_A=60

# ---- post-merge FINER high-K 2D (LOOP B) — the pivot -------------------------
# HUMAN DIRECTIVE: these are J112's parameters, NOT J582's.
#   DROPPED from J582: class2D_force_max=False  (was set in BOTH failing and winning runs here, so
#     it explains nothing, costs several-fold runtime, and smears off-centre features)
#   DROPPED from J582: prepare_window_inner_radius/outer_radius 0.8/0.9 (no situation-matched
#     evidence; carried forward untested from a branch that contributed zero particles)
#   NEVER SET: random_seed — left to CryoSPARC's per-job generation.
#   NOT HERE: class2D_K. It is the ADAPTIVE knob — decided at run time by decide_k(), round 1 from
#     the pool-ORIGIN band above (the merge+dedup makes this a newly-enlarged pool). The reference's
#     K=300 on 1,407,630 particles is a RECORDED COMPARISON for a pool of that size, not a constant.
# [v2] This whole block was VINDICATED by the generalisation test and is unchanged.
FINER_2D_PARAMS = {
    'class2D_max_res': 4,
    'class2D_max_res_align': 4,
    'class2D_num_full_iter_batch': 40,
    'class2D_num_full_iter_batchsize_per_class': 200,
}
RUN_REFERENCE_SELECT_2D_CROSSCHECK = True   # optional, NOT wired downstream (ref J594)
REFSEL_PARAMS = {'alpha_min': 0.6, 'max_res': 4}   # window_inner_A is DERIVED (see cell 3).
                                                   # It MUST be set explicitly or the live job
                                                   # fails, even though MCP shows default=null.

# ---- ab-initio ---------------------------------------------------------------
ABINIT_PARAMS = {
    'abinit_K': 4,
    'abinit_center': False,
    'abinit_init_res': 20,
    'abinit_max_res': 4,          # NOT the coarse 8 of the abandoned early attempt
    'abinit_minisize_init': 300,
    'abinit_minisize': 1000,
    'abinit_minisize_epsilon': 0.005,
}

# ---- refines -----------------------------------------------------------------
NU_RES_INIT_BINNED   = 7      # ref J1032 / J1026
NU_RES_INIT_UNBINNED = 10     # ref J1072 — seeded from the BINNED volume
UNBIN_TRIGGER_FACTOR = 1.3    # un-bin when reported res <= 1.3 x binned Nyquist.
                              # ref un-binned at 1.10x (paid -0.58 A); correctly DEFERRED at 2.0x.
                              # [v2] VINDICATED TWICE on the test: it deferred at 2.11x (where
                              # un-binning alone cost +0.20 A) and fired at 1.09x for -0.37 A.
                              # DO NOT override it — agentM did once, and was wrong.
FSC_TIE_A            = 0.2    # two maps closer than this are a TIE — decide on real-space metrics
CLASS_SCREEN_KEEP_FRAC_IN_LARGEST = 0.80   # a class below this is junk-like; ref class_2 = 0.739

# ---- [v2] the connectivity-sustained-volume check (finding #14) --------------
# vis_res.assess() tests LEGIBILITY AT A POINT. It returned CONSISTENT for J1530 (2.81 A reported),
# a map that sustained only 23k/13k A^3 at the connectivity knee against J1525's 119k/68k A^3 —
# 18-31% of the expected protein volume, LESS THAN HALF of its own focus-mask footprint, with
# 4-13x more fragments per unit volume and frac_in_largest 0.427 vs 0.556. Legibility-at-a-point
# and VOLUME COVERAGE are different axes, and only the second caught it. This check is numeric
# (VR.auto_display_levels on the unsharp map) and needs no render.
CONNECTIVITY_VOL_MIN_FRAC = 0.50   # flag if the sustained volume at the FULLER knee level is below
                                   # this fraction of the expected protein molecular volume
                                   # (J1525: 119k/74k = 1.61x OK | J1530: 23k/74k = 0.31x FLAGGED)
SURFACE_RENDER_SIZE   = 1600       # px. surface_judge.py's default 900 gave only 5.9 px/A on the
SURFACE_MIN_PX_PER_A  = 10.0       # test deliverable — below the legibility floor, where a
                                   # side-chain knob and a bump are the same handful of pixels.
                                   # At 10.4 px/A the same map read CONSISTENT (delta 0.33).

# ---- focused local refinement -------------------------------------------------
FOCUS_CORE_RADIUS_A  = 40.0   # ref 40 A. DERIVE IT: the MINIMUM of the radial profile of ORDERED
                              # (high-texture) voxels. Ref profile: protein lobe peaks r 25-30 A,
                              # clean minimum at 45-50 A, separate detergent bump at 50-60 A.
FOCUS_MASK_DILATOR   = 3      # px. human-set project default (memory: focus_mask_default_dil3_pad5)
FOCUS_MASK_PAD       = 5      # px. SWEPT 5 vs 15 in the reference -> 2.95 vs 3.00 A = a TIE by the
                              # <0.2 A rule. The question is UNRESOLVED; 5 is the project default.
FOCUS_MASK_THRESHOLD = 0.5    # REQUIRED for volume_tools(volume_out_type='mask') or it fails
                              # "mask max != 1"
LOCAL_RES_INIT_FACTOR = 1.5   # refine_res_init ~ 1.5 x the CURRENT reported resolution.
                              # ref 1.5 x 3.06 = 4.59, used 4.5. Do NOT copy an absolute 4 A onto a
                              # worse map -- that inverts the logic that made it work.
LOCAL_R_EXTENT = 10           # ref J1077
LOCAL_S_EXTENT = 7            # ref J1077
LOCAL_USE_ALIGNMENT_PRIOR = True   # ref J1077; the prior SIGMAS stay at CryoSPARC defaults
# [v2] LOOP E honest expectation: on the test, a focused local refine ON TOP OF an already-good
# un-binned map was a NEGATIVE result (J1530, rejected above). The loop's stop condition was right
# to end it. Run it, but judge it on the FSC *and* the real-space metrics *and* the connectivity
# volume — not on the masked FSC alone, which a tighter mask improves by shrinking the window.

# ---- deliberately NOT in this pipeline (all measured or judged negative here) --
#   rebalance_3D        : thinned 4% and made the map 0.21 A WORSE (6.54 -> 6.75 A)
#   junk-sink hetero    : split 35.5/36.4/28.1% = did not differentiate; the pool was already clean.
#                         [v2] and on the test a junk-sink seeded with the REAL TARGET as a decoy
#                         drained 103,807 genuine particles — never seed a decoy with a top-metric
#                         candidate (subC.md).
#   class_3D            : killed at iter 449/725 after ~40 h; question answered more cheaply
#   ctf_refine_local    : two sample-matched precedent negatives (+0.22 A, +0.27 A)
#   micelle subtraction : no precedent anywhere in the history bank; CryoSPARC staff advise against
#   template_picker_gpu : the reference's template branch contributed ZERO particles (ncc>=0.70 did
#                         not purify -- scores are unimodal at this SNR)
#   ctf_refine_global   : UNTESTED, explicitly NOT ruled out -- see validation_plan.md, run it as a
#                         one-flag A/B via nonuniform_refine_new's own refine_ctf_global_refine

# ---- [v2] WORK_DIR is PROJECT-SCOPED (finding #3) ---------------------------
# The first run of this notebook wrote its state to PIPELINE_DIR/'runs', i.e. a directory shared by
# every dataset this pipeline is ever run on. Harmless once; the second dataset would silently
# resume into the first one's job UIDs. The scope key below is also written INTO the state file and
# checked on every load (assert_state_scope in cell 3).
STATE_SCOPE = '_'.join(x for x in (PROJECT_UID, WORKSPACE_TITLE, DATASET_TAG) if x)
WORK_DIR    = (Path(os.environ.get('PIPELINE_WORK_DIR') or PIPELINE_DIR / 'runs')
               / STATE_SCOPE)   # renders, masks, relion runs, pipeline_state.json; set
                                # PIPELINE_WORK_DIR when the checkout is read-only

# ---- [v2.2] SAMPLE_INFO — the single source of sample metadata for lib/ ------
# Keys are the ones the profile helpers read (formerly base_datainfo.yaml). Edit the variables above, not this.
SAMPLE_INFO = {
    'sample_name':                  SAMPLE_NAME,
    'micrographs_glob':             MICROGRAPHS_GLOB,
    'pixel_size_A':                 PIXEL_SIZE_A,
    'accelerating_voltage_kV':      ACCEL_KV,          # select2d_utils optics (class ranker)
    'spherical_aberration_mm':      CS_MM,             # select2d_utils optics (class ranker)
    'total_exposure_dose_e_per_A2': TOTAL_DOSE,
    'particle_size_A':              PARTICLE_SIZE_A,   # diagnose2d pre-3D gate
    'particle_character':           PARTICLE_CHARACTER,  # diagnose2d pre-3D gate
    'symmetry':                     SYMMETRY,          # diagnose2d particle-count reminder
    'expected_core_mass_kDa':       EXPECTED_CORE_MASS_KDA,
    'physical_signature':           PHYSICAL_SIGNATURE,
    'model_mask_chimerax':          CHIMERAX_EXE,      # vis_res surface render
}

print('CELL 2 parameters loaded.')
print(f'  sample                   = {SAMPLE_NAME}: {PARTICLE_CHARACTER}')
print(f'  optics                   = {PIXEL_SIZE_A} A/px, {ACCEL_KV} kV, Cs {CS_MM} mm, {TOTAL_DOSE} e/A^2; '
      f'D = {PARTICLE_SIZE_A} A, {SYMMETRY}')
print(f'  WORK_DIR (project-scoped) = {WORK_DIR}')
print(f'  physical signature       = {"SET in this cell" if (PHYSICAL_SIGNATURE or "").strip() else "EMPTY -> every vision call will say NOT SET"}')
print(f'  job mode                 = {"BLOCKING (poll to completion)" if WAIT_FOR_JOBS else "QUEUE-AND-RETURN"}')
print(f'  picking branches         = 2 (blob {BLOB_BAND_FRAC[0]:.2f}-{BLOB_BAND_FRAC[1]:.2f} x D, topaz)')

In [ ]:
# ============================================================================
# CELL 3 — ALL FUNCTION DEFINITIONS
#   * vendored-tool import with LOUD failure (never a silent skip)
#   * dimensional derivation from the NEW sample's optics
#   * job builders + queue + POLL-TO-COMPLETION *or* QUEUE-AND-RETURN + error
#     surfacing + resume-from-UID (scoped per dataset)
#   * live output-group resolution (MCP's output slot names are known-stale)
#   * ONE LLM transport (the Claude Code CLI, headless) + the decision helpers,
#     every one of them carrying the PHYSICAL SIGNATURE verbatim
#   * decide_k / decide_topaz_threshold — the ADAPTIVE knobs, decided per round,
#     never scheduled
#   * [v2] live-GPU admission control, >1M subgroup splitting, the geometric
#     extract guard, branch survivorship, the connectivity-volume check
# (long on purpose; collapse it)
# ============================================================================
import os, re, sys, json, time, math, hashlib, shutil, subprocess, statistics, traceback
from pathlib import Path


class ToolMissing(RuntimeError):
    """A tested decision tool is not importable. NEVER caught to skip a decision."""


class JobFailed(RuntimeError):
    pass


class GateHalt(RuntimeError):
    """The pre-3D gate says stop. Do not spend GPU on a set the gate would have stopped."""


class VisionUnavailable(RuntimeError):
    """The LLM decision transport could not answer. EVERY caller has a deterministic fallback."""


LLMUnavailable = VisionUnavailable          # the transport is no longer vision-only


class KMemoryGuard(RuntimeError):
    """K x box^2 exceeds what the target lane's smallest card survived. Refuse to queue."""


# ---------------------------------------------------------------------------
# 1. vendored tools — fail LOUDLY
# ---------------------------------------------------------------------------
_REQUIRED_TOOLS = ['2d_select/select2d_utils.py', '2d_select/relion_2d_selection_pipeline.py',
                   '2d_select/diagnose2d_utils.py', '2d_select/2d_select_criteria.yaml',
                   'vis_res/vis_res_utils.py', 'vis_res/surface_judge.py',
                   'build_model_mask/build_focus_mask.py', 'cs_connect.py']

missing = [f for f in _REQUIRED_TOOLS if not (PROFILE_DIR / f).exists()]
if missing:
    raise ToolMissing(
        f'profile tools missing from {PROFILE_DIR}: {missing}\n'
        'This notebook refuses to run without its deciders; run it from a complete EM-Claw checkout.')

for _d in reversed(LIB_DIRS):
    if str(_d) not in sys.path:
        sys.path.insert(0, str(_d))

try:
    import select2d_utils as S2          # PRIMARY 2D decider (RELION class_ranker)
    import diagnose2d_utils as D2        # PRE-3D gate
    import vis_res_utils as VR           # resolution lie-detector + count-robust metrics
    import surface_judge as SJ           # the paired surface second look
    from cs_connect import get_cs
except Exception as _e:                  # noqa: BLE001 — we re-raise as ToolMissing on purpose
    raise ToolMissing(f'profile tools present but not importable: {_e!r}\n'
                      f'{traceback.format_exc()}') from _e

WORK_DIR = Path(WORK_DIR)
WORK_DIR.mkdir(parents=True, exist_ok=True)
(WORK_DIR / 'relion_runs').mkdir(parents=True, exist_ok=True)


def write_sample_info(info, path):
    """[v2.2] Serialise cell 2's SAMPLE_INFO for the lib/ tools, which take a path.
    Strings are JSON-quoted (valid YAML, safe for multi-line text) EXCEPT `model_mask_chimerax`,
    which vis_res_utils.surface_config reads with a plain line scan that does not strip quotes."""
    lines = ['# GENERATED by cell 3 from cell 2 SAMPLE_INFO — overwritten every run, do not edit.']
    for k, v in info.items():
        if k == 'model_mask_chimerax':
            v = str(v or '').strip()
            assert '#' not in v and '\n' not in v, f'unsupported character in CHIMERAX_EXE: {v!r}'
            lines.append(f'{k}: {v}')
        else:
            lines.append(f'{k}: {json.dumps(v)}')
    Path(path).write_text('\n'.join(lines) + '\n')
    return str(path)


# point every tool at THIS notebook's inline sample info + this environment (never the reference's)
DATAINFO_YAML = write_sample_info(SAMPLE_INFO, WORK_DIR / 'sample_info.generated.yaml')
S2.DATAINFO_PATH = DATAINFO_YAML
D2.DATAINFO_PATH = DATAINFO_YAML
VR.DATAINFO_PATH = DATAINFO_YAML
S2.RELION_CLASS_RANKER = RELION_CLASS_RANKER
S2.RELION_PYTHON = RELION_PYTHON
S2.DEFAULT_WORKROOT = WORK_DIR / 'relion_runs'
if CHIMERAX_RENDER_HOST:
    os.environ.setdefault('VIS_RES_RENDER_HOST', CHIMERAX_RENDER_HOST)
if CHIMERAX_RENDER_LD_PATH:
    os.environ.setdefault('VIS_RES_RENDER_LD_PATH', CHIMERAX_RENDER_LD_PATH)


# ---------------------------------------------------------------------------
# 1b. [v2] THE PHYSICAL SIGNATURE — read once, injected into EVERY vision prompt
#     Finding #13: the pipeline distilled the PROCEDURE but not the target's physical
#     description, so a vision call invented an accept/reject rule from the sample's NAME
#     ("transporter -> channel -> hollow") and rejected the real target.
#     Source: cell 2's PHYSICAL_SIGNATURE ONLY (no external yaml) -> else NOT SET, said out
#     loud in every prompt.
# ---------------------------------------------------------------------------
_SIG_NOT_SET = ('physical_signature NOT SET — the criteria you apply below are YOUR OWN INFERENCE, '
                'not the human\'s description of this specimen. Say so in your answer, and do NOT '
                'reason from the sample\'s NAME to what its density should look like (that is the '
                'exact failure this line exists to prevent).')


def physical_signature(verbose=False):
    """The specimen description, or '' if the human supplied none. NEVER invents one."""
    sig = (globals().get('PHYSICAL_SIGNATURE') or '').strip()
    if verbose:
        if sig:
            print('PHYSICAL SIGNATURE (cell 2 PHYSICAL_SIGNATURE):\n    ' + sig.replace('\n', '\n    '))
        else:
            print('!! PHYSICAL SIGNATURE NOT SET in cell 2. Every vision call will say so. This is '
                  'the single input whose absence cost the first generalisation test its 3D stage '
                  '— write one paragraph in cell 2.')
    return sig


def sig_block(what='this judgement'):
    """The verbatim signature, wrapped for a prompt. Always call this — never paraphrase."""
    sig = physical_signature()
    if not sig:
        return ('\nPHYSICAL SIGNATURE OF THE SPECIMEN: ' + _SIG_NOT_SET + '\n')
    return ('\nPHYSICAL SIGNATURE OF THE SPECIMEN (supplied by the human — it GOVERNS ' + what +
            ', use it verbatim and do not substitute an inference from the sample\'s name):\n'
            + sig + '\n\nSCOPE: the signature is an IDENTITY test (is this the target, or something '
            'else?), NOT a quality override. A blurry, smeared, featureless average or map is a BAD '
            'result on the ordinary EM standard no matter which part of the target its blur belongs '
            'to. Both gates must pass.\n')


def control_images(kind):
    """[v2] Vendored labelled controls (finding #13). Returns [] if disabled or missing."""
    if not USE_VISUAL_CONTROLS:
        return []
    want = {'abinit': [CONTROL_ABINIT_TARGET, CONTROL_ABINIT_ARTIFACT, CONTROL_FINAL_MAP],
            'final': [CONTROL_FINAL_MAP]}.get(kind, [])
    have = [str(p) for p in want if Path(p).exists()]
    if len(have) < len(want):
        print(f'!! some vendored controls are missing from {CONTROLS_DIR} — the {kind} vision call '
              f'will run WITHOUT the positive control it is supposed to compare against. '
              f'(present: {[Path(h).name for h in have]})')
    return have


CONTROL_LEGEND = (
    'ATTACHED CONTROLS (labelled, from a KNOWN outcome on this sample class — compare against them, '
    'not against your prior):\n'
    '  * "reference_final_map_J1077_2.95A_slices.png" = POSITIVE. The reference project\'s own final '
    'map at 2.95 A, native slices + projections, 10 A bar.\n'
    '  * "abinit_TARGET_J1458_class_3.png" = POSITIVE at AB-INITIO resolution. This class refined '
    'alone to 3.34 A and then 2.97 A.\n'
    '  * "abinit_ARTIFACT_J1458_class_0.png" = NEGATIVE. Same job, an isotropic hollow shell — the '
    'SGD spherical-shell artifact. A screen that preferred THIS one cost the test its whole 3D stage.\n')


# ---------------------------------------------------------------------------
# 1c. [v2] fingerprint of this cell (finding #2: the vendor-a-copy pattern creates one
#     drift point per copy — a runner that exec()s a hand-copied header can be stale).
# ---------------------------------------------------------------------------
NOTEBOOK_PATH = PIPELINE_DIR / 'transporter_lowSNR_micelle_pipeline.ipynb'


def cell_fingerprint(cell_index=2, path=None):
    """SHA-256 of the notebook's function cell AS IT IS ON DISK. '' if unreadable."""
    p = Path(path or NOTEBOOK_PATH)
    try:
        src = ''.join(json.loads(p.read_text())['cells'][cell_index]['source'])
        return hashlib.sha256(src.encode()).hexdigest()[:16]
    except Exception as e:                             # noqa: BLE001 — informational only
        print(f'(could not fingerprint {p}: {e!r})')
        return ''


def check_tool_binaries():
    """Warn LOUDLY about anything the deciders shell out to. Never silently degrade."""
    problems = []
    physical_signature(verbose=True)
    for k in ('abinit',):
        control_images(k)
    if not Path(RELION_CLASS_RANKER).exists():
        problems.append(f'RELION class_ranker not found at {RELION_CLASS_RANKER} '
                        '-> relion_rank_select CANNOT run')
    if not Path(RELION_PYTHON).exists():
        problems.append(f'RELION python not found at {RELION_PYTHON}')
    if not Path(TOPAZ_EXE).exists():
        problems.append(f'topaz not found at {TOPAZ_EXE} -> the third picking branch cannot run')
    for k in ('accelerating_voltage_kV', 'spherical_aberration_mm', 'pixel_size_A',
              'particle_size_A', 'particle_character', 'symmetry'):
        if SAMPLE_INFO.get(k) in (None, ''):
            problems.append(f'cell 2 SAMPLE_INFO[{k!r}] is empty -> the deciders fall back to '
                            'GENERIC values for it')
    if not str(MICROGRAPHS_GLOB).strip():
        problems.append('MICROGRAPHS_GLOB is empty in cell 2 -> STEP 1 cannot import micrographs')
    cfg = VR.surface_config(DATAINFO_YAML)
    if not cfg.get('chimerax') or not Path(cfg['chimerax']).exists():
        problems.append(f'CHIMERAX_EXE not set or not found ({CHIMERAX_EXE!r}) -> the vis_res SURFACE '
                        'second look is UNAVAILABLE; a verdict carrying SURFACE_UNAVAILABLE is '
                        'explicitly UNCONFIRMED')
    for p in problems:
        print('!! ' + p)
    if not problems:
        print('all decider binaries/paths resolve OK')
    llm_status()          # NEVER a `problem`: a missing CLI degrades to the printed fallbacks
    return problems


def llm_cli_path():
    """Resolve the Claude Code CLI, or None. NEVER raises — the notebook runs without it."""
    p = (CLAUDE_CLI or '').strip()
    if p and Path(p).exists() and os.access(p, os.X_OK):
        return p
    w = shutil.which(p) if p and os.sep not in p else None
    return w or shutil.which('claude')


def llm_status(verbose=True):
    """Print whether the LLM decision calls are ENABLED or the notebook is on fallbacks."""
    exe = llm_cli_path()
    if not exe:
        if verbose:
            print(f'!! LLM DECISIONS DISABLED — no `claude` CLI at {CLAUDE_CLI!r} and none on PATH.\n'
                  '   The notebook still runs: every LLM decision has a DETERMINISTIC fallback and\n'
                  '   each one prints that it fell back and why. Specifically, decide_k() will HOLD\n'
                  '   class2D_K at its current value for every round (it never descends), and the\n'
                  '   visual reads (pre-3D gate, ab-initio class screen, vis_res) print their image\n'
                  '   paths for you to read yourself. Set CLAUDE_CLI in cell 2 to enable them.')
        return None
    ver = ''
    try:
        r = subprocess.run([exe, '--version'], capture_output=True, text=True, timeout=60)
        ver = (r.stdout or r.stderr or '').strip().splitlines()[0] if r.returncode == 0 else ''
    except Exception as e:                      # noqa: BLE001 — availability probe only
        ver = f'(--version failed: {e!r})'
    if verbose:
        print(f'LLM DECISIONS ENABLED via the Claude Code CLI: {exe} {ver}\n'
              '   Headless `claude -p ... --output-format json`, authenticated from\n'
              '   ~/.claude/.credentials.json — a Claude SUBSCRIPTION is enough, no API key.\n'
              '   Each call spends your subscription usage (~$0.14-equivalent per short call when\n'
              '   measured on this server); decide_k() fires once per completed 2D round.')
    return exe


# ---------------------------------------------------------------------------
# 2. dimensional derivation — from the NEW sample's optics, not the reference's
# ---------------------------------------------------------------------------
_FFT_FRIENDLY = [96, 100, 108, 120, 128, 144, 150, 160, 180, 192, 200, 216, 224, 240, 256,
                 264, 288, 300, 320, 336, 360, 384, 400, 432, 440, 448, 480, 512, 540, 576]


def _snap_box(px):
    return min(_FFT_FRIENDLY, key=lambda b: (abs(b - px), b))


def derive_dimensions(verbose=True):
    """Every dimensional knob, re-derived from THIS sample. Reference values printed alongside."""
    D = float(PARTICLE_SIZE_A)
    ps = float(PIXEL_SIZE_A)
    raw = BOX_TARGET_MULTIPLE * D / ps
    box_full = _snap_box(raw)
    box_bin = _snap_box(box_full / BIN_FACTOR)
    d = {
        'D_A': D,
        'psize_A': ps,
        'box_full_px': box_full,
        'box_bin_px': box_bin,
        'box_extent_A': box_full * ps,
        'box_multiple': box_full * ps / D,
        'psize_binned_A': ps * box_full / box_bin,
        'nyquist_binned_A': 2.0 * ps * box_full / box_bin,
        'nyquist_full_A': 2.0 * ps,
        'blob_A': (round(BLOB_BAND_FRAC[0] * D), round(BLOB_BAND_FRAC[1] * D)),
        'topaz_par_diam_A': round(TOPAZ_PAR_DIAM_FRAC * D),
        'dedup_min_dist_A': round(DEDUP_MIN_DIST_FRAC * D),
        'refsel_window_inner_A': round(0.80 * box_full * ps),
    }
    if verbose:
        print(f"""DERIVED DIMENSIONS  (reference zll_260130 value in brackets)
  particle D                 {d['D_A']:.0f} A            [100 A]
  pixel size                 {d['psize_A']:.4f} A/px      [0.824]
  FULL box   {BOX_TARGET_MULTIPLE}x D / psize = {raw:.0f} px -> snapped {d['box_full_px']} px
                             = {d['box_extent_A']:.1f} A = {d['box_multiple']:.2f}x D   [300 px = 247.2 A = 2.47x]
  BINNED box (bin {BIN_FACTOR})        {d['box_bin_px']} px @ {d['psize_binned_A']:.3f} A/px,
                             Nyquist {d['nyquist_binned_A']:.3f} A                 [150 px, 1.648, 3.296 A]
  UNBINNED Nyquist           {d['nyquist_full_A']:.3f} A                 [1.648 A]
  blob band (ONE branch)     {d['blob_A'][0]}/{d['blob_A'][1]} A               [ref ran TWO: 100/150 and 90/130;
                                                       this is their UNION, 0.90-1.50 x D]
  topaz par_diam             {d['topaz_par_diam_A']} A                   [100]
  dedup min_dist             {d['dedup_min_dist_A']} A                    [60]
  reference_select window_inner_A {d['refsel_window_inner_A']} A          [200]  (MUST be explicit)
  un-bin trigger             report <= {UNBIN_TRIGGER_FACTOR} x {d['nyquist_binned_A']:.3f} = {UNBIN_TRIGGER_FACTOR * d['nyquist_binned_A']:.2f} A  [<= 4.28 A]""")
    return d


# ---------------------------------------------------------------------------
# 3. connection + state (resume-from-UID)
# ---------------------------------------------------------------------------
STATE_PATH = Path(WORK_DIR) / 'pipeline_state.json'
_SCOPE_KEY = '__scope__'


def _scope_now():
    return {'project': PROJECT_UID, 'workspace_title': WORKSPACE_TITLE,
            'dataset_tag': DATASET_TAG, 'micrographs_glob': MICROGRAPHS_GLOB,
            'pixel_size_A': PIXEL_SIZE_A, 'cell3_sha256': cell_fingerprint()}


def load_state():
    if STATE_PATH.exists():
        return json.loads(STATE_PATH.read_text())
    return {}


def save_state(st):
    st.setdefault(_SCOPE_KEY, _scope_now())
    STATE_PATH.parent.mkdir(parents=True, exist_ok=True)
    STATE_PATH.write_text(json.dumps(st, indent=2, sort_keys=True))


def assert_state_scope(strict=True):
    """[v2] finding #3 — REFUSE to resume a state file written for a different dataset.

    The first run of this notebook wrote its state to a directory shared by every dataset
    (PIPELINE_DIR/'runs'), so a second dataset would have silently resumed into the first one's
    job UIDs. WORK_DIR is now scoped by project+workspace(+tag) AND the scope is recorded inside
    the file, because a scoped path alone is only a convention — this is the check.
    """
    st = load_state()
    if not st:
        print(f'state: fresh (no {STATE_PATH.name} yet in {STATE_PATH.parent})')
        return True
    old, new = st.get(_SCOPE_KEY), _scope_now()
    if not old:
        print(f'!! {STATE_PATH} carries NO scope record (written by an older version of this '
              f'notebook). It will be stamped with the CURRENT scope on the next write — make sure '
              f'it really belongs to {new["project"]}/{new["workspace_title"]!r}.')
        return True
    diff = {k: (old.get(k), new.get(k)) for k in ('project', 'workspace_title', 'dataset_tag',
                                                  'micrographs_glob')
            if old.get(k) != new.get(k)}
    if diff:
        msg = (f'STATE SCOPE MISMATCH in {STATE_PATH}:\n' +
               '\n'.join(f'    {k}: state has {a!r}, this run is {b!r}' for k, (a, b) in diff.items()) +
               '\nResuming would reuse ANOTHER dataset\'s job UIDs. Point DATASET_TAG/WORKSPACE_TITLE '
               'at the right run, or delete/rename this state file, or set RESUME=False.')
        if strict:
            raise RuntimeError(msg)
        print('!! ' + msg)
        return False
    if old.get('cell3_sha256') and old['cell3_sha256'] != new.get('cell3_sha256'):
        print(f'   (note: the function cell has CHANGED since this state was written — '
              f'{old["cell3_sha256"]} -> {new.get("cell3_sha256")}. Completed jobs are still reused; '
              f'this is only so a drifted hand-copied runner is visible. Finding #2.)')
    print(f'state scope OK: {new["project"]} / {new["workspace_title"]!r}'
          + (f' / {new["dataset_tag"]}' if new.get('dataset_tag') else '')
          + f'  ({len([k for k in st if k != _SCOPE_KEY])} remembered jobs)')
    return True


def remember(key, uid):
    st = load_state()
    st[key] = uid
    save_state(st)


def recall(key):
    return load_state().get(key) if RESUME else None


def forget(key):
    st = load_state()
    st.pop(key, None)
    save_state(st)
    print(f'forgot {key!r} — the next run of that step will re-create the job')


def connect():
    """Authenticated connection + project + workspace (created if the title is new)."""
    cs = get_cs()
    proj = cs.find_project(PROJECT_UID)
    ws = None
    try:
        for w in (cs.cli.list_workspaces(PROJECT_UID) or []):
            if w.get('title') == WORKSPACE_TITLE:
                ws = proj.find_workspace(w['uid'])
                break
    except Exception as e:
        print(f'(could not list workspaces: {e!r}; a new one will be created)')
    if ws is None:
        ws = proj.create_workspace(title=WORKSPACE_TITLE)
        print(f'created workspace {ws.uid} "{WORKSPACE_TITLE}"')
    else:
        print(f'reusing workspace {ws.uid} "{WORKSPACE_TITLE}"')
    print(f'project {PROJECT_UID} dir = {proj.dir()}')
    return cs, proj, ws


# ---------------------------------------------------------------------------
# 4. live output-group resolution
#    MCP's OUTPUT slot names are known-stale (memory: mcp_output_slot_names_stale).
#    Four instances were recorded in this very reference project:
#      remove_duplicate_particles -> `particles_kept`   (MCP says `particles`)
#      homo_abinit -> `particles_class_K`/`volume_class_K` (MCP does not enumerate them)
#      volume_tools(out_type=mask)  -> `mask`           (MCP says `volume`)
#      import_volumes(out_name=mask)-> `imported_mask_1`(MCP says `imported_volume_1`)
#    So NEVER hard-code an output name: ask the live job.
# ---------------------------------------------------------------------------
def live_outputs(job):
    job.refresh()
    return [g['name'] for g in job.doc.get('output_result_groups', [])]


def resolve_out(job, *candidates):
    names = live_outputs(job)
    for c in candidates:
        if c in names:
            return c
    raise KeyError(f'{job.uid} ({job.type}) has none of {candidates}; live output groups = {names}')


def particles_group(job):
    """[v2] The particle output group of a select_2D / dedup / merge job, whichever it is.

    A branch's final step can now be a select_2D ('particles_selected') OR a
    remove_duplicate_particles used as the union of subgroup selections ('particles_kept'), so
    callers must not hard-code either.
    """
    return resolve_out(job, 'particles_selected', 'particles_kept', 'particles')


def n_items(job, group):
    job.refresh()
    for g in job.doc.get('output_result_groups', []):
        if g['name'] == group:
            n = g.get('num_items')
            if n is not None:
                return int(n)
    try:
        return len(job.load_output(group))
    except Exception:
        return -1


# ---------------------------------------------------------------------------
# 5. job builder + queue + POLL-TO-COMPLETION + error surfacing
# ---------------------------------------------------------------------------
def _fmt(params):
    return ', '.join(f'{k}={v!r}' for k, v in sorted(params.items()))


def surface_errors(job):
    """Print everything CryoSPARC knows about why a job died."""
    job.refresh()
    print(f'--- {job.uid} ({job.type}) status={job.status} ---')
    errs = job.doc.get('errors_run') or []
    for e in errs:
        print('ERROR:', e.get('message', e))
    for k in ('failed_job_reason', 'killed_job_reason'):
        if job.doc.get(k):
            print(f'{k}: {job.doc[k]}')
    try:
        log = job.log()
        tail = log[-4000:] if isinstance(log, str) else str(log)[-4000:]
        print('--- tail of job log ---')
        print(tail)
    except Exception as e:
        print(f'(could not read job log: {e!r})')


def poll(job, poll_s=None, timeout_h=None):
    """Block until the job reaches a terminal state. Raises JobFailed with the reason."""
    poll_s = poll_s or POLL_SECONDS
    deadline = time.time() + (timeout_h or JOB_TIMEOUT_H) * 3600
    last = None
    t0 = time.time()
    while True:
        job.refresh()
        st = job.status
        if st != last:
            print(f'  [{time.strftime("%H:%M:%S")}] {job.uid} -> {st}')
            last = st
        if st == 'completed':
            print(f'  {job.uid} COMPLETED in {(time.time() - t0) / 60:.1f} min')
            return job
        if st in ('failed', 'killed'):
            surface_errors(job)
            raise JobFailed(f'{job.uid} ({job.type}) ended with status={st}')
        if time.time() > deadline:
            raise JobFailed(f'{job.uid} still {st} after {timeout_h or JOB_TIMEOUT_H} h — NOT '
                            f'assumed failed. Check the server, then re-run this cell (RESUME will '
                            f'pick {job.uid} back up).')
        time.sleep(poll_s)


# [v2] finding #4 — the FULL non-terminal status set. subC's hand-copied queue-only helper
# omitted 'started' (and 'waiting'), so a same-turn re-run did not resume J1431: it created a
# SECOND job, J1432, on the SAME explicit GPU ids, and both ran (an explicit gpus= list is not an
# exclusive reservation against your own jobs). Any status not here and not terminal must be
# added, never guessed around.
ACTIVE_STATUSES = ('queued', 'running', 'launched', 'starting', 'started', 'waiting')
TERMINAL_BAD = ('failed', 'killed')


def require_completed(job, what=''):
    """Downstream cells call this instead of assuming. Loud in queue-and-return mode."""
    job.refresh()
    if job.status != 'completed':
        raise JobFailed(f'{job.uid} ({job.type}) is {job.status}, not completed{" — " + what if what else ""}. '
                        f'In QUEUE-AND-RETURN mode (WAIT_FOR_JOBS=False) you must re-execute the '
                        f'cells once the job finishes; RESUME will pick {job.uid} back up rather '
                        f'than re-creating it.')
    return job


def run_job(key, proj, ws, job_type, connections, params, title,
            gpus=None, lane=None, hostname=None, poll_s=None, timeout_h=None, queue=True,
            wait=None, admission_mib=None):
    """Create (or RESUME) a job, queue it, and either POLL to completion or RETURN immediately.

    `key` is the resume key. `wait` defaults to the WAIT_FOR_JOBS global:
      True  -> block until terminal (a human at a notebook)
      False -> queue, confirm it launched, return the job object (an agent turn / a long job)
    `admission_mib` (optional) runs the LIVE per-GPU memory check on `gpus` immediately before
    .queue() and may RE-ROUTE to freer ids on the same lane (finding #11).
    """
    wait = WAIT_FOR_JOBS if wait is None else wait
    uid = recall(key)
    if uid:
        job = proj.find_job(uid)
        job.refresh()
        if job.status == 'completed':
            print(f'RESUME  {key}: {uid} ({job.type}) already completed — reusing')
            return job
        if job.status in ACTIVE_STATUSES:
            if wait:
                print(f'RESUME  {key}: {uid} is {job.status} — polling')
                return poll(job, poll_s, timeout_h)
            print(f'RESUME  {key}: {uid} is {job.status} — NOT re-creating, NOT polling '
                  f'(WAIT_FOR_JOBS=False). Re-run this cell later to pick it up.')
            return job
        print(f'RESUME  {key}: {uid} is {job.status} — creating a fresh job')
    def _norm_conn(v):
        # a single connection is a (job_uid, output_name) TUPLE; a MULTI-input connection is a
        # LIST of such tuples (memory: cryosparc_live_job_gotchas -- create_job connections need
        # TUPLES not lists for the pair, but the multi-input container must stay a list).
        if isinstance(v, (list, tuple)) and len(v) == 2 and all(isinstance(x, str) for x in v):
            return tuple(v)
        if isinstance(v, (list, tuple)):
            return [_norm_conn(x) for x in v]
        return v
    conns = {k: _norm_conn(v) for k, v in connections.items()}
    job = proj.create_job(ws.uid, job_type, connections=conns, params=params, title=title)
    remember(key, job.uid)
    print(f'CREATE  {key}: {job.uid}  {job_type}  "{title}"')
    print(f'        params: {_fmt(params) or "(all defaults)"}')
    print(f'        inputs: {conns}')
    if not queue:
        return job
    lane_ = lane or LANE
    if admission_mib and gpus:
        gpus = gpu_admission_check(lane_, gpus, admission_mib, label=f'{key} ({job_type})')
    job.queue(lane=lane_, hostname=hostname or HOSTNAME,
              **({'gpus': gpus} if gpus else {}))
    if wait:
        return poll(job, poll_s, timeout_h)
    # QUEUE-AND-RETURN: confirm it actually launched (a job that dies on queue must not look OK),
    # print what it got, and hand the object back.
    time.sleep(QUEUE_CONFIRM_S)
    job.refresh()
    slots = (job.doc.get('resources_allocated') or {}).get('slots', {}).get('GPU')
    print(f'  QUEUED (not waiting) {job.uid}: status={job.status} after {QUEUE_CONFIRM_S}s, '
          f'GPU slots={slots}')
    if job.status in TERMINAL_BAD:
        surface_errors(job)
        raise JobFailed(f'{job.uid} ({job.type}) failed immediately on queue: status={job.status}')
    print(f'  -> re-execute this cell when it is done; RESUME will reuse {job.uid}.')
    return job


# ---------------------------------------------------------------------------
# 6. health guards (each one exists because it silently bit somebody)
# ---------------------------------------------------------------------------
def check_picks_per_mic(pick_job, group, n_mics, label):
    n = n_items(pick_job, group)
    ppm = n / max(n_mics, 1)
    print(f'  {label}: {n:,} picks over {n_mics:,} mics = {ppm:.1f} picks/mic '
          f'(ref blob 729-881, topaz 190)')
    if ppm < MIN_PICKS_PER_MIC:
        print(f'!! HALT-WORTHY: {ppm:.1f} picks/mic < {MIN_PICKS_PER_MIC}. For topaz_extract this is '
              f'almost certainly the ADVANCED `particle_threshold` gate: its default 0 can return '
              f'~1 particle per micrograph while the job still reports "completed". [v2] STEP 4 '
              f'derives that threshold with decide_topaz_threshold() from the model\'s OWN RAW '
              f'log-odds scores — if you are seeing this after that, the threshold was applied on '
              f'the wrong axis (pick_stats/ncc_score is a DIFFERENT quantity). Re-read '
              f'topaz_particles_prediction.txt and re-run; do not "fix" it by reverting to 0.')
    return ppm


def mic_shape_px(proj, job, group):
    """[v2] (H, W) of a micrograph, read LIVE. MIC_SHAPE_PX in cell 2 overrides. (None, None) if
    neither is available — the extract guard then falls back to the flat tolerance and says so."""
    if MIC_SHAPE_PX:
        return int(MIC_SHAPE_PX[0]), int(MIC_SHAPE_PX[1])
    try:
        ds = job.load_output(group, slots=['micrograph_blob'])
        sh = ds['micrograph_blob/shape'][0]
        return int(sh[0]), int(sh[1])
    except Exception as e:                             # noqa: BLE001 — informational
        print(f'   (could not read the micrograph shape off {job.uid}/{group}: {e!r}; '
              f'set MIC_SHAPE_PX in cell 2 to enable the geometric extract prediction)')
        return None, None


def predicted_extract_loss(box_px, mic_hw):
    """[v2] finding #5 — the geometric edge loss of a FIRST extract: 1 - ((W-b)/W)((H-b)/H).

    This is an UPPER BOUND and it is PICKER-DEPENDENT. Blob proposes picks right up to the frame
    edge and loses exactly this fraction (test: 15.96% / 15.72% against 15.73% predicted). Topaz
    self-excludes an edge margin (coordinates spanned 110-3960 in a 4096 frame) and lost only
    0.53%. So a yield ABOVE the prediction is normal; only a shortfall BELOW it is a fault.
    """
    h, w = (mic_hw or (None, None))
    if not h or not w or box_px >= min(h, w):
        return None
    return 1.0 - ((w - box_px) / float(w)) * ((h - box_px) / float(h))


def check_extract_count(extract_job, group, expected, first_extract=True, mic_hw=None,
                        box_px=None, tol=None):
    """GPU extract has TWICE silently truncated while exiting 'completed' (memory:
    gpu_extract_silent_truncation). [v2] compare against the GEOMETRIC prediction for a FIRST
    extract, and keep the tight tolerance for a re-extract of already-centred particles."""
    n = n_items(extract_job, group)
    lost = 1.0 - n / max(expected, 1)
    pred = predicted_extract_loss(int(box_px or 0), mic_hw) if first_extract else 0.0
    if first_extract and pred is not None:
        tol = EXTRACT_SHORTFALL_TOL if tol is None else tol
        print(f'  extract {extract_job.uid}: {n:,} of {expected:,} picks kept — lost {lost * 100:.2f}%, '
              f'geometry predicts up to {pred * 100:.2f}% for box {box_px} on '
              f'{mic_hw[1]}x{mic_hw[0]} px (an UPPER bound; a picker with an edge margin loses less)')
        if lost > pred + tol:
            print(f'!! SHORTFALL: {(lost - pred) * 100:.2f} points BELOW the geometric prediction '
                  f'(tolerance {tol * 100:.0f}%). Silent truncation loses particles, it never adds '
                  f'them — RE-RUN this extract with compute_num_gpus=0 (CPU) and compare '
                  f'(memory: gpu_extract_silent_truncation).')
        elif lost < pred - 0.01:
            print(f'   (kept MORE than geometry predicts — normal for a picker that self-excludes an '
                  f'edge margin; not a fault. Test: topaz kept 99.47% against 84.27% predicted.)')
    else:
        tol = REEXTRACT_TOL if tol is None else tol
        print(f'  re-extract {extract_job.uid}: {n:,} particles (expected ~{expected:,}, '
              f'lost {lost * 100:.2f}%)')
        if lost > tol:
            print(f'!! {lost * 100:.1f}% of particles are MISSING. Up to ~1% is the benign '
                  f'edge-of-frame exclusion from recenter_using_shifts (reference 0.80%, test '
                  f'0.7-0.8%). More than that is the GPU-extract silent-truncation failure mode — '
                  f'RE-RUN with compute_num_gpus=0 (CPU) and compare.')
    return n


# ---------------------------------------------------------------------------
# 6b. [v2] LIVE GPU admission control (finding #11)
#     A static resource model (card class x job size) gives false confidence on a shared node:
#     J1437 (K=250 @ box 160 = 6.40e6, UNDER the small-card ceiling) died with
#     CUDA_ERROR_OUT_OF_MEMORY because a small-card node assigned GPU0/GPU1, carrying ~8 GB of load the
#     CryoSPARC scheduler cannot see. Take the reading LIVE, on the ids you are about to use, as
#     the LAST thing before .queue() — foreign load flips inside a single turn.
# ---------------------------------------------------------------------------
def live_gpu_table(lane, timeout=60):
    """[(index, used_mib, total_mib, util_pct)] on `lane`, or [] if it cannot be read."""
    cmd = ([] if lane in ('', None, 'localhost') else [SSH_CMD, str(lane)]) + [
        'nvidia-smi', '--query-gpu=index,memory.used,memory.total,utilization.gpu',
        '--format=csv,noheader,nounits']
    try:
        r = subprocess.run(cmd, capture_output=True, text=True, timeout=timeout)
        if r.returncode != 0:
            print(f'   (live GPU read failed on {lane}: rc={r.returncode} {r.stderr.strip()[:200]})')
            return []
        out = []
        for line in r.stdout.strip().splitlines():
            i, used, total, util = [x.strip() for x in line.split(',')]
            out.append((int(i), int(used), int(total), int(util)))
        return out
    except Exception as e:                             # noqa: BLE001 — availability probe
        print(f'   (live GPU read unavailable on {lane}: {e!r})')
        return []


def live_free_mib(lane, gpu_ids=None):
    """{id: free_mib} for the requested ids (all ids if None)."""
    tbl = live_gpu_table(lane)
    return {i: (tot - used) for i, used, tot, _u in tbl
            if gpu_ids is None or i in set(gpu_ids)}


def predicted_gpu_mib(K, box_px, n_gpus):
    """Convert the K x box^2 budget into an actual per-GPU MiB estimate (MIB_PER_KBOX2_UNIT is
    the 7.0e6 small-card ceiling factored into an explicit number, so it can be compared against
    a MEASURED free-memory reading instead of a card-class label)."""
    return (int(K) * int(box_px) ** 2) * MIB_PER_KBOX2_UNIT / max(int(n_gpus or 1), 1)


def gpu_admission_check(lane, gpu_ids, need_mib, label='', reroute=None):
    """Require live free >= need_mib + margin on EVERY requested id. Re-route or refuse.

    Returns the (possibly re-routed) gpu id list. Everything it does is printed — a silent clamp
    on a resource decision is how J1437 happened.
    """
    reroute = GPU_ADMISSION_REROUTE if reroute is None else reroute
    ids = list(gpu_ids or [])
    need = float(need_mib) + GPU_FREE_MARGIN_MIB
    tbl = live_gpu_table(lane)
    if not tbl:
        print(f'!! GPU ADMISSION: could not read live memory on {lane} — proceeding on the static '
              f'K x box^2 guard alone for {label}. That is exactly the blind spot that killed '
              f'J1437; check nvidia-smi yourself if this job OOMs.')
        return ids
    info = {i: (used, tot, util) for i, used, tot, util in tbl}
    print(f'  GPU ADMISSION on {lane} for {label}: need {need:,.0f} MiB free per GPU '
          f'({need_mib:,.0f} predicted + {GPU_FREE_MARGIN_MIB:,} margin)')
    for i in ids:
        if i not in info:
            print(f'    gpu{i}: NOT PRESENT on {lane}')
            continue
        used, tot, util = info[i]
        print(f'    gpu{i}: {tot - used:,} MiB free of {tot:,}  util {util}%'
              + ('  <- SHORT' if (tot - used) < need else '')
              + (f'  <- {util}% FOREIGN UTIL (expect ~65% more wall-clock)'
                 if util >= GPU_UTIL_WARN_PCT else ''))
    short = [i for i in ids if i not in info or (info[i][1] - info[i][0]) < need]
    if not short:
        return ids
    if not reroute:
        raise KMemoryGuard(f'{label}: GPUs {short} on {lane} are short of memory RIGHT NOW '
                           f'(need {need:,.0f} MiB free each). Not queueing.')
    cand = sorted((i for i in info if (info[i][1] - info[i][0]) >= need),
                  key=lambda i: -(info[i][1] - info[i][0]))
    if len(cand) < len(ids):
        raise KMemoryGuard(
            f'{label}: only {len(cand)} GPU(s) on {lane} have {need:,.0f} MiB free '
            f'({cand}), but {len(ids)} were requested. REFUSING to queue rather than repeating '
            f'J1437 (a job that fitted the card-class formula and still OOM\'d on the ids it was '
            f'actually given). Options: wait, use LANE={BIG_K_LANE!r}, or lower the job size.')
    new = sorted(cand[:len(ids)])
    print(f'  -> RE-ROUTED {ids} -> {new} (freest ids on {lane}, measured just now)')
    return new


# ---------------------------------------------------------------------------
# 7. THE ONE LLM TRANSPORT (so the notebook does not depend on being agent-run)
#    Shells out to the CLAUDE CODE CLI in headless mode:
#        claude -p '<prompt>' --output-format json   [--allowedTools Read]
#    * authenticates from ~/.claude/.credentials.json -> a Claude SUBSCRIPTION is enough;
#      NO ANTHROPIC_API_KEY and no API account (that is why this is not the HTTP API).
#    * images are attached by ABSOLUTE path inside the prompt + --allowedTools Read.
#      A relative path is NOT resolved from the CWD — always absolute.
#    * subprocess.run(list, timeout=...) — never shell=True, never unbounded.
#    Every caller has a DETERMINISTIC fallback right next to it, and says when it used it.
# ---------------------------------------------------------------------------
def _llm_json(prompt, image_paths=None, timeout=None, schema_hint=None, label='llm'):
    """Ask the CLI for a strict JSON answer; return a dict, or raise LLMUnavailable.

    Raises (never returns a guess) when: the CLI is absent, it times out, it exits non-zero,
    its envelope is not JSON, or the answer contains no JSON object.
    """
    exe = llm_cli_path()
    if not exe:
        raise LLMUnavailable(
            f'the Claude Code CLI was not found (CLAUDE_CLI={CLAUDE_CLI!r}, not on PATH), so the '
            f'{label} decision cannot be made by the model. The deterministic fallback next to '
            'this call will be used, and every input is printed so a human can decide instead.')
    imgs = [str(Path(p).resolve()) for p in (image_paths or []) if p and Path(p).exists()]
    if image_paths and not imgs:
        raise LLMUnavailable(f'{label}: none of the images exist: {list(image_paths)}')
    full = prompt
    if imgs:
        full = ('Read these image files (absolute paths) with the Read tool, then answer:\n'
                + '\n'.join('  ' + p for p in imgs) + '\n\n' + prompt)
    if schema_hint:
        full += ('\n\nAnswer with ONLY a JSON object (no prose, no code fence) of the form:\n'
                 + schema_hint)
    cmd = [exe, '-p', full, '--output-format', 'json']
    if imgs:
        cmd += ['--allowedTools', 'Read']
    t0 = time.time()
    try:
        r = subprocess.run(cmd, capture_output=True, text=True,
                           timeout=timeout or CLAUDE_CLI_TIMEOUT_S)
    except subprocess.TimeoutExpired:
        raise LLMUnavailable(f'{label}: the CLI did not answer within '
                             f'{timeout or CLAUDE_CLI_TIMEOUT_S}s')
    except Exception as e:                       # noqa: BLE001 — surfaced as LLMUnavailable
        raise LLMUnavailable(f'{label}: could not run {exe}: {e!r}') from e
    if r.returncode != 0:
        raise LLMUnavailable(f'{label}: CLI exited {r.returncode}. stderr tail:\n'
                             f'{(r.stderr or "")[-800:]}')
    try:
        env = json.loads(r.stdout)
    except Exception as e:                       # noqa: BLE001
        raise LLMUnavailable(f'{label}: CLI stdout was not the JSON envelope: '
                             f'{(r.stdout or "")[:400]!r}') from e
    if isinstance(env, dict) and env.get('is_error'):
        raise LLMUnavailable(f'{label}: CLI reported is_error: {str(env.get("result"))[:400]}')
    text = env.get('result') if isinstance(env, dict) else str(env)
    if not isinstance(text, str):
        raise LLMUnavailable(f'{label}: envelope has no string `result`: {str(env)[:400]}')
    m = re.search(r'\{.*\}', text, re.S)
    if not m:
        raise LLMUnavailable(f'{label}: the answer contained no JSON object:\n{text[:800]}')
    try:
        out = json.loads(m.group(0))
    except Exception as e:                       # noqa: BLE001
        raise LLMUnavailable(f'{label}: the answer was not parseable JSON:\n'
                             f'{m.group(0)[:800]}') from e
    print(f'   [{label}] claude CLI answered in {time.time() - t0:.0f}s'
          + (f' (cost {env["total_cost_usd"]:.3f} USD-equivalent of your subscription usage)'
             if isinstance(env, dict) and isinstance(env.get('total_cost_usd'), (int, float))
             else ''))
    return out


def vision_json(image_paths, prompt, schema_hint):
    """Strict-JSON answer about IMAGES. Same single transport; raises VisionUnavailable."""
    return _llm_json(prompt, image_paths=image_paths, schema_hint=schema_hint, label='vision')


# ---------------------------------------------------------------------------
# 8. DECISION helper — 2D selection (PRIMARY decider: RELION class_ranker)
# ---------------------------------------------------------------------------
def rank_select(cs, proj, ws, class2d_job, mode, key, title,
                min_score=None, drop=None, add=None):
    """relion_rank_select -> guard_adjust(mode) -> select_2D. Fails loudly if the tool cannot run."""
    uid = recall(key)
    if uid:
        j = proj.find_job(uid)
        j.refresh()
        if j.status == 'completed':
            print(f'RESUME  {key}: {uid} select already completed — reusing')
            return j, None
    r = S2.relion_rank_select(proj, class2d_job.uid, min_score=min_score or RANKER_MIN_SCORE,
                              datainfo_path=DATAINFO_YAML)
    keep = S2.guard_adjust(r, mode=mode, drop=drop, add=add)
    print(f'  VISUAL GUARD (secondary, optional) — read these if you want to nudge the set:')
    print(f'    all classes : {r["scored_png"]}')
    print(f'    borderline  : {r["borderline_png"]}')
    # [v2] the guard is a VISION call, so it gets the physical signature verbatim (finding #13 /
    # subC.md). Judge every class on BOTH gates: quality (is it a good average?) and identity
    # (is it the target?). The signature decides the second one only.
    print('  ' + sig_block('which classes are the TARGET').replace('\n', '\n  '))
    if not keep:
        raise RuntimeError(f'{class2d_job.uid}: the ranker kept ZERO classes at mode={mode}. '
                           'Do not proceed — inspect the montage.')
    r['keep_after_guard'] = list(keep)      # evidence for decide_k (the selected-CLASS fraction)
    sel = S2.queue_select2d(cs, PROJECT_UID, ws.uid, class2d_job.uid, keep, title=title)
    remember(key, sel.uid)
    return sel, r


# ---------------------------------------------------------------------------
# 8b. DECISION helper — class2D_K is an ADAPTIVE knob, decided per round.
#     Full rule, sources and design: K_DECISION_SPEC.md next to this notebook.
#     Human directive 2026-08-20: "the K of 2D class should be decided based on run results,
#     not fixed" / "call LLM to analyse and define K during running based on particles number,
#     particles quality."
#     THREE things used to be one literal (a K ladder). They are now three separate sources of
#     truth: K = decide_k() ; the ROUND COUNT = the stop condition ONLY ; the SELECT STRICTNESS
#     = loose on round 1, tight once the stop condition fires, none in between.
# ---------------------------------------------------------------------------
K_RULE = (
 'RULE — apply it, do not invent one. Two independent sources; they agree.\n'
 '\n'
 'A. WHAT THE HISTORY BANK MEASURED (18,259 class_2D jobs across 163 projects that reached\n'
 '   better than 3.5 A; 5,107 with the input count resolvable; 35,713 round-to-round transitions):\n'
 '   * HOLD is the norm and the DEFAULT action. 61.5% of parent->child 2D transitions keep K\n'
 '     unchanged. Down (16.6%) and up (16.8%) are statistically identical. Median K_child/K_parent\n'
 '     is exactly 1.00 in EVERY bin - binned by the previous round selected-class fraction AND by\n'
 '     its particle keep-rate. 26% of projects used one K for every 2D job in the project.\n'
 '   * Particles-per-class is NOT a target and must not be solved for. Median 4,470 but p10-p90\n'
 '     spans 899 -> 22,444 (25x); individual jobs span 37 -> 206,170. corr(log N_in, log K) = 0.205,\n'
 '     slope 0.077: a 10x change in particle count buys only ~1.2x change in K. One project ran\n'
 '     K=200 on inputs from 62k to 4.87M. ppc is a CONSEQUENCE of holding K while N shrinks.\n'
 '   * K is a function of the pool ORIGIN, not of the previous round quality. High K (200-400) is\n'
 '     the fresh-or-newly-enlarged-pool setting: median input for K >= 200 is 918k with a median of\n'
 '     1 upstream 2D round; K=400 sits at median input 1.65M. Set K when the pool is new, then HOLD\n'
 '     it for every subsequent cleaning round on that branch.\n'
 '   * The one previous-round observable with measured signal is the SELECTED-CLASS FRACTION, and\n'
 '     it points UP. When a round kept 60-80% of its classes, precedents raised K 3x more often\n'
 '     than they lowered it (30% vs 10%). Read that as "the classification has stopped\n'
 '     discriminating - split finer, or stop cleaning", NEVER as "reduce K". A very low particle\n'
 '     keep-rate (< 0.2) also skews up (31% vs 17%).\n'
 '   * The ONLY empirically supported reason to LOWER K is a COLLAPSE IN N. The bank best map\n'
 '     (1.97 A) cut K 200 -> 100 -> 60 only after a round discarded ~89% of the particles\n'
 '     (369k -> 39k -> 30k -> 22k). K tracked the collapse in N, not the round index - and the same\n'
 '     project went back to K=200/400 on later fresh multi-million re-picks.\n'
 '   * Guard-rails from the measured distribution: do not let particles-per-class fall below\n'
 '     ~200-400 (the observed floor where practitioners stopped cleaning or re-merged), and do not\n'
 '     exceed ~20,000-25,000 on a first pass (round-1 p90 = 25,143).\n'
 '   * A RISING selected-class fraction round over round means STOP THE 2D LOOP, not shrink K\n'
 '     (GLUT5: 93/200 -> 147/200 -> 145/200 -> 130/200 at constant K=200 - the set getting clean\n'
 '     showed up in the select ratio, never in K).\n'
 '   * Descending ladders have ZERO support: 0 occurrences of 100->75->50 in 74,623 three-step\n'
 '     lineage triples; K=75 appears 13 times in 20,029 jobs and NEVER in a descending chain.\n'
 '     Strictly-descending 3-chains are 1.9% of all chains, strictly-ascending 1.7% - noise.\n'
 '   * Sanity context: K is bimodal - 200 (43%) and 100 (38%) cover 81% of all jobs; 39% of the\n'
 '     variance in K is project/operator habit, not dataset. Both dominant values produce best maps.\n'
 '\n'
 'B. WHAT THE MANUALS DOCUMENT (CryoSPARC guide + staff-participated forum):\n'
 '   * K is NOT a signal-budgeting knob. The guide holds signal-per-class constant with\n'
 '     class2D_num_full_iter_batchsize_per_class, not with K: "by choosing a number of particles\n'
 '     per class instead of the total number, the amount of information provided to each class is\n'
 '     held constant regardless of the number of classes." So K does NOT scale linearly with pool\n'
 '     size. K is a VIEW / JUNK-SEPARATION knob.\n'
 '   * The one documented trigger to RAISE K: junk visible INSIDE an otherwise-clean class - "when\n'
 '     class count is insufficient, junk particles may be grouped into good classes because there\n'
 '     is not adequate room for a class that is entirely junk." This failure mode is especially\n'
 '     live for a micelle-embedded particle, where junk and a small ordered core both sit against a\n'
 '     large low-contrast disc.\n'
 '   * NOT a trigger to lower K: near-duplicate classes showing the same view together with high\n'
 '     per-particle ESS - the guide calls this "typically not a concern". High ESS is a CONVERGENCE\n'
 '     signal; the documented fix is more online-EM iterations, not fewer classes.\n'
 '   * NO documented trigger to lower K exists. Empty/near-empty classes, class-size skew,\n'
 '     per-class resolution and ranker-score distributions were each checked and returned NO\n'
 '     SUGGESTION. The manuals supply NO reason to go down; the bank supplies exactly one (N\n'
 '     collapsed ~10x), and that is the only one you may use.\n'
 '   * Starting range 50-200 classes for a pool in the hundreds of thousands; a deliberate high-K\n'
 '     "finer" pass on a large merged pool is 100-300, ceiling ~400.\n'
 '   * Separate knob, do not conflate: batchsize_per_class 200-500 when SNR is low. Raising K does\n'
 '     NOT substitute for giving each class more signal.\n'
 '\n'
 'DECISION SPACE:\n'
 '  * "hold" is the DEFAULT and needs no trigger (use trigger "none_observed").\n'
 '  * "raise" requires ONE of: junk_in_clean_class (you must SEE it in the montage and name the\n'
 '    classes), select_fraction_high (selected-class fraction 0.60-0.80), keep_rate_very_low\n'
 '    (particle keep-rate < 0.20).\n'
 '  * "lower" is admissible on ONE trigger only: n_collapsed - the particle count has fallen\n'
 '    roughly an order of magnitude since the current K was set. "The set looks cleaner" is NOT a\n'
 '    reason to lower K; a cleaner set shows up in the select ratio, never in K. Any other\n'
 '    justification for lowering will be REFUSED by the caller and the value held.\n'
 '  * Justify k_next by naming the SPECIFIC observable it keys on, in "evidence".')

_K_SCHEMA = ('{"k_next": <int>, "direction": "hold|raise|lower", '
             '"trigger": "junk_in_clean_class|select_fraction_high|keep_rate_very_low|'
             'n_collapsed|none_observed", '
             '"evidence": "<the specific thing seen, e.g. classes 7,19,44 each show a clean TM disc '
             'with a streaked crescent at the rim>", '
             '"confidence": "high|medium|low", "why": "<one sentence>"}')


def k_memory_guard(K, box_px, lane=None, raise_on_fail=True, quiet=False):
    """HARD BOUND, checked BEFORE the job is queued. Returns (fits, k_max_for_this_lane).

    K x box^2 drives the 2D footprint. MEASURED on this cluster (memory: small_card_2d_oom):
    K=164 at box 208 = 7.10e6 OOM'd on the 11-12 GB small cards, while the reference's
    K=300 at box 150 = 6.75e6 ran. Big-K rounds belong on the A40 nodes.
    """
    lane = str(lane or LANE)
    box_px = int(box_px)
    kb2 = int(K) * box_px ** 2
    small = lane in SMALL_CARD_LANES
    # LANE-CORRECT ceiling. The small-card formula applies ONLY to SMALL_CARD_LANES; on an A40
    # lane the binding limit is K_CEILING. Returning the small-card number unconditionally put a
    # FALSE 'K <= 273' into decide_k's evidence text and suppressed a real junk_in_clean_class
    # up-trigger on an A40 node (P5/W23 round-4 blob2, 2026-08-21).
    k_fit = (max(K_MIN, int(KBOX2_MAX_SMALL_CARD // max(box_px ** 2, 1)))
             if small else K_CEILING)
    if not quiet:
        print(f'  memory guard: K={K} x box {box_px}^2 = {kb2 / 1e6:.2f}e6 on lane {lane} — '
              + (f'11-12 GB cards, limit {KBOX2_MAX_SMALL_CARD / 1e6:.2f}e6 -> K <= {k_fit} here'
                 if small else
                 f'not a small-card lane ({SMALL_CARD_LANES}), so the '
                 f'{KBOX2_MAX_SMALL_CARD / 1e6:.2f}e6 limit does not apply'))
    if small and kb2 > KBOX2_MAX_SMALL_CARD:
        msg = (f'REFUSING TO QUEUE: K={K} at box {box_px} px = {kb2 / 1e6:.2f}e6 exceeds the '
               f'{KBOX2_MAX_SMALL_CARD / 1e6:.2f}e6 K x box^2 that the {lane} cards have survived '
               f'(K=164 @ box 208 = 7.10e6 OOM\'d there). Do ONE of:\n'
               f'  (a) run this round on LANE = HOSTNAME = {BIG_K_LANE!r} (the A40 nodes) — this is '
               f'where big-K 2D belongs;\n'
               f'  (b) keep the lane and accept K <= {k_fit};\n'
               f'  (c) split the stack into sub-groups and classify them concurrently.\n'
               f'This is a HARD bound, not a suggestion — it is not clamped silently.')
        if raise_on_fail:
            raise KMemoryGuard(msg)
        if not quiet:
            print('!! ' + msg)
        return False, k_fit
    return True, k_fit


def k_clamp(K, why=''):
    """Clamp into [K_MIN, K_CEILING]. A clamp is PRINTED, never silent."""
    K0 = int(K)
    K = max(K_MIN, min(K_CEILING, K0))
    if K != K0:
        print(f'!! K CLAMPED {K0} -> {K} into [{K_MIN}, {K_CEILING}] {why}\n'
              f'   (below {K_MIN} CryoSPARC silently disables Force-max = a different algorithm; '
              f'above {K_CEILING} iteration time balloons and OOM is reported even on 24 GB cards)')
    return K


def k_round1_from_pool(n_pool, origin, tag='', box_px=None, lane=None):
    """Round 1 has no previous round, so K comes from the pool's ORIGIN — never from a ppc target.

    Band = the intersection of the guide's documented 50-200 starting range and the bank's measured
    practice (K>=200 median input 918k; K=400 median input 1.65M; 100 and 200 cover 81% of jobs).
    The reference's own round-1 values are printed as COMPARISON ONLY.
    """
    n = int(n_pool or 0)
    if n < K_POOL_SMALL_N:
        K, band = K_ROUND1_SMALL, f'pool < {K_POOL_SMALL_N:,}'
    elif n >= K_POOL_LARGE_N:
        K, band = K_ROUND1_LARGE, f'pool >= {K_POOL_LARGE_N:,}'
        if n / max(K, 1) > PPC_CEIL_FIRST_PASS:
            K, band = K_ROUND1_HUGE, (f'pool >= {K_POOL_LARGE_N:,} AND K={K_ROUND1_LARGE} would '
                                      f'still leave {n / K_ROUND1_LARGE:,.0f} particles/class, above '
                                      f'the {PPC_CEIL_FIRST_PASS:,} first-pass ceiling')
    else:
        K, band = K_ROUND1_DEFAULT, 'fresh/newly-enlarged pool (the default band)'
    K = k_clamp(K, f'(round-1 band for {tag})')
    print(f'K_DECISION [{tag} r1]: K={K}  source=POOL-ORIGIN BAND (no previous round to key on)\n'
          f'   pool = {n:,} particles, origin = {origin}\n'
          f'   band  : {band}\n'
          f'   -> {n / max(K, 1):,.0f} particles/class (a CONSEQUENCE of the band, never a target)\n'
          f'   ref (comparison only, NOT an input): zll_260130 ran r1 K=100 on each blob branch '
          f'(5.10M / 6.16M picks) and on topaz (1.33M), and K=300 on the 1.41M merged+deduped pool.')
    _k_softguards(n, K, first_pass=True)
    if box_px:
        k_memory_guard(K, box_px, lane, raise_on_fail=True)
    return {'k': K, 'direction': 'initial', 'trigger': 'pool_origin_band', 'evidence': band,
            'confidence': 'n/a', 'why': f'round-1 band for a pool of {n:,} from {origin}',
            'source': 'band', 'n_pool': n}


def _k_softguards(n_in, K, first_pass=False):
    """Two SOFT guard-rails from the measured distribution: printed, never enforced silently."""
    ppc = n_in / max(K, 1)
    if ppc < PPC_FLOOR_HIGH:
        print(f'   !! soft guard-rail: {ppc:,.0f} particles/class is below the ~{PPC_FLOOR_LOW}-'
              f'{PPC_FLOOR_HIGH} floor where practitioners stopped cleaning or re-merged. The fix '
              f'is to STOP the loop or re-merge — NOT to cut K.')
    if first_pass and ppc > PPC_CEIL_FIRST_PASS:
        print(f'   !! soft guard-rail: {ppc:,.0f} particles/class is above the first-pass ceiling '
              f'(~{PPC_CEIL_FIRST_PASS:,}, round-1 p90) — K is low for this pool.')
    if K not in K_BIMODAL_PRIOR and abs(K - K_BIMODAL_PRIOR[0]) > 25 and abs(K - K_BIMODAL_PRIOR[1]) > 25:
        print(f'   (note: K={K} is near neither {K_BIMODAL_PRIOR[0]} nor {K_BIMODAL_PRIOR[1]}, which '
              f'cover 81% of the bank — worth a second look unless a trigger justified it.)')


def _n_selected_classes(sel_job, rank_result=None):
    """How many CLASSES the select kept — the selected-class fraction is the one measured signal."""
    try:
        return n_items(sel_job, resolve_out(sel_job, 'templates_selected', 'templates',
                                            'class_averages'))
    except Exception:
        pass
    if rank_result and rank_result.get('keep_after_guard') is not None:
        return len(rank_result['keep_after_guard'])
    return None


def _k_evidence(proj, tag, history, class2d_job=None, rank_result=None):
    """Assemble the evidence table decide_k() keys on — all of it from the completed round."""
    ev = {'tag': tag, 'rounds': history, 'montage': None, 'classes': None, 'scores': None}
    if rank_result:
        ev['montage'] = rank_result.get('scored_png')
        sc = sorted(float(v) for v in (rank_result.get('scores') or {}).values())
        if sc:
            ev['scores'] = {
                'n': len(sc), 'min': sc[0], 'max': sc[-1], 'median': statistics.median(sc),
                'min_score_cut': rank_result.get('min_score'),
                'n_below_cut': sum(1 for s in sc
                                   if s < float(rank_result.get('min_score') or 0)),
                'deciles': [round(sc[int(q * (len(sc) - 1))], 3) for q in
                            (0.0, 0.1, 0.25, 0.5, 0.75, 0.9, 1.0)]}
    if class2d_job is not None:
        try:
            ci = S2.build_class_info(proj, class2d_job.uid)
            ns = sorted(int(c['n_particles']) for c in ci)
            rs = sorted(float(c['res_A']) for c in ci if c.get('res_A'))
            mean_n = (sum(ns) / len(ns)) if ns else 0
            ev['classes'] = {
                'K_actual': len(ci), 'sizes_min': ns[0] if ns else None,
                'sizes_p10': ns[int(0.1 * (len(ns) - 1))] if ns else None,
                'sizes_median': statistics.median(ns) if ns else None,
                'sizes_p90': ns[int(0.9 * (len(ns) - 1))] if ns else None,
                'sizes_max': ns[-1] if ns else None,
                'n_near_empty': sum(1 for x in ns if x < 0.10 * mean_n),
                'res_best': rs[0] if rs else None,
                'res_median': statistics.median(rs) if rs else None,
                'res_worst': rs[-1] if rs else None}
        except Exception as e:                   # noqa: BLE001 — evidence is best-effort
            print(f'   (per-class metadata unavailable for {class2d_job.uid}: {e!r})')
    return ev


def _k_collapse_check(history, k_current):
    """Has N fallen ~an order of magnitude SINCE the current K was set? The one down-trigger.

    [v2] finding #7 — the RATIO alone measures shrinkage-from-origin, not current state. On the
    generalisation test blob1 (7.77x -> HOLD) and blob2 (8.67x -> LOWER K) straddled the 8.0x bar
    while their pools were in nearly identical condition (378,983 vs 414,505 particles; 1,263 vs
    1,382 particles/class at K=300); the split came from a 22% difference in their ROUND-1 ORIGIN
    pools, an artefact of the two picker bands. So the trigger now needs an ABSOLUTE corroboration
    as well: the branch must ALSO be below the particles-per-class floor where the one measured
    precedent actually cut K (369k -> 39k at K=200 is ppc 195). BOTH conditions print.
    """
    n_at_set, set_round = None, None
    for h in history:
        if h.get('K') == k_current and n_at_set is None:
            n_at_set, set_round = h.get('n_in'), h.get('round')
    n_now = history[-1].get('n_out') if history else None
    if not n_at_set or not n_now:
        return {'collapsed': False, 'ratio': None, 'n_at_set': n_at_set, 'n_now': n_now,
                'set_round': set_round, 'ppc_now': None, 'ratio_ok': False, 'ppc_ok': False}
    ratio = n_at_set / max(n_now, 1)
    ppc_now = n_now / max(int(k_current), 1)
    ratio_ok = ratio >= N_COLLAPSE_FACTOR
    ppc_ok = ppc_now < N_COLLAPSE_PPC_MAX
    return {'collapsed': bool(ratio_ok and ppc_ok), 'ratio': ratio, 'n_at_set': n_at_set,
            'n_now': n_now, 'set_round': set_round, 'ppc_now': ppc_now,
            'ratio_ok': ratio_ok, 'ppc_ok': ppc_ok}


def _k_evidence_text(ev, k_current, collapse, box_px, lane):
    lines = ['ROUNDS SO FAR (this branch/pool only):',
             '  round  K   select  n_in         n_out        kept%   classes_kept/K  class_frac']
    for h in ev['rounds']:
        cf = h.get('class_frac')
        lines.append(f'  {h["round"]:<6d} {h.get("K", 0):<3d} {h.get("mode", "?"):<7s} '
                     f'{h.get("n_in", 0):<12,d} {h.get("n_out", 0):<12,d} '
                     f'{100 * h.get("frac", 0):<6.1f}  '
                     f'{str(h.get("n_classes_kept")) + "/" + str(h.get("K")):<15s} '
                     f'{("%.3f" % cf) if cf is not None else "n/a"}')
    c = ev.get('classes')
    if c:
        lines += ['', 'PER-CLASS PARTICLE COUNTS (round just finished):',
                  f'  K actually reported = {c["K_actual"]}; sizes min/p10/median/p90/max = '
                  f'{c["sizes_min"]:,} / {c["sizes_p10"]:,} / {c["sizes_median"]:,.0f} / '
                  f'{c["sizes_p90"]:,} / {c["sizes_max"]:,}',
                  f'  near-empty classes (< 10% of the mean class size) = {c["n_near_empty"]}',
                  f'  per-class resolution best/median/worst = {c["res_best"]} / '
                  f'{c["res_median"]} / {c["res_worst"]} A']
    s = ev.get('scores')
    if s:
        lines += ['', 'RELION class_ranker SCORE DISTRIBUTION (round just finished):',
                  f'  n={s["n"]}, min={s["min"]:.3f}, median={s["median"]:.3f}, max={s["max"]:.3f}, '
                  f'cut={s["min_score_cut"]}, below cut={s["n_below_cut"]}',
                  f'  quantiles [0,10,25,50,75,90,100]% = {s["deciles"]}',
                  '  (a clearly SEPARATED low-score mode = a junk mode still present)']
    lines += ['', 'N-COLLAPSE CHECK (the only admissible reason to LOWER K):',
              f'  current K={k_current} was set at round {collapse["set_round"]} with '
              f'n_in={collapse["n_at_set"]:,}' if collapse.get('n_at_set') else
              '  current K was not set within this loop (no in-loop comparison available)']
    if collapse.get('ratio'):
        lines.append(f'  n now = {collapse["n_now"]:,}  ->  fallen {collapse["ratio"]:.2f}x '
                     f'(the rule needs >= {N_COLLAPSE_FACTOR:.0f}x; measured precedent 369k -> 39k '
                     f'= 9.5x)  ratio_ok = {collapse["ratio_ok"]}')
        lines.append(f'  ABSOLUTE corroboration (required as well): {collapse["ppc_now"]:,.0f} '
                     f'particles/class now, the rule needs < {N_COLLAPSE_PPC_MAX} (the precedent '
                     f'cut K at ppc 195)  ppc_ok = {collapse["ppc_ok"]}')
        lines.append(f'  -> n_collapsed = {collapse["collapsed"]}  (the ratio alone measures '
                     f'shrinkage FROM THE ORIGIN, not current state — two branches in the same '
                     f'condition straddled the bar on the reference test purely because their '
                     f'round-1 pools differed by 22%)')
    lines += ['', 'BOUNDS THAT WILL BE ENFORCED ON YOUR ANSWER:',
              f'  K_MIN={K_MIN}, K_CEILING={K_CEILING} (clamped, and the clamp is printed)',
              f'  memory guard on lane {lane} at box {box_px} px: K <= '
              f'{k_memory_guard(k_current, box_px, lane, raise_on_fail=False, quiet=True)[1]}',
              '  a "lower" without trigger n_collapsed is REFUSED and the value held.']
    if ev.get('montage'):
        lines += ['', 'THE MONTAGE (attached): every class average of the round just finished, with '
                  'its class_ranker score and a keep/reject border. This is where the ONE documented '
                  'up-trigger lives: junk sitting INSIDE an otherwise-clean class (a clean particle '
                  'average with a streak/blob/edge riding along in the same average). Name the '
                  'classes if you see it.']
    else:
        lines += ['', '!! NO MONTAGE AVAILABLE for this round (the select was resumed from a '
                  'previous run, so the ranker render was not regenerated). The one documented '
                  'up-trigger is VISUAL, so without it prefer "hold" unless a numeric trigger is '
                  'unambiguous.']
    return '\n'.join(lines)


def decide_k(proj, tag, next_round, history, k_current, class2d_job=None, rank_result=None,
             box_px=None, lane=None, pool_note=''):
    """DECISION — class2D_K for the NEXT round, from the round that just finished.

    LLM primary (the same one transport as every other decision cell), deterministic fallback,
    loud about which one it used. Returns a decision record that goes into history[].
    """
    k_current = int(k_current)
    ev = _k_evidence(proj, tag, history, class2d_job, rank_result)
    collapse = _k_collapse_check(history, k_current)
    box_px = int(box_px or 0) or 1
    lane = str(lane or LANE)
    ev_text = _k_evidence_text(ev, k_current, collapse, box_px, lane)
    print(f'\nK_DECISION [{tag} r{next_round}] — evidence for the next round (current K={k_current}):')
    print(ev_text)

    def _hold(reason, source):
        print(f'K_DECISION: held at {k_current} — {reason}')
        return {'k': k_current, 'direction': 'hold', 'trigger': 'none_observed',
                'evidence': reason, 'confidence': 'n/a', 'why': reason, 'source': source,
                'round': next_round}

    prompt = (f'You are choosing class2D_K for the NEXT round of iterative 2D cleaning in '
              f'CryoSPARC.\n\nCONTEXT: branch/pool "{tag}", about to run round {next_round}. '
              f'The CURRENT K is {k_current}. {pool_note}\n'
              + sig_block('what counts as junk INSIDE an otherwise-clean class average')
              + f'\n{K_RULE}\n\nEVIDENCE FROM THE RUN:\n{ev_text}\n')
    try:
        ans = _llm_json(prompt, image_paths=[ev['montage']] if ev.get('montage') else None,
                        schema_hint=_K_SCHEMA, label=f'decide_k {tag} r{next_round}')
    except LLMUnavailable as e:
        print(f'!! LLM DECISION UNAVAILABLE: {e}')
        return _hold(f'no vision available, no documented numeric trigger. '
                     f'(The one documented adjustment trigger is VISUAL, so with no read of the '
                     f'montage there is no sourced basis to move K. The loop continues on its '
                     f'numeric stop condition alone; K is never silently descended.)', 'fallback')

    direction = str(ans.get('direction', 'hold')).lower()
    trigger = str(ans.get('trigger', 'none_observed')).lower()
    try:
        k_next = int(ans.get('k_next', k_current))
    except Exception:                             # noqa: BLE001
        k_next = k_current
    print(f'   model answer: {json.dumps(ans)[:600]}')

    refusal = None
    if direction == 'lower':
        if trigger != 'n_collapsed':
            refusal = (f'"lower" was justified by trigger {trigger!r}. The ONLY admissible reason '
                       f'to lower K is n_collapsed (~{N_COLLAPSE_FACTOR:.0f}x drop in N since this K '
                       f'was set). Neither the manuals nor the bank document any other; "the set '
                       f'looks cleaner" is explicitly not one.')
        elif not collapse['collapsed']:
            refusal = (f'trigger n_collapsed is not supported by the numbers: N fell '
                       f'{(collapse["ratio"] or 1):.2f}x since K={k_current} was set, and the rule '
                       f'needs >= {N_COLLAPSE_FACTOR:.0f}x.')
        elif k_next >= k_current:
            refusal = f'direction "lower" but k_next {k_next} >= current {k_current} — inconsistent.'
    elif direction == 'raise':
        if trigger == 'junk_in_clean_class':
            if not ev.get('montage'):
                refusal = ('trigger junk_in_clean_class is a VISUAL observation, but no montage was '
                           'available for this round, so it cannot have been observed.')
        elif trigger == 'select_fraction_high':
            cf = history[-1].get('class_frac')
            if cf is None or not (SELECT_FRAC_HIGH_BAND[0] <= cf <= SELECT_FRAC_HIGH_BAND[1]):
                refusal = (f'trigger select_fraction_high needs the selected-class fraction in '
                           f'{SELECT_FRAC_HIGH_BAND}; measured {cf}.')
        elif trigger == 'keep_rate_very_low':
            fr = history[-1].get('frac')
            if fr is None or fr >= KEEP_RATE_VERY_LOW:
                refusal = (f'trigger keep_rate_very_low needs a particle keep-rate < '
                           f'{KEEP_RATE_VERY_LOW}; measured {fr}.')
        else:
            refusal = (f'"raise" was justified by trigger {trigger!r}, which is not one of the three '
                       f'documented up-triggers.')
        if refusal is None and k_next <= k_current:
            refusal = f'direction "raise" but k_next {k_next} <= current {k_current} — inconsistent.'
    elif direction == 'hold':
        k_next = k_current
    else:
        refusal = f'unrecognised direction {direction!r}.'

    if refusal:
        print(f'!! K DECISION REFUSED: {refusal}')
        return _hold('the model\'s proposal was refused by the rule (above); holding is the '
                     'default action', 'llm-refused')

    k_next = k_clamp(k_next, f'(proposed by the model for {tag} r{next_round})')
    fits, k_fit = k_memory_guard(k_next, box_px, lane, raise_on_fail=False, quiet=True)
    if not fits:
        print(f'!! the proposed K={k_next} would exceed the K x box^2 memory guard on lane {lane} '
              f'(max K here = {k_fit}). NOT queueing it and NOT silently clamping the science knob: '
              f'holding at {k_current}. To honour the decision, move this loop to LANE = '
              f'{BIG_K_LANE!r} and re-run the cell.')
        return _hold(f'proposed K={k_next} exceeds the memory guard on lane {lane}', 'llm-refused')
    _k_softguards(history[-1].get('n_out', 0), k_next, first_pass=False)
    print(f'K_DECISION: {direction.upper()} -> K={k_next} (was {k_current})  '
          f'trigger={trigger}  confidence={ans.get("confidence")}\n'
          f'   evidence: {ans.get("evidence")}\n'
          f'   why     : {ans.get("why")}\n'
          f'   ref (comparison only): zll_260130 went 100 -> 75 -> 50 per branch and 300 post-merge; '
          f'that sequence has 0 occurrences in 74,623 bank lineage triples and is NOT an input here.')
    return {'k': k_next, 'direction': direction, 'trigger': trigger,
            'evidence': ans.get('evidence'), 'confidence': ans.get('confidence'),
            'why': ans.get('why'), 'source': 'llm', 'round': next_round}


def stop_2d_loop(tag, history):
    """The ONLY thing that decides how many rounds a 2D clean loop runs.

    Returns (stop, reason, hard). hard=True -> break NOW; hard=False -> run ONE final "tight"
    round and then stop. Nothing here depends on K, and K depends on nothing here.
    """
    if not history:
        return False, '', False
    h = history[-1]
    if len(history) >= MAX_2D_ROUNDS_SAFETY:
        return True, (f'MAX_2D_ROUNDS_SAFETY={MAX_2D_ROUNDS_SAFETY} runaway guard (NOT the '
                      f'reference count, which was 3)'), True
    if h['n_out'] < MIN_PARTICLES_FOR_3D:
        return True, (f'{h["n_out"]:,} < MIN_PARTICLES_FOR_3D={MIN_PARTICLES_FOR_3D:,} — further '
                      f'cleaning would starve the 3D stage'), True
    if h['frac'] >= PLATEAU_FRAC:
        return True, f'kept-fraction plateau ({h["frac"]:.3f} >= {PLATEAU_FRAC})', False
    # [v2] finding #9 — class_frac is SELECT-MODE DEPENDENT (loose / none / tight apply different
    # cuts), so a tight round is NOT comparable to a loose one. On the generalisation test round
    # 5's drop to 0.460 / 0.545 was an artefact of the stricter tight rule, not a quality
    # regression. Compare a round only against the most recent round run in the SAME mode.
    cur = h.get('class_frac')
    prev_same = next((p for p in reversed(history[:-1])
                      if p.get('mode') == h.get('mode') and p.get('class_frac') is not None), None)
    if cur is not None and prev_same is not None:
        prev = prev_same['class_frac']
        rise = cur - prev
        # [v2] finding #8 — a BIG rise qualifies at the SOFTER floor. blob1 came in at 0.570 with a
        # +0.257 rise, three points short of the uniform 0.60 floor, and ran another round for
        # nothing; blob2 (0.477) is under either floor, so a genuinely dirty branch keeps cleaning.
        floor = (CLASS_FRAC_CLEAN_MIN_SOFT if rise > CLASS_FRAC_BIG_RISE else CLASS_FRAC_CLEAN_MIN)
        if rise > CLASS_FRAC_RISE_EPS and cur >= floor:
            return True, (f'selected-class fraction RISING ({prev:.3f} -> {cur:.3f}, +{rise:.3f}, '
                          f'>= floor {floor} [{"SOFT floor, big rise" if floor == CLASS_FRAC_CLEAN_MIN_SOFT else "standard floor"}], '
                          f'both rounds in select mode {h.get("mode")!r}) — the bank\'s "the branch '
                          f'is clean, stop cleaning" signal (GLUT5 93/200 -> 147/200 -> 145/200 at '
                          f'constant K). A rising select ratio is a reason to STOP, never to shrink '
                          f'K'), False
    elif cur is not None and len(history) >= 2:
        print(f'   (no same-mode predecessor for the rising-class-fraction test: this round is '
              f'{h.get("mode")!r} and the previous rounds are '
              f'{[p.get("mode") for p in history[:-1]]} — class_frac is select-mode dependent, so a '
              f'cross-mode comparison would be meaningless. Loop continues on its other conditions.)')
    return False, '', False


# ---------------------------------------------------------------------------
# 8c. [v2] ONE 2D ROUND — with >1M SUBGROUP SPLITTING (finding #6)
#     Project convention (memory: split_large_2d_into_subgroups): a stack over ~1M particles is
#     split into subgroups that classify CONCURRENTLY on free GPUs, under ONE global ranker
#     threshold, with the selections merged afterwards (the split is disjoint, so no dedup). The
#     distilled pipeline had no size-aware logic except the memory guard, so the generalisation
#     test ran ~3M as a single job per branch — round 1 alone took ~3h50m.
#     Concurrency is bounded by DISJOINT GPU ids per subgroup: an explicit gpus= list is NOT an
#     exclusive reservation against your own jobs (J1387/J1388, J1431/J1432 both co-scheduled).
# ---------------------------------------------------------------------------
def plan_subgroups(n_in, n_gpus=None):
    n_gpus = len(n_gpus) if isinstance(n_gpus, (list, tuple)) else int(n_gpus or len(GPUS_MULTI))
    if n_in <= SPLIT_2D_OVER_N:
        return 1
    want = int(math.ceil(n_in / float(SPLIT_2D_TARGET_N)))
    return max(1, min(want, SPLIT_2D_MAX_JOBS, n_gpus))


def _gpu_chunks(gpus, n):
    """Split the GPU ids into n DISJOINT groups, using every id (the remainder is spread, not
    dropped). Disjoint is the point: an explicit gpus= list is NOT an exclusive reservation
    against your own jobs — two of this project's own jobs have twice been co-scheduled onto the
    same physical cards (J1387/J1388, J1431/J1432)."""
    gpus = list(gpus)
    n = max(1, min(int(n), len(gpus)))
    base, extra = divmod(len(gpus), n)
    out, i = [], 0
    for k in range(n):
        take = base + (1 if k < extra else 0)
        out.append(gpus[i:i + take])
        i += take
    return out


def class2d_round(cs, proj, ws, tag, rnd, src_uid, src_group, K, box_px, mode,
                  extra_params=None, gpus=None, lane=None, min_score=None):
    """ONE round: class_2D_new -> relion_rank_select -> select_2D, split into concurrent
    subgroups when the pool is over SPLIT_2D_OVER_N. Returns everything the loop + decide_k need."""
    gpus = list(gpus or GPUS_MULTI)
    lane = lane or LANE
    src_job = proj.find_job(src_uid)
    n_in = n_items(src_job, src_group)
    n_sub = plan_subgroups(n_in, len(gpus))
    k_memory_guard(K, box_px, lane, raise_on_fail=True)

    if n_sub == 1:
        sources = [(src_uid, src_group)]
        chunks = [gpus]
    else:
        print(f'  SUBGROUP SPLIT: {n_in:,} particles > SPLIT_2D_OVER_N={SPLIT_2D_OVER_N:,} '
              f'-> {n_sub} concurrent class_2D jobs of ~{n_in // n_sub:,} each, ONE global ranker '
              f'threshold, selections merged after (disjoint split, no dedup needed).')
        sp = run_job(f'{tag}_split_r{rnd}', proj, ws, 'particle_sets',
                     {'particles_A': (src_uid, src_group)},
                     {'set_operation': 'split', 'set_split_num': n_sub, 'set_split_random': True},
                     f'{tag} split r{rnd} into {n_sub} subgroups', wait=True)
        require_completed(sp, 'the split must finish before its subgroups can be classified')
        groups = [g for g in live_outputs(sp)
                  if g.startswith('split_') or g.startswith('particles_set')]
        groups = sorted(groups)[:n_sub]
        if len(groups) < n_sub:
            raise KeyError(f'{sp.uid}: expected {n_sub} split output groups, found {groups} '
                           f'(live groups = {live_outputs(sp)})')
        sources = [(sp.uid, g) for g in groups]
        chunks = _gpu_chunks(gpus, n_sub)
        for (u, g), ch in zip(sources, chunks):
            print(f'    subgroup {g}: {n_items(sp, g):,} particles -> gpus {ch}')

    # ---- queue them all (concurrently when split), then poll each -------------
    jobs = []
    for i, ((u, g), ch) in enumerate(zip(sources, chunks)):
        params = {'class2D_K': K, 'compute_use_ssd': COMPUTE_USE_SSD,
                  'compute_num_gpus': len(ch)}
        params.update(extra_params or {})
        suffix = '' if n_sub == 1 else f'_s{i + 1}'
        j = run_job(f'{tag}_class2d_r{rnd}{suffix}', proj, ws, 'class_2D_new',
                    {'particles': (u, g)}, params,
                    f'{tag} class_2D r{rnd}{suffix} (K={K}'
                    + (f', subgroup {i + 1}/{n_sub}' if n_sub > 1 else '') + ')',
                    gpus=ch, lane=lane,
                    wait=(WAIT_FOR_JOBS and n_sub == 1),
                    admission_mib=predicted_gpu_mib(K, box_px, len(ch)))
        jobs.append(j)
    if n_sub > 1 or not WAIT_FOR_JOBS:
        for j in jobs:
            if WAIT_FOR_JOBS:
                poll(j)
            else:
                require_completed(j, 'queue-and-return mode: re-run this cell when it is done')

    # ---- select each subgroup under ONE global threshold ---------------------
    sels, rks, n_cls = [], [], 0
    for i, j in enumerate(jobs):
        suffix = '' if n_sub == 1 else f'_s{i + 1}'
        sel, rk = rank_select(cs, proj, ws, j, mode, f'{tag}_select_r{rnd}{suffix}',
                              f'{tag} select_2D r{rnd}{suffix} ({mode}) [RELION ranker + guard]',
                              min_score=min_score)
        sels.append(sel)
        rks.append(rk)
        c = _n_selected_classes(sel, rk)
        n_cls += (c or 0)

    if len(sels) == 1:
        out_job = sels[0]
    else:
        out_job = run_job(f'{tag}_selmerge_r{rnd}', proj, ws, 'remove_duplicate_particles',
                          {'particles': [(s.uid, resolve_out(s, 'particles_selected', 'particles'))
                                         for s in sels]},
                          {},   # union tool; the subgroups are DISJOINT, so ~0 duplicates expected
                          f'{tag} merge subgroup selections r{rnd} ({len(sels)} subgroups)',
                          wait=True)
        require_completed(out_job)
    out_group = resolve_out(out_job, 'particles_kept', 'particles_selected', 'particles')
    n_out = n_items(out_job, out_group)

    # K_total MUST come from the jobs that ACTUALLY RAN, not from the loop's in-memory K. On a
    # RESUMED round decide_k() is re-invoked and is non-deterministic, so the loop's K can diverge
    # from the K the resumed job was created with. MEASURED 2026-09-04: blob r4 ran J2128 at K=400
    # while the loop believed 300 -> class_frac recorded 136/300=0.453 instead of 136/400=0.340.
    # class_frac feeds stop_2d_loop's RISING test, so a wrong denominator can manufacture or
    # suppress a rise and CHANGE A STOP. Ground truth is the job's own params_spec.
    def _actual_K(job):
        try:
            spec = proj.find_job(job.uid).doc.get('params_spec', {}) or {}
            v = (spec.get('class2D_K') or {}).get('value')
            return int(v) if v else None
        except Exception:
            return None

    _ks = [_actual_K(j) for j in jobs]
    if all(k is not None for k in _ks):
        K_total = sum(_ks)
        if K_total != K * len(jobs):
            print(f'  !! K DIVERGENCE on a RESUMED round: the loop believes K={K} x {len(jobs)} '
                  f'= {K * len(jobs)}, but the jobs actually ran {_ks} = {K_total}. '
                  f'Using the JOBS as ground truth for class_frac.')
    else:
        K_total = K * len(jobs)
        print(f'  !! could not read class2D_K back from {[j.uid for j in jobs]}; '
              f"falling back to the loop's K={K} x {len(jobs)}. class_frac may be unreliable.")
    return {'n_in': n_in, 'n_out': n_out, 'n_sub': len(jobs), 'K_total': K_total,
            'n_classes_kept': n_cls or None,
            'class_frac': (n_cls / K_total) if n_cls else None,
            'class_job': jobs[0], 'class_jobs': jobs, 'selects': sels,
            'rank_result': rks[0], 'out_job': out_job, 'out_group': out_group}


def clean_2d_loop(cs, proj, ws, tag, src_uid, src_group, dims, pool_origin=None,
                  gpus=None, extra_params=None):
    """LOOP A — one branch's own 2D clean loop. GENUINELY UNBOUNDED, ended only by stop_2d_loop().

    Three decoupled sources of truth (see K_DECISION_SPEC.md):
      K              <- k_round1_from_pool() on round 1, then decide_k() after every round
      round count    <- stop_2d_loop() ONLY
      select mode    <- 'loose' on round 1, 'tight' on the round the stop condition ends on
    Reference, as a COMPARISON ONLY: 3 rounds per branch at K 100/75/50; round 1 kept 17-22% on
    blob and 83% on topaz, rounds 2-3 kept 77-86%. Neither that count nor those K values is used.
    """
    box_px = int(dims.get('box_bin_px') or dims.get('box_full_px'))
    lane = LANE
    history, cur_uid, cur_group = [], src_uid, src_group
    selects = []
    try:
        n_pool = n_items(proj.find_job(src_uid), src_group)
    except Exception:                              # noqa: BLE001
        n_pool = 0
    k_dec = k_round1_from_pool(n_pool, pool_origin or f'fresh picks for branch {tag}',
                               tag=tag, box_px=box_px, lane=lane)
    i = 0
    while True:
        stop, reason, hard = stop_2d_loop(tag, history)
        if stop and hard:
            print(f'!! {tag} STOP: {reason}')
            break
        is_final, final_reason = stop, reason
        mode = 'loose' if i == 0 else ('tight' if is_final else 'none')
        K = int(k_dec['k'])
        R = class2d_round(cs, proj, ws, tag, i + 1, cur_uid, cur_group, K, box_px, mode,
                          extra_params=extra_params, gpus=gpus or GPUS_MULTI, lane=lane)
        c, rk = R['class_job'], R['rank_result']
        n_in, n_out = R['n_in'], R['n_out']
        frac = n_out / max(n_in, 1)
        n_cls, class_frac = R['n_classes_kept'], R['class_frac']
        history.append({'round': i + 1, 'K': K, 'mode': mode, 'class2d': c.uid,
                        'select': R['out_job'].uid, 'n_sub': R['n_sub'], 'K_total': R['K_total'],
                        'n_in': n_in, 'n_out': n_out, 'frac': frac,
                        'n_classes_kept': n_cls, 'class_frac': class_frac,
                        'k_decision': k_dec})
        selects.append(R['out_job'])
        print(f'  {tag} round {i + 1}: K={K} ({k_dec["source"]}/{k_dec["direction"]}'
              f'{"" if k_dec.get("trigger") in (None, "none_observed") else ", trigger=" + str(k_dec["trigger"])}) '
              f'mode={mode}  {n_in:,} -> {n_out:,}  (kept {frac * 100:.1f}%)  classes '
              f'{n_cls if n_cls is not None else "?"}/{R["K_total"]}'
              f'{"" if class_frac is None else " = %.3f" % class_frac}'
              f'{"" if R["n_sub"] == 1 else "  [%d concurrent subgroups]" % R["n_sub"]}\n'
              f'      [ref, comparison only: r1 blob 17-22% / topaz 83%; r2-r3 77-86%. '
              f'TEST: r1 blob 41.9-49.0% / topaz 85.0%]')
        cur_uid, cur_group = R['out_job'].uid, R['out_group']
        stop, reason, hard = stop_2d_loop(tag, history)
        if is_final:
            print(f'  {tag} STOP: final ("tight") round done — {final_reason}.')
            break
        if stop and hard:
            print(f'!! {tag} STOP: {reason}')
            break
        k_dec = decide_k(proj, tag, i + 2, history, K, class2d_job=c, rank_result=rk,
                         box_px=box_px, lane=lane,
                         pool_note=f'This is a per-picker cleaning branch ({tag}); its pool has not '
                                   f'been enlarged since round 1.')
        i += 1
    print(f'  {tag} K TRAJECTORY AS DECIDED: '
          + ' -> '.join(f'{h["K"]}({h["k_decision"]["source"]})' for h in history)
          + '   [ref, comparison only: 100 -> 75 -> 50]')
    return {'tag': tag, 'history': history, 'selects': selects,
            'final_uid': cur_uid, 'final_group': cur_group,
            'final_select': selects[-1] if selects else None}


# ---------------------------------------------------------------------------
# 8d. [v2] DECISION helper — topaz_extract `particle_threshold` (finding #10)
#     An ADAPTIVE knob that shipped as a literal default. It is the ONLY gate that limits what
#     topaz_extract writes out, and its right value is a property of the TRAINED MODEL, not of the
#     sample: on the SAME micrographs, one model at the default 0 gave 144.5 picks/mic and another
#     gave 5.2/mic; ours would have given 271.94/mic. The default can also return ~1 particle per
#     micrograph while the job still reports "completed".
#
#     THE SUB-TRAP (it cost a failed job): particle_threshold gates on TOPAZ'S RAW LOG-ODDS SCORE
#     — the `score` column of <job_dir>/topaz_particles_prediction.txt, here -6.000 ... +9.826 —
#     NOT on CryoSPARC's pick_stats/ncc_score (positive, to ~158 on the same model). A threshold
#     derived from the wrong column (78.4) was a plausible mid-range number on the wrong axis, and
#     the job "completed" with ZERO particles. GENERALISE: the score a picker reports on its
#     OUTPUT and the score its own gating PARAMETER filters on can be different quantities.
# ---------------------------------------------------------------------------
def topaz_raw_scores(proj, diag_job):
    """The RAW topaz score column of the diagnostic pass. The ONLY axis particle_threshold acts on."""
    import numpy as np                                 # noqa: PLC0415
    jdir = Path(proj.dir()) / diag_job.uid
    cands = sorted(jdir.glob('topaz_particles_prediction*.txt')) or sorted(jdir.glob('*prediction*.txt'))
    if not cands:
        raise FileNotFoundError(
            f'no topaz_particles_prediction.txt under {jdir}. That file is the ONLY place the raw '
            f'score lives, and particle_threshold gates on THAT column — not on the '
            f'pick_stats/ncc_score CryoSPARC shows on the output particles. Without it, set '
            f'TOPAZ_PARTICLE_THRESHOLD in cell 2 by hand; do NOT re-derive from ncc_score and do '
            f'NOT fall back to the default 0.')
    f = cands[-1]
    sc = np.loadtxt(str(f), skiprows=1, usecols=3, dtype='float32')
    print(f'  raw topaz scores from {f.name}: {sc.size:,} candidates, '
          f'min {sc.min():.4f}, max {sc.max():.4f}  '
          f'(this is the LOG-ODDS axis; CryoSPARC\'s ncc_score is a DIFFERENT quantity)')
    return sc


def decide_topaz_threshold(proj, diag_job, n_mics, target_ppm=None, blob_ppm=None):
    """DECISION — particle_threshold for the FINAL topaz_extract, from the model's own scores."""
    import numpy as np                                 # noqa: PLC0415
    sc = topaz_raw_scores(proj, diag_job)
    target = target_ppm or TOPAZ_TARGET_PICKS_PER_MIC or blob_ppm or 150.0
    target = float(min(max(target, MIN_PICKS_PER_MIC * 2.0), TOPAZ_MAX_PICKS_PER_MIC))
    if target_ppm is None and TOPAZ_TARGET_PICKS_PER_MIC is None:
        print(f'  target picks/mic = {target:.1f} — anchored on '
              + (f'the BLOB branch on THIS dataset ({blob_ppm:.1f}/mic)' if blob_ppm
                 else 'the default 150/mic') +
              ', clamped into [%.0f, %d]. (ref topaz 190/mic; TEST 144.5/mic — comparisons only.)'
              % (MIN_PICKS_PER_MIC * 2.0, TOPAZ_MAX_PICKS_PER_MIC))
    order = np.sort(sc)[::-1]
    want = int(round(target * n_mics))
    print(f'  {"threshold":>12s} {"kept":>14s} {"picks/mic":>10s}')
    for q in (0.999, 0.99, 0.95, 0.90, 0.75, 0.50, 0.25, 0.10):
        t = float(np.quantile(sc, 1 - q)) if q < 1 else float(sc.min())
        k = int((sc >= t).sum())
        print(f'  {t:12.3f} {k:14,d} {k / max(n_mics, 1):10.2f}')
    if want >= order.size:
        thr = float(order[-1])
        print(f'!! the target ({want:,} particles) is at or above every candidate topaz found '
              f'({order.size:,}) — taking the floor {thr:.3f}, which keeps everything.')
    else:
        thr = float(order[want])
    kept = int((sc >= thr).sum())
    ppm = kept / max(n_mics, 1)
    print(f'  -> particle_threshold = {thr:.3f}  predicts {kept:,} particles = {ppm:.2f}/mic')
    # ---- HARD BOUNDS, checked before anything is queued ----------------------
    if not (sc.min() <= thr <= sc.max()):
        raise ValueError(f'REFUSED: {thr} is outside this model\'s raw score range '
                         f'[{sc.min():.3f}, {sc.max():.3f}]. This is exactly the J1423 failure — a '
                         f'threshold on the wrong axis "completes" with ZERO particles.')
    if ppm < MIN_PICKS_PER_MIC:
        raise ValueError(f'REFUSED: {ppm:.2f} picks/mic is below MIN_PICKS_PER_MIC='
                         f'{MIN_PICKS_PER_MIC}. Lower the target or retrain the picker.')
    if ppm > TOPAZ_MAX_PICKS_PER_MIC:
        raise ValueError(f'REFUSED: {ppm:.2f} picks/mic exceeds TOPAZ_MAX_PICKS_PER_MIC='
                         f'{TOPAZ_MAX_PICKS_PER_MIC} — at that rate the picker is proposing noise.')
    return {'particle_threshold': round(thr, 3), 'predicted_n': kept, 'predicted_ppm': ppm,
            'score_min': float(sc.min()), 'score_max': float(sc.max()), 'target_ppm': target}


# ---------------------------------------------------------------------------
# 9. DECISION helper — the PRE-3D GATE (hard gate)
# ---------------------------------------------------------------------------
_GATE_SCHEMA = ('{"box_verdict":"ok|too_small|too_large","clean_verdict":"clean|some_junk|dirty",'
                '"secondary_structure":true|false,"vision_res_A":<number>,'
                '"distinct_views_seen":<int>,"unsure":true|false,"why":"<one sentence>"}')


def branch_survivorship(proj, feed_job, feed_group, branches):
    """[v2] finding #12 — WHICH BRANCH SURVIVED, by UID composition.

    diagnose2d's `trace_branches` reports the SELECTION branches behind a set by walking class
    averages, and in THIS pipeline topology that answer is structurally unavailable: LOOP B runs a
    pooled class_2D + select_2D on the already-merged stack, so per-picker class provenance is gone
    (it reported ONE branch for the test's gate feed). Intersecting particle UIDs answers the same
    question directly and cheaply.

    branches = [{'tag': 'blob', 'job': <final select job>, 'group': <group>}, ...]
    """
    print('\nBRANCH COMPOSITION / SURVIVORSHIP of the gate feed (UID intersection — the gate\'s own '
          'per-branch tracing cannot see through a pooled post-merge 2D):')
    try:
        feed_uids = set(feed_job.load_output(feed_group, slots=['uid'])['uid'].tolist())
    except Exception as e:                             # noqa: BLE001
        print(f'   unavailable: could not load the feed UIDs ({e!r}). The gate still runs on its '
              f'four measured checks; you just will not know which branch dominates.')
        return []
    rows, n_feed = [], len(feed_uids)
    print(f'   gate feed {feed_job.uid}: {n_feed:,} particles')
    print(f'   {"branch":10s} {"final select":14s} {"branch n":>12s} {"in feed":>12s} '
          f'{"% of feed":>10s} {"% surviving":>12s}')
    for b in branches:
        try:
            u = set(b['job'].load_output(b['group'], slots=['uid'])['uid'].tolist())
            inter = len(u & feed_uids)
            rows.append({'tag': b['tag'], 'uid': b['job'].uid, 'n': len(u), 'in_feed': inter,
                         'share': inter / max(n_feed, 1), 'survivorship': inter / max(len(u), 1)})
            print(f'   {b["tag"]:10s} {b["job"].uid:14s} {len(u):12,d} {inter:12,d} '
                  f'{100 * inter / max(n_feed, 1):9.1f}% {100 * inter / max(len(u), 1):11.1f}%')
        except Exception as e:                         # noqa: BLE001
            print(f'   {b["tag"]:10s} {b["job"].uid:14s}   unavailable ({e!r})')
    print('   [ref TEST P5/W23, for comparison: blob1 11.9% of feed at 35.5% survivorship, '
          'blob2 19.9% at 45.6%, topaz 68.2% at 77.0% — the numbers that retired the second blob '
          'band and refuted "topaz was under-cleaned".]')
    if rows:
        worst = min(rows, key=lambda r: r['survivorship'])
        if worst['survivorship'] < 0.20:
            print(f'!! branch {worst["tag"]!r} contributed only {100 * worst["survivorship"]:.1f}% of '
                  f'its own particles to the feed — the post-merge pass rejected most of it. That is '
                  f'a branch worth dropping or redoing, and it is invisible in the headline gate '
                  f'verdict.')
    return rows


def pre_3d_gate(proj, feed_job, feed_group=None, allow_minor=True, branches=None):
    """diagnose_2d + assess_2d on the job FEEDING the ab-initio. Raises GateHalt on SERIOUS/ASK_HUMAN.

    The tool traces the particle lineage itself and diagnoses every selection branch it can see;
    [v2] `branches` additionally runs the UID-composition/survivorship check (finding #12), which
    is the only per-branch answer available once a pooled post-merge 2D has run.
    """
    ev = D2.diagnose_2d(proj, feed_job.uid, source_group=feed_group,
                        out_dir=str(WORK_DIR / f'gate_{feed_job.uid}'),
                        datainfo_path=DATAINFO_YAML, symmetry=SYMMETRY)
    print(D2.build_prompt(ev))
    pngs = list(ev.get('panel_pngs') or [])
    for extra in ('views_png', 'profile_png'):
        if ev.get(extra):
            pngs.append(ev[extra])
    print('GATE FIGURES (read these):')
    for p in pngs:
        print('   ', p)
    if branches:
        branch_survivorship(proj, feed_job, feed_group or particles_group(feed_job), branches)
    vision = {}
    try:
        vision = vision_json(pngs, D2.build_prompt(ev) + sig_block('what a real class average of '
                                                                   'this specimen looks like'),
                             _GATE_SCHEMA)
        print('inline vision call ->', json.dumps(vision))
    except VisionUnavailable as e:
        print(f'!! VISION UNAVAILABLE: {e}')
        print('!! assess_2d will run on the MEASURED NUMBERS ALONE (every vision arg is optional). '
              'That is a weaker gate: read the panels above yourself and, if you disagree, re-run '
              'this cell with `vision` filled in by hand.')
    rep = D2.assess_2d(ev,
                       box_verdict=vision.get('box_verdict'),
                       clean_verdict=vision.get('clean_verdict'),
                       secondary_structure=vision.get('secondary_structure'),
                       vision_res_A=vision.get('vision_res_A'),
                       distinct_views_seen=vision.get('distinct_views_seen'),
                       unsure=bool(vision.get('unsure', False)),
                       notes=vision.get('why', '') + ' [inline notebook gate]')
    print('\n' + rep['subc_return'])
    print(f"\nreference result on the equivalent job (J595): HEALTHY, no flags — 1,304,332 ptcls, "
          f"particle 75.2-120.5 A, box_ratio 2.05, junk mass 1%, class-res median 11.13 A vs "
          f"Nyquist 3.86 A, 30 view groups / 7 distinct views")
    status = rep['status']
    if status in ('SERIOUS', 'ASK_HUMAN'):
        raise GateHalt(f'pre-3D gate = {status}. {rep["agentM_action"]["text"]}\n'
                       f'Do NOT run the ab-initio. Fix the flagged branch first '
                       f'(report: {rep.get("report_path")}).')
    if status == 'MORE_2D_CLEANING':
        raise GateHalt('pre-3D gate = MORE_2D_CLEANING: the COUNT is sufficient but the set is not '
                       'clean. Go back and run another 2D classify -> select round on the SAME '
                       'branch (LOOP B), then re-run this gate. This is a cleaning loop, not a '
                       're-plan.')
    if status == 'MINOR' and not allow_minor:
        raise GateHalt('pre-3D gate = MINOR and allow_minor=False.')
    return ev, rep


# ---------------------------------------------------------------------------
# 10. DECISION helpers — 3D
# ---------------------------------------------------------------------------
def reported_fsc(proj, job_uid, label=''):
    """CryoSPARC's REPORTED masked+corrected FSC (fsc_noisesub). The first reference, always.

    Never hand-roll a mask-free FSC to override it: it is unnecessary for a global refine and
    BIASED AGAINST a focused local refine by construction (a whole-box FSC averages in exactly the
    region the job was told to give up). That error produced a false "the focus mask did not help"
    in the reference.
    """
    r = VR.fsc_resolution_from_job(proj, job_uid)
    if not r:
        print(f'  {job_uid} {label}: no FSC asset — read it off the job plot and pass it by hand')
        return None
    print(f'  {job_uid} {label}: REPORTED corrected FSC = {r["res_A"]:.2f} A '
          f'(column {r["column"]}, iter {r["iteration"]}); mask bought '
          f'{r.get("mask_effect_A", float("nan")):.2f} A; per-curve: '
          + ', '.join(f'{k}={v:.2f}' for k, v in (r.get('res_by_column') or {}).items()
                      if v is not None))
    return r


_CLASS_SCHEMA = ('{"classes":[{"label":"<label>","is_target":true|false,'
                 '"matches_artifact":true|false,"isotropic_hollow":true|false,'
                 '"note":"<one sentence naming the NON-SHARED feature you used>"}],'
                 '"best":"<label>","why_best":"<one sentence>"}')


def screen_abinit_classes(proj, abinit_job, out_dir=None):
    """DECISION — which ab-initio classes are worth refining.

    PRIMARY (deterministic): vis_res.compare_volumes count-robust metrics at a FIXED enclosed
    volume (n_frag, frac_in_largest). In the reference this measurement CONFIRMED the human's
    class_0 call and OVERTURNED the agent's eyeball ranking, which had been an artefact of
    per-panel independent normalisation flattering the higher-count class.
      reference: class_0 3 frag / 0.920  (BEST) | class_3 43 / 0.861
                 class_1 54 / 0.839            | class_2 141 / 0.739 (dropped as junk-like)
    SECONDARY: the inline vision read of the shared-window contact sheet.
    """
    out_dir = Path(out_dir or WORK_DIR / f'abinit_screen_{abinit_job.uid}')
    out_dir.mkdir(parents=True, exist_ok=True)
    names = live_outputs(abinit_job)
    vol_groups = sorted([n for n in names if n.startswith('volume_class_')],
                        key=lambda s: int(s.rsplit('_', 1)[1]))
    if not vol_groups:
        raise KeyError(f'{abinit_job.uid}: no volume_class_* output groups; live groups = {names}')
    jdir = Path(proj.dir()) / abinit_job.uid
    mrcs, labels = [], []
    for g in vol_groups:
        k = g.rsplit('_', 1)[1]
        pats = [f'{abinit_job.uid}_class_{k}_final_volume.mrc',
                f'{abinit_job.uid}_class_{int(k):02d}_final_volume.mrc',
                f'*class_{k}*final_volume*.mrc', f'*class_{int(k):02d}*final_volume*.mrc',
                f'*class_{k}*volume*.mrc', f'*class_{int(k):02d}*volume*.mrc']
        hits = []
        for pat in pats:
            hits = sorted(jdir.glob(pat))
            if hits:
                break
        if not hits:
            print(f'!! no volume file found for {g} in {jdir}')
            continue
        mrcs.append(str(hits[-1]))
        labels.append(f'class_{k}')
    if not mrcs:
        raise FileNotFoundError(f'no ab-initio class volumes under {jdir}')
    cmp_ = VR.compare_volumes(mrcs, out_dir=out_dir, labels=labels)
    rows = []
    for lab, g, m in zip(labels, vol_groups, cmp_['metrics']):
        pg = g.replace('volume_', 'particles_')
        n = n_items(abinit_job, pg) if pg in names else -1
        rows.append({'label': lab, 'volume_group': g, 'particles_group': pg, 'n': n,
                     'n_frag': m['n_frag'], 'frac_in_largest': m['frac_in_largest'],
                     'Rg_A': m['Rg_A']})
    rows.sort(key=lambda r: (-r['frac_in_largest'], r['n_frag']))
    print('\nCOUNT-ROBUST CLASS METRICS (fixed enclosed volume; shared window per view type)')
    print(f'{"class":10s} {"n_particles":>12s} {"n_frag":>7s} {"frac_in_largest":>16s} {"Rg_A":>7s}')
    for r in rows:
        flag = '' if r['frac_in_largest'] >= CLASS_SCREEN_KEEP_FRAC_IN_LARGEST else '   <- junk-like'
        print(f'{r["label"]:10s} {r["n"]:12,d} {r["n_frag"]:7d} {r["frac_in_largest"]:16.3f} '
              f'{r["Rg_A"]:7.1f}{flag}')
    print(f'\nreference: class_0 3/0.920 (BEST) | class_3 43/0.861 | class_1 54/0.839 | '
          f'class_2 141/0.739 (dropped)')
    print(f'contact sheets (read them as the SECONDARY check): {cmp_.get("sheets")}')

    # ---- the metric candidate set (PRIMARY, deterministic) -------------------
    keep = [r for r in rows if r['frac_in_largest'] >= CLASS_SCREEN_KEEP_FRAC_IN_LARGEST]
    if not keep:
        keep = rows[:1]
        print('!! no class cleared the frac_in_largest floor; keeping the single best and flagging it')
    metric_best = rows[0]['label']

    # ---- the IDENTITY read: signature + vendored labelled controls ----------
    # [v2] finding #13. This call gets (a) the human's physical signature, (b) a POSITIVE control
    # (the reference project's own final map, and the target at ab-initio resolution) and (c) a
    # NEGATIVE control (the artifact that a screen once preferred). Without them the screen has
    # nothing to compare against but its own prior, and on the generalisation test that prior was
    # a linguistic association from the sample's name.
    ctrl = control_images('abinit')
    vision_best, v = None, {}
    try:
        v = vision_json((cmp_.get('sheets') or []) + ctrl,
                        'These are ab-initio class volumes from ONE homo_abinit job, rendered on '
                        'ONE shared intensity window per view type (native slices on the top row, '
                        'projections below).\n'
                        + sig_block('which class is the TARGET and which is an artifact')
                        + (('\n' + CONTROL_LEGEND) if ctrl else '')
                        + '\nFor EACH class decide whether it is the TARGET or the ARTIFACT, and '
                          'name the NON-SHARED feature you used. A feature both can show (e.g. '
                          '"hollow") cannot discriminate — find a directional/geometric one. '
                          'Beware: a class with more particles looks better simply because it is '
                          'less noisy, and per-panel normalisation flatters the bigger class.',
                        _CLASS_SCHEMA)
        print('inline vision read ->', json.dumps(v, indent=2))
        vision_best = v.get('best')
    except VisionUnavailable as e:
        print(f'   (identity vision read unavailable: {e})')
        print(f'   READ THESE YOURSELF: {(cmp_.get("sheets") or []) + ctrl}')
        print('   The metrics alone will choose the candidates. The IDENTITY question — is the '
              'winner the target or a look-alike artifact? — has NOT been asked.')

    # ---- METRIC/VISION DISAGREEMENT: refine BOTH, let the FSC decide --------
    labels = {r['label'] for r in rows}
    if vision_best in labels and vision_best != metric_best:
        print('\n' + '=' * 78)
        print(f'METRIC/VISION DISAGREEMENT: metrics rank {metric_best} '
              f'(n_frag {rows[0]["n_frag"]}, frac_in_largest {rows[0]["frac_in_largest"]:.3f}); '
              f'vision picks {vision_best} because {v.get("why_best", "(no reason given)")}. '
              f'Refining BOTH as separate arms in LOOP C. Human: this is a judgement call worth '
              f'your eye.')
        print('=' * 78)
        print('WHY THIS RULE EXISTS: on P5/W23 the screen rejected the LARGEST class, which was also '
              'best on BOTH count-robust metrics, on an inverted vision criterion. The 3D stage then '
              'ran on an ab-initio spherical-shell artifact and plateaued at 6.46 A, and the '
              'rejected class was additionally used as a junk-sink DECOY that drained 103,807 '
              'genuine particles. Refining both arms costs minutes. (subC.md METRIC/VISION '
              'DISAGREEMENT; and NEVER seed a junk-sink decoy with a top-metric candidate.)')
        if vision_best not in {r['label'] for r in keep}:
            keep = keep + [r for r in rows if r['label'] == vision_best]
    elif vision_best:
        print(f'\nmetric-best and vision-best AGREE on {metric_best}.')

    print(f'\nCANDIDATE CLASSES -> {[r["label"] for r in keep]}')
    return rows, keep, cmp_


# ---------------------------------------------------------------------------
# 10b. [v2] CONNECTIVITY-SUSTAINED VOLUME (finding #14)
#      vis_res.assess() tests LEGIBILITY AT A POINT: can you read secondary structure where the
#      density is? It returned CONSISTENT for J1530 (2.81 A reported) — a map that sustained only
#      23k/13k A^3 at the connectivity knee against J1525's 119k/68k A^3, i.e. 18-31% of the
#      expected protein volume, LESS THAN HALF of its own focus-mask footprint, with 4-13x more
#      fragments per unit volume and frac_in_largest 0.427 vs 0.556. Legibility-at-a-point and
#      VOLUME COVERAGE are different axes; only the second caught it. A tight local-refine mask
#      can buy a better MASKED FSC by shrinking the comparison window over noisier density.
#      This is numeric (VR.auto_display_levels) and needs no ChimeraX render.
# ---------------------------------------------------------------------------
def connectivity_volume(mrc_path, label='', mass_kDa=None, quiet=False):
    """{fuller, finer} enclosed volume + n_frag at the auto-picked connectivity-knee levels."""
    import numpy as np, mrcfile                        # noqa: PLC0415 — local to the check
    p = Path(mrc_path)
    try:
        un = VR.unsharpened_sibling(p)
        use = un if Path(un).exists() else p
        with mrcfile.open(str(use), permissive=True) as m:
            vol = m.data.astype('float32').copy()
            vx = float(m.voxel_size.x) or 1.0
        scan = VR.auto_display_levels(vol, vx)
        rows = [dict(r, tag=t) for t, r in zip(('fuller', 'finer'), scan['chosen'])]
    except Exception as e:                             # noqa: BLE001
        print(f'   connectivity volume unavailable for {label or p.name}: {e!r}')
        return None
    expect = VR.volume_for_mass_kDa(mass_kDa or EXPECTED_CORE_MASS_KDA)
    out = {'label': label or p.name, 'mrc': str(use), 'expected_A3': expect, 'rows': rows}
    if not quiet:
        print(f'  CONNECTIVITY-SUSTAINED VOLUME — {out["label"]}  '
              f'(expected protein molecular volume for {mass_kDa or EXPECTED_CORE_MASS_KDA:g} kDa '
              f'= {expect:,.0f} A^3)')
        for r in rows:
            frac = r['enclosed_A3'] / max(expect, 1)
            fpv = r['n_frag'] / max(r['enclosed_A3'] / 1000.0, 1e-9)
            print(f'    {r["tag"]:6s} level {r["level"]:.4f}  {r["enclosed_A3"] / 1000:8.1f}k A^3 '
                  f'= {frac:5.2f}x expected   {r["n_frag"]:4d} frags = {fpv:5.2f} per kA^3   '
                  f'connectivity {r["frac_in_largest"]:.2f}'
                  + ('   <- BELOW the %.2f floor' % CONNECTIVITY_VOL_MIN_FRAC
                     if (r['tag'] == 'fuller' and frac < CONNECTIVITY_VOL_MIN_FRAC) else ''))
        print('    [ref TEST: J1525 (kept) 119k/68k A^3 at 0.36/1.25 frags per kA^3; J1530 '
              '(REJECTED despite a 0.16 A better FSC) 23k/13k A^3 at 4.65/2.23 per kA^3]')
    f = rows[0]['enclosed_A3'] / max(expect, 1)
    out['fuller_frac_of_expected'] = f
    out['flagged'] = bool(f < CONNECTIVITY_VOL_MIN_FRAC)
    if out['flagged'] and not quiet:
        print(f'!! FLAG: the fuller connectivity knee sustains only {f:.2f}x the expected protein '
              f'volume. Much of the density inside the window the refine was scoped to does not '
              f'hold together at any readable level. A better masked FSC on a map like this is the '
              f'mask shrinking the comparison window, not a better map.')
    return out


def compare_connectivity(entries):
    """entries = [(label, mrc_path)]. Prints the A/B and returns the ranked list (best first)."""
    got = [c for c in (connectivity_volume(m, label=l) for l, m in entries) if c]
    if len(got) < 2:
        return got
    print('\n  A/B on VOLUME COVERAGE (the axis assess() does not test):')
    for c in sorted(got, key=lambda c: -c['fuller_frac_of_expected']):
        r = c['rows'][0]
        print(f'    {c["label"]:34s} {r["enclosed_A3"] / 1000:7.1f}k A^3 '
              f'({c["fuller_frac_of_expected"]:.2f}x expected)  {r["n_frag"]:4d} frags')
    return sorted(got, key=lambda c: -c['fuller_frac_of_expected'])


def pick_best_arm(proj, arms, cmp_out_dir=None):
    """DECISION — FSC-FIRST. arms = [{'label','job'}...]. < FSC_TIE_A apart is a TIE."""
    scored = []
    for a in arms:
        r = reported_fsc(proj, a['job'].uid, a['label'])
        scored.append({**a, 'res_A': (r or {}).get('res_A'),
                       'mask_effect_A': (r or {}).get('mask_effect_A')})
    have = [s for s in scored if s['res_A'] is not None]
    if not have:
        raise RuntimeError('no arm produced a readable FSC — read the job plots by hand')
    have.sort(key=lambda s: s['res_A'])
    best = have[0]
    ties = [s for s in have if s['res_A'] - best['res_A'] < FSC_TIE_A]
    print(f'\nARMS RANKED ON THE REPORTED MASKED+CORRECTED FSC:')
    for s in have:
        print(f'  {s["label"]:34s} {s["job"].uid:8s} {s["res_A"]:.2f} A')
    print(f'reference: class_0 ALONE (351,748 ptcls) 3.64 A  BEAT  merged 0+1+3 (1,087,189) 6.54 A '
          f'— a 3x smaller pool won, because merging DECORRELATED the half-maps.')
    if len(ties) > 1:
        print(f'\n!! {len(ties)} arms within {FSC_TIE_A} A = a TIE. An FSC difference below ~0.2 A is '
              f'inside estimation noise and moves with mask tightness alone. Deciding on '
              f'count-robust real-space metrics instead:')
        mrcs, labels = [], []
        for s in ties:
            try:
                mrcs.append(str(SJ.final_volume_map(proj, s['job'].uid, sharp=False)))
                labels.append(s['label'])
            except Exception as e:
                print(f'   (no volume for {s["label"]}: {e!r})')
        if mrcs:
            c = VR.compare_volumes(mrcs, out_dir=cmp_out_dir or WORK_DIR / 'arm_tiebreak',
                                   labels=labels)
            for lab, m in zip(labels, c['metrics']):
                print(f'   {lab:34s} n_frag={m["n_frag"]:4d}  frac_in_largest={m["frac_in_largest"]:.3f}')
            print(f'   sheets: {c.get("sheets")}')
            # [v2] finding #14 — add VOLUME COVERAGE to the tie-break. frac_in_largest is measured
            # at a fixed top-voxel occupancy and can tie while the two maps sustain wildly
            # different amounts of connected density.
            conn = compare_connectivity(list(zip(labels, mrcs)))
            order = sorted(zip(labels, c['metrics']),
                           key=lambda t: (-t[1]['frac_in_largest'], t[1]['n_frag']))
            winner_label = order[0][0]
            if conn and len(conn) > 1:
                cbest = conn[0]['label']
                if cbest != winner_label:
                    print(f'   !! the occupancy metrics prefer {winner_label} but VOLUME COVERAGE '
                          f'prefers {cbest} ({conn[0]["fuller_frac_of_expected"]:.2f}x vs '
                          f'{[x["fuller_frac_of_expected"] for x in conn if x["label"] == winner_label]}'
                          f' x expected). Volume coverage wins this tie-break: a map that does not '
                          f'sustain the protein\'s own volume at any readable level is the worse map, '
                          f'whatever its masked FSC says (J1530, 2.81 A reported, rejected on exactly '
                          f'this).')
                    winner_label = cbest
            best = next(s for s in ties if s['label'] == winner_label)
            print(f'   -> real-space winner: {best["label"]}')
    print(f'\nWINNER: {best["label"]} = {best["job"].uid} at {best["res_A"]:.2f} A')
    return best, have


def unbin_gate(res_A, dims):
    """DECISION — pure arithmetic. Un-bin only inside the regime where it pays."""
    nyq = dims['nyquist_binned_A']
    trig = UNBIN_TRIGGER_FACTOR * nyq
    ratio = res_A / nyq
    print(f'  reported {res_A:.2f} A vs binned Nyquist {nyq:.3f} A = {ratio:.2f}x away '
          f'(trigger: <= {UNBIN_TRIGGER_FACTOR}x, i.e. res <= {trig:.2f} A)')
    print(f'  reference: un-binned at 1.10x and paid -0.58 A; correctly DEFERRED at 2.0x, where '
          f'un-binning would have been a waste (binning discards only information ABOVE the binned '
          f'Nyquist).')
    go = res_A <= trig
    print(f'  -> {"UN-BIN NOW" if go else "DO NOT UN-BIN YET — keep purifying in the cheap binned box"}')
    return go


# ---------------------------------------------------------------------------
# 11. DECISION helper — final resolution validation (vis_res)
# ---------------------------------------------------------------------------
_VIS_SCHEMA = ('{"secondary_structure":true|false,"vision_estimate_A":<number>,'
               '"fsc_trustworthy":true|false|null,"why":"<two sentences>"}')


def vis_res_check(proj, job_uid, mass_kDa=None, surface=True):
    """Render slices (the RESOLUTION read) + the paired surface (the second look), then assess().

    `vis_res` is the LIE-DETECTOR, not a second FSC. If it returns NEEDS_SURFACE_CHECK the verdict
    is NOT a verdict — no resolution is reported until the surface resolves it, and a verdict
    carrying SURFACE_UNAVAILABLE is explicitly UNCONFIRMED.
    """
    f = reported_fsc(proj, job_uid, 'deliverable')
    fsc_A = (f or {}).get('res_A')
    mrc = SJ.final_volume_map(proj, job_uid, sharp=False)
    out = WORK_DIR / f'vis_{job_uid}'
    sl = VR.render_map_slices(str(mrc), out_dir=str(out))
    print(f'  SLICES  (resolution read, native sampling): {sl["contact_sheet"]}')
    prompt = VR.build_prompt(fsc_A) + sig_block('whether this density is the target at all')
    print(prompt)
    sheets = [sl['contact_sheet']]
    # [v2] finding #14 — VOLUME COVERAGE, the axis assess() does not test. Numeric, no render.
    conn = connectivity_volume(SJ.final_volume_map(proj, job_uid, sharp=True),
                               label=job_uid, mass_kDa=mass_kDa)
    surf = None
    if surface:
        try:
            # [v2] SIZE MATTERS AND IT IS NOT COSMETIC. surface_judge's own default (900 px) gave
            # 5.9 px/A on the test deliverable — under the ~10 px/A floor a side-chain knob and a
            # bump are the same handful of pixels, and the map reads a full angstrom worse than it
            # is. Render big, then CHECK the measured px/A.
            surf = SJ.surface_judge(proj, job_uid, mass_kDa or EXPECTED_CORE_MASS_KDA,
                                    out_dir=str(out / 'surface'), datainfo_path=DATAINFO_YAML,
                                    size=SURFACE_RENDER_SIZE,
                                    render_host=CHIMERAX_RENDER_HOST,
                                    render_ld_path=CHIMERAX_RENDER_LD_PATH)
            for k in ('sheet', 'sheets', 'contact_sheet'):
                v = surf.get(k) if isinstance(surf, dict) else None
                if isinstance(v, str):
                    sheets.append(v)
                elif isinstance(v, (list, tuple)):
                    sheets.extend(v)
            ppa = (surf or {}).get('px_per_A')
            if ppa is not None:
                print(f'  SURFACE render resolution: {ppa} px/A '
                      f'(floor {SURFACE_MIN_PX_PER_A})')
                if ppa < SURFACE_MIN_PX_PER_A:
                    print(f'!! BELOW THE LEGIBILITY FLOOR. Re-render larger (raise '
                          f'SURFACE_RENDER_SIZE) or zoom (surface_panel(..., zoom_cube_A=64, '
                          f'px_per_A_target=20)) BEFORE reading it. At 5.9 px/A the test '
                          f'deliverable read ~1 A worse than it does at 10.4 px/A.')
            print(f'  SURFACE (second look): {sheets[1:]}')
        except Exception as e:
            print(f'!! SURFACE RENDER UNAVAILABLE ({e!r}). A verdict without it that needs it is '
                  f'explicitly UNCONFIRMED (SURFACE_UNAVAILABLE). Flat slices under-read a 3-4 A '
                  f'map by ~1-1.5 A.')
    vision = {}
    try:
        vision = vision_json(sheets + control_images('final'),
                             prompt + ('\n' + CONTROL_LEGEND if control_images('final') else ''),
                             _VIS_SCHEMA)
        print('inline vision call ->', json.dumps(vision))
    except VisionUnavailable as e:
        print(f'!! VISION UNAVAILABLE: {e}')
        print('!! READ the sheets above yourself and re-run this cell with `vision` filled in. '
              'assess() will NOT guess for you.')
        return {'status': 'VISION_REQUIRED', 'fsc_res_A': fsc_A, 'sheets': sheets}
    verdict = VR.assess(secondary_structure=bool(vision.get('secondary_structure')),
                        vision_estimate_A=vision.get('vision_estimate_A'),
                        fsc_res_A=fsc_A,
                        fsc_trustworthy=vision.get('fsc_trustworthy'),
                        notes=vision.get('why', ''),
                        surface_checked=bool(surf),
                        surface_unavailable=not bool(surf))
    print('\nVIS_RES VERDICT:', json.dumps({k: verdict[k] for k in
                                            ('status', 'flags', 'interpretable')
                                            if k in verdict}, default=str))
    print(f'reference: J1077 2.95 A -> CONSISTENT, interpretable, vision 3.3 A (delta 0.35); '
          f'J1072 3.06 A -> CONSISTENT, vision 3.6 A')
    if verdict.get('status') == 'NOT_INTERPRETABLE':
        print('!! NOT_INTERPRETABLE: the FSC number is MEANINGLESS. Do not report it.')
    # [v2] carry the volume-coverage evidence in the verdict, and say plainly that a CONSISTENT
    # here does NOT clear a flagged volume: assess() is a legibility test, and it returned
    # CONSISTENT for a map sustaining 18-31% of its expected protein volume (finding #14).
    verdict['connectivity'] = conn
    if conn and conn.get('flagged'):
        print('!! VOLUME-COVERAGE FLAG stands alongside this verdict. assess() tests LEGIBILITY AT '
              'A POINT and can return CONSISTENT for a map that does not sustain its own protein '
              'volume. Do not report this map as the deliverable without explaining the gap — and '
              'if you are choosing between two refines, run compare_connectivity() on both.')
    return verdict


print('CELL 3 loaded: '
      + ', '.join(['derive_dimensions', 'connect', 'run_job', 'poll', 'require_completed',
                   'resolve_out', 'particles_group', 'assert_state_scope', 'class2d_round',
                   'clean_2d_loop', 'stop_2d_loop', 'rank_select', 'k_round1_from_pool',
                   'decide_k', 'k_memory_guard', 'gpu_admission_check', 'live_free_mib',
                   'decide_topaz_threshold', 'check_extract_count', 'pre_3d_gate',
                   'branch_survivorship', 'screen_abinit_classes', 'pick_best_arm',
                   'connectivity_volume', 'compare_connectivity', 'unbin_gate', 'vis_res_check',
                   'physical_signature', 'sig_block', '_llm_json']))
print(f'   function-cell fingerprint (finding #2): {cell_fingerprint() or "unreadable"}')
print(f'   job mode: {"BLOCKING" if WAIT_FOR_JOBS else "QUEUE-AND-RETURN"}   '
      f'2D subgroup split over {SPLIT_2D_OVER_N:,}   surface render {SURFACE_RENDER_SIZE} px')
llm_status()

## STEP 0 — connect, check the deciders, derive the dimensions

Nothing is queued here. If a decider binary is missing you find out **now**, not four days into a
2D ladder.

In [ ]:
# STEP 0
problems = check_tool_binaries()
DIMS = derive_dimensions()
cs, proj, ws = connect()
assert_state_scope()          # [v2] finding #3 — refuse to resume ANOTHER dataset's state
print(f'\nstate file: {STATE_PATH}  (delete a key, or call forget("<key>"), to force a re-run)')
print('current state:', json.dumps(load_state(), indent=2))

## STEP 1 — import micrographs + patch CTF

The reference started here too: the input is **motion-corrected micrographs**, so there is no
movie import, no motion correction and no gain-flip question.

`curate_exposures` is **deliberately skipped** — in the reference only 159/6999 (2.3 %) of
micrographs were borderline (median ctf_fit 2.98 Å, median ice 1.01) and the downstream 2D absorbed
the residual. Curate only if more than ~10 % of your micrographs are bad.

Set `REUSE_CTF_UID` if you already have a `patch_ctf_estimation_multi` job for this dataset.

In [ ]:
# STEP 1
REUSE_CTF_UID = None      # e.g. 'J83' to reuse an existing patch_ctf job (the reference did this
                          # across every one of its workspaces)

if REUSE_CTF_UID:
    ctf = proj.find_job(REUSE_CTF_UID); ctf.refresh()
    assert ctf.status == 'completed', f'{REUSE_CTF_UID} is {ctf.status}'
    remember('ctf', ctf.uid)
    print(f'reusing CTF job {ctf.uid}')
else:
    assert MICROGRAPHS_GLOB, 'MICROGRAPHS_GLOB is an ENV FIELD and is REQUIRED — never copy the reference path'
    imp = run_job('import', proj, ws, 'import_micrographs',
                  {},
                  {'blob_paths': MICROGRAPHS_GLOB,          # ENV
                   'psize_A': PIXEL_SIZE_A,
                   'accel_kv': ACCEL_KV,
                   'cs_mm': CS_MM,
                   'total_dose_e_per_A2': TOTAL_DOSE,
                   'skip_header_check': False},
                  'import_micrographs')
    ctf = run_job('ctf', proj, ws, 'patch_ctf_estimation_multi',
                  {'exposures': (imp.uid, resolve_out(imp, 'imported_micrographs', 'exposures'))},
                  {'compute_num_gpus': len(GPUS_MULTI)},
                  'patch_ctf', gpus=GPUS_MULTI)

CTF_GROUP = resolve_out(ctf, 'exposures', 'micrographs', 'exposures_success')
N_MICS = n_items(ctf, CTF_GROUP)
print(f'\nCTF job {ctf.uid}, output group {CTF_GROUP!r}, {N_MICS:,} micrographs  [ref 6,999]')

## STEP 2 — picking branch 1: **one** blob picker over the full diameter band

**Breadth, not a looser threshold — but breadth by RANGE, not by duplicated branches.** [v2]

The reference ran **two** blob branches (1.00–1.50 × D and 0.90–1.30 × D), each with its own full
2D clean loop. On the first generalisation test those two branches contributed only **11.9 %** and
**19.9 %** of the pre-3D gate feed, at **35.5 % / 45.6 %** survivorship through the post-merge pass,
while the single Topaz branch contributed **68.2 %** at **77.0 %**. The bands overlap over
1.00–1.30 D by construction, so the second band roughly **doubled the 2D cost of the blob arm**
(5 rounds, ~3 h 50 m for round 1 alone on ~3 M particles) for a small, largely redundant addition.

So: **one blob picker spanning the union of the two bands** (`0.90 · D → 1.50 · D`), and the
pipeline is **two branches — blob + Topaz** — merged and deduplicated. Picker breadth is still the
principle; it is now expressed as *blob-range breadth + a complementary ML picker*.

`extract_micrographs_multi` **needs the `micrographs` input** fed from the CTF job's `exposures`
group — a required input that is easy to omit (all three plans in the reference forgot it once).

**The extract guard now predicts its own baseline** [v2, finding #5]: a *first* extract from raw
picks loses a large, **box-dependent** fraction at the frame edge —
`1 − ((W−box)/W)·((H−box)/H)` — 15.73 % for box 336 on 4096², and the blob first-extracts lost
15.96 % / 15.72 %. That is geometry, not a fault. The figure is an **upper bound** and is
**picker-dependent** (Topaz self-excludes an edge margin and kept 99.47 %), so only a count
**below** prediction is flagged.

In [ ]:
# STEP 2
MIC_HW = mic_shape_px(proj, ctf, CTF_GROUP)        # [v2] (H, W) for the geometric extract guard
print(f'micrograph shape = {MIC_HW}   box = {DIMS["box_full_px"]} px -> geometric first-extract '
      f'loss up to {(predicted_extract_loss(DIMS["box_full_px"], MIC_HW) or 0) * 100:.2f}%')


def pick_and_extract(tag, band, key_prefix):
    pk = run_job(f'{key_prefix}_pick', proj, ws, 'blob_picker_gpu',
                 {'micrographs': (ctf.uid, CTF_GROUP)},
                 {'diameter': band[0], 'diameter_max': band[1]},
                 f'blob_pick {band[0]}/{band[1]} ({tag})', gpus=GPUS_SINGLE)
    require_completed(pk)
    pg = resolve_out(pk, 'particles')
    ppm = check_picks_per_mic(pk, pg, N_MICS, f'{tag} blob {band[0]}/{band[1]}')
    ex = run_job(f'{key_prefix}_extract', proj, ws, 'extract_micrographs_multi',
                 {'micrographs': (ctf.uid, CTF_GROUP), 'particles': (pk.uid, pg)},
                 {'box_size_pix': DIMS['box_full_px'],
                  'bin_size_pix': DIMS['box_bin_px'],
                  'compute_num_gpus': len(GPUS_MULTI)},
                 f'extract {tag} (box {DIMS["box_full_px"]} bin {DIMS["box_bin_px"]})',
                 gpus=GPUS_MULTI)
    require_completed(ex)
    eg = resolve_out(ex, 'particles')
    # [v2] FIRST extract -> compare against the GEOMETRIC prediction, not a flat 5%
    check_extract_count(ex, eg, n_items(pk, pg), first_extract=True, mic_hw=MIC_HW,
                        box_px=DIMS['box_full_px'])
    return pk, ex, eg, ppm


# ONE blob branch over the merged band (v2). The reference's two bands are recorded in the
# markdown above and in recipe.json; they are NOT run.
pk1, ex1, ex1_g, BLOB_PPM = pick_and_extract('blob', DIMS['blob_A'], 'blob')
print(f'\nblob branch: {n_items(pk1, "particles"):,} picks -> {n_items(ex1, ex1_g):,} extracted '
      f'   [ref, TWO bands: 5.10M (729/mic) + 6.16M (881/mic)]')

## STEP 3 — LOOP A1: the blob branch's own 2D clean loop

**Do not hoist this past the merge.** Each picking branch gets its own `class_2D → select_2D`
loop. A shared post-merge clean could not have revealed the reference's key branch signature:
blob keeps **17–22 %** of its picks at round 1 while Topaz keeps **83 %** (test: **41.9–49.0 %** vs
**85.0 %** — same story).

The loop is **genuinely unbounded**: `stop_2d_loop()` is the *only* thing that ends it — a
kept-fraction plateau, a **rising selected-class fraction** (the bank's "this branch is clean, stop
cleaning" signal), the `MIN_PARTICLES_FOR_3D` floor, or the `MAX_2D_ROUNDS_SAFETY` runaway guard.
The reference needed 3 rounds per branch, the test needed 5; **neither count is copied**.

Two [v2] corrections inside the stop condition:
* `class_frac` is **select-mode dependent** (loose / none / tight apply different cuts), so a round
  is compared only against the most recent round run in the **same mode**. On the test, round 5's
  drop to 0.460 / 0.545 was an artefact of the stricter `tight` rule, not a quality regression.
* A **big** rise (> `CLASS_FRAC_BIG_RISE`) qualifies at the softer floor `0.55`: one branch came in
  at 0.570 with a +0.257 rise, three points short of the uniform 0.60, and ran an extra round for
  nothing. A branch that is genuinely still dirty (0.477) is under either floor.

**`class2D_K` is decided at run time, per round** (`k_round1_from_pool` for round 1, then
`decide_k()` after every completed round — see the header and `K_DECISION_SPEC.md`). The
reference's 100/75/50 appears only as a printed comparison. [v2] The one **down**-trigger,
`n_collapsed`, now needs an absolute corroboration too (particles-per-class below ~400), because
the ratio alone measures *shrinkage from the origin*: on the test two branches in near-identical
condition landed on opposite sides of the 8× bar purely because their round-1 pools differed by 22 %.

**A pool over 1 M particles is split** [v2, finding #6] into up to `SPLIT_2D_MAX_JOBS` subgroups
that classify **concurrently on disjoint GPUs**, under **one global ranker threshold**, with the
selections merged afterwards (the split is disjoint, so no dedup). The test ran ~3 M as a single
job per branch: round 1 alone took ~3 h 50 m.

**Two things can stop this cell hard:** the `K × box²` memory guard (it names `BIG_K_LANE` and refuses
to queue), and [v2] the **live GPU admission check** — the K × box² budget is converted into a
per-GPU MiB estimate and compared against `nvidia-smi` on the *specific ids* immediately before
queueing, because a job that fitted the card-class formula still OOM'd on GPUs carrying ~8 GB of
foreign load the scheduler cannot see.

In [ ]:
# STEP 3
b1 = clean_2d_loop(cs, proj, ws, 'blob', ex1.uid, ex1_g, DIMS)
print(f'\n{b1["tag"]}: {len(b1["history"])} rounds -> {b1["final_select"].uid} '
      f'({b1["history"][-1]["n_out"]:,} particles)')

## STEP 4 — picking branch 2: Topaz, bootstrapped off blob's **own FINAL (tight) select**

No external model, no reference volume. `topaz_train` is fed the **FINAL (tight)** select of the
blob branch — not its round-1 loose select, which is what the reference did. The test confirmed the
change on three independent measurements: round-1 keep rate **85.0 %** (vs blob 41.9–49.0 %), merge
overlap **22.4 %** (vs the reference's 41.5 %), and LOOP-B survivorship **77.0 %** (vs blob
35.5–45.6 %). Topaz learns whatever positives it is given.

### `particle_threshold` is an ADAPTIVE knob, and it costs a job to get wrong [v2, finding #10]

It is the **only** gate that limits what `topaz_extract` writes out, and the right value belongs to
the **trained model**, not to the sample: on the *same* micrographs one model at the default `0`
gave 144.5 picks/mic, another gave 5.2/mic, and ours would have given 271.94/mic — all nominally
"default". The default can also return ~1 particle per micrograph while the job reports
`completed`.

So this step runs **two passes**:

1. a **diagnostic** extract with `particle_threshold = threshold = -6` — a *no-op* filter, so every
   NMS candidate is written out **with its score**;
2. the **final** extract at the threshold `decide_topaz_threshold()` derives from that pass.

> **The sub-trap.** `particle_threshold` gates on **Topaz's raw log-odds score** — the `score`
> column of `<job_dir>/topaz_particles_prediction.txt`, here `−6.000 … +9.826` — **not** on
> CryoSPARC's `pick_stats/ncc_score` (positive, to ~158 **on the same model**). A threshold derived
> from the wrong column (78.4) was a plausible mid-range number on the wrong axis, and the job
> "completed" with **zero particles**. The decision function reads the raw file and refuses
> anything outside its range. *Generalise: the score a picker reports on its output and the score
> its own gating parameter filters on can be different quantities.*

Then the Topaz branch runs its **own** 2D clean loop, exactly like the blob branch — its own rounds,
its own K decisions. That per-branch loop is what makes the branch signature visible, which is why
it is never hoisted past the merge.

In [ ]:
# STEP 4
seed_sel = b1['selects'][0] if TOPAZ_TRAIN_ON_ROUND1_LOOSE else b1['final_select']
seed_g = particles_group(seed_sel)
print(f'topaz seed = {seed_sel.uid} ({seed_g}, {n_items(seed_sel, seed_g):,} particles) -- the '
      f'blob branch\'s {"round-1 LOOSE" if TOPAZ_TRAIN_ON_ROUND1_LOOSE else "FINAL TIGHT"} select')

tt = run_job('topaz_train', proj, ws, 'topaz_train',
             {'micrographs': (ctf.uid, CTF_GROUP), 'particles': (seed_sel.uid, seed_g)},
             {'exec_path': TOPAZ_EXE,                       # ENV
              'par_diam': DIMS['topaz_par_diam_A'],
              'num_particles': TOPAZ_NUM_PARTICLES,
              'num_distribute': TOPAZ_NUM_DISTRIBUTE},
             'topaz_train (blob FINAL tight-select seed)', gpus=GPUS_MULTI)
require_completed(tt)
TT_MODEL = resolve_out(tt, 'topaz_model', 'model')

# ---- particle_threshold: ADAPTIVE (finding #10) ------------------------------
if TOPAZ_PARTICLE_THRESHOLD is None:
    # pass 1 — DIAGNOSTIC: particle_threshold == the candidate floor == a NO-OP filter, so every
    # candidate is written out WITH ITS RAW SCORE. This is the only way to see the model's scale.
    txd = run_job('topaz_extract_diag', proj, ws, 'topaz_extract',
                  {'micrographs': (ctf.uid, CTF_GROUP), 'model': (tt.uid, TT_MODEL)},
                  {'exec_path': TOPAZ_EXE,
                   'par_diam': DIMS['topaz_par_diam_A'],
                   'num_distribute': TOPAZ_NUM_DISTRIBUTE,
                   'threshold': TOPAZ_DIAG_FLOOR,
                   'particle_threshold': TOPAZ_DIAG_FLOOR},   # == threshold => no-op
                  'topaz_extract DIAGNOSTIC (no-op filter, for the raw score distribution)',
                  gpus=GPUS_MULTI)
    require_completed(txd, 'the diagnostic pass must finish before its scores can be read')
    TOPAZ_DECISION = decide_topaz_threshold(proj, txd, N_MICS, blob_ppm=BLOB_PPM)
    PT = TOPAZ_DECISION['particle_threshold']
else:
    TOPAZ_DECISION, PT = {'particle_threshold': TOPAZ_PARTICLE_THRESHOLD, 'predicted_n': None}, \
        TOPAZ_PARTICLE_THRESHOLD
    print(f'particle_threshold pinned by hand in cell 2: {PT} (no diagnostic pass). Make sure this '
          f'is on TOPAZ\'S RAW LOG-ODDS axis, not on pick_stats/ncc_score.')

# pass 2 — FINAL extract at the decided threshold
tx = run_job('topaz_extract', proj, ws, 'topaz_extract',
             {'micrographs': (ctf.uid, CTF_GROUP), 'model': (tt.uid, TT_MODEL)},
             {'exec_path': TOPAZ_EXE,                       # ENV
              'par_diam': DIMS['topaz_par_diam_A'],
              'num_distribute': TOPAZ_NUM_DISTRIBUTE,
              'threshold': TOPAZ_DIAG_FLOOR,                # candidate floor, unchanged
              'particle_threshold': PT},                    # DECIDED, never the default 0
             f'topaz_extract FINAL (particle_threshold={PT}, raw-score axis)', gpus=GPUS_MULTI)
require_completed(tx)
tx_g = resolve_out(tx, 'particles')
n_tx = n_items(tx, tx_g)
check_picks_per_mic(tx, tx_g, N_MICS, 'topaz')
if TOPAZ_DECISION.get('predicted_n'):
    dev = n_tx / max(TOPAZ_DECISION['predicted_n'], 1) - 1
    print(f'  delivered {n_tx:,} vs predicted {TOPAZ_DECISION["predicted_n"]:,} '
          f'({dev * 100:+.2f}%)   [test: +0.02%]')
    if abs(dev) > 0.05:
        print('!! the delivered count is more than 5% off the prediction — the threshold was '
              'applied to a different quantity than the one you measured. STOP and re-read the raw '
              'prediction file before spending 2D time on this set.')

ex3 = run_job('topaz_extract_box', proj, ws, 'extract_micrographs_multi',
              {'micrographs': (ctf.uid, CTF_GROUP), 'particles': (tx.uid, tx_g)},
              {'box_size_pix': DIMS['box_full_px'], 'bin_size_pix': DIMS['box_bin_px'],
               'compute_num_gpus': len(GPUS_MULTI)},
              f'extract topaz (box {DIMS["box_full_px"]} bin {DIMS["box_bin_px"]})',
              gpus=GPUS_MULTI)
require_completed(ex3)
ex3_g = resolve_out(ex3, 'particles')
# FIRST extract for this branch -> geometric prediction. Topaz self-excludes an edge margin, so a
# yield well ABOVE the prediction is expected here and is NOT a fault (test: 99.47% kept).
check_extract_count(ex3, ex3_g, n_tx, first_extract=True, mic_hw=MIC_HW,
                    box_px=DIMS['box_full_px'])

b3 = clean_2d_loop(cs, proj, ws, 'topaz', ex3.uid, ex3_g, DIMS)
print(f'\ntopaz: {len(b3["history"])} rounds -> {b3["final_select"].uid} '
      f'({b3["history"][-1]["n_out"]:,} particles)')

## STEP 5 — MERGE + DEDUP (the first merge point)

**Both** branches' final tight selects — blob and Topaz — are merged and deduplicated. The dedup
rate is the evidence that the pickers were genuinely complementary rather than redundant: the
reference removed **41.5 %** (2.41 M → 1.41 M) across three branches; the test removed **22.4 %**
across three, the lower figure being the (deliberate) consequence of training Topaz on the *tight*
select, so its picks agree less with blob's loose ones.

`min_dist_A ≈ 0.6 × particle diameter`.

In [ ]:
# STEP 5
BRANCHES = [{'tag': 'blob', 'job': b1['final_select'], 'group': b1['final_group']},
            {'tag': 'topaz', 'job': b3['final_select'], 'group': b3['final_group']}]
conns = {'particles': [(b['job'].uid, b['group']) for b in BRANCHES],
         'micrographs': (ctf.uid, CTF_GROUP)}
n_sum = sum(n_items(b['job'], b['group']) for b in BRANCHES)

merge = run_job('merge_dedup', proj, ws, 'remove_duplicate_particles',
                conns, {'min_dist_A': DIMS['dedup_min_dist_A']},
                f'dedup merge (blob+topaz, min_dist {DIMS["dedup_min_dist_A"]} A)')
require_completed(merge)
MERGE_G = resolve_out(merge, 'particles_kept', 'particles')
n_merge = n_items(merge, MERGE_G)
print(f'\nMERGE: {n_sum:,} summed -> {n_merge:,} deduped '
      f'({100 * (1 - n_merge / max(n_sum, 1)):.1f}% overlap removed)   '
      f'[ref 2.41M -> 1.41M = 41.5% over THREE branches; test 22.4%]')
if n_merge / max(n_sum, 1) > 0.95:
    print('!! < 5% overlap: your pickers are NOT complementary — they are finding different things, '
          'or one of them is picking noise. Inspect the per-branch 2D montages before continuing.')
for b in BRANCHES:
    print(f'   branch {b["tag"]:6s} {b["job"].uid:8s} {n_items(b["job"], b["group"]):>10,} particles '
          f'({100 * n_items(b["job"], b["group"]) / max(n_sum, 1):.1f}% of the summed pool)')

## STEP 6 — LOOP B: the pivot — ONE finer, high-K, resolution-capped 2D pass

This single job pair is what unlocked the reference reconstruction.

**Parameters are `J112`'s, per the human directive — not `J582`'s.** Dropped from J582:
`class2D_force_max=False` (set in *both* the failing and the winning runs here, so it explains
nothing, and it costs several-fold runtime) and `prepare_window_inner_radius/outer_radius`
0.8/0.9 (no situation-matched evidence). **`random_seed` is never pinned.**
`compute_num_gpus` / `compute_use_ssd` come from *your* environment.

**`class2D_K` is NOT one of J112's carried parameters here.** The merge + dedup has just created a
*newly enlarged* pool, which is exactly the moment the history bank sets K — so round 1 takes it
from `k_round1_from_pool()` (the pool-origin band) and any further round takes it from the same
`decide_k()` as the per-branch loops. The reference's **K = 300 on 1,407,603 particles** is recorded
as the comparison for a pool of that size, not as a constant — and it is what the band returns for
a ≥ 1.5 M pool. The round count again comes only from `stop_2d_loop()`.

In [ ]:
# STEP 6 — LOOP B
# Same three decoupled sources of truth as LOOP A (K_DECISION_SPEC.md):
#   K           <- k_round1_from_pool() on round 1 (the merge+dedup makes this a NEWLY-ENLARGED
#                  pool, which is exactly the bank's "set K here" moment), then decide_k()
#   round count <- stop_2d_loop() ONLY
#   select mode <- LOOSE throughout at this stage (the human directive for the pivot pass)
# [v2] the round itself goes through class2d_round(), so a >1M pool is split into concurrent
# subgroups under one global ranker threshold and the selections are merged after (finding #6).
finer_hist, cur_uid, cur_group = [], merge.uid, MERGE_G
finer_class, finer_out = None, None
BOX_2D = DIMS['box_bin_px']
k_dec = k_round1_from_pool(n_merge, 'merge + remove_duplicate_particles across the picking '
                                    'branches = a newly-enlarged pool',
                           tag='finer2d', box_px=BOX_2D, lane=LANE)
i = 0
while True:
    stop, reason, hard = stop_2d_loop('finer2d', finer_hist)
    if stop:
        print(f'\nLOOP B STOP: {reason}')
        break
    K = int(k_dec['k'])
    params = dict(FINER_2D_PARAMS)          # J112 params, verbatim -- and NO class2D_K in them
    # NOTE: no class2D_force_max, no prepare_window_*, no random_seed. See the markdown above.
    R = class2d_round(cs, proj, ws, 'finer2d', i + 1, cur_uid, cur_group, K, BOX_2D, 'loose',
                      extra_params=params, gpus=GPUS_MULTI, lane=LANE,
                      min_score=RANKER_MIN_SCORE)
    finer_class, finer_out, rk = R['class_job'], R['out_job'], R['rank_result']
    n_in, n_out = R['n_in'], R['n_out']
    frac = n_out / max(n_in, 1)
    n_cls, class_frac = R['n_classes_kept'], R['class_frac']
    finer_hist.append({'round': i + 1, 'K': K, 'mode': 'loose', 'class2d': finer_class.uid,
                       'select': finer_out.uid, 'n_sub': R['n_sub'], 'K_total': R['K_total'],
                       'n_in': n_in, 'n_out': n_out, 'frac': frac,
                       'n_classes_kept': n_cls, 'class_frac': class_frac, 'k_decision': k_dec})
    print(f'\nfiner 2D r{i + 1}: K={K} ({k_dec["source"]}/{k_dec["direction"]})  '
          f'{n_in:,} -> {n_out:,} kept ({frac * 100:.1f}%)  classes '
          f'{n_cls if n_cls is not None else "?"}/{R["K_total"]}'
          f'{"" if class_frac is None else " = %.3f" % class_frac}'
          f'{"" if R["n_sub"] == 1 else "  [%d concurrent subgroups]" % R["n_sub"]}\n'
          f'   [ref, COMPARISON ONLY: ONE round at K=300 on 1,407,603 -> 1,304,332 = 92.7%, '
          f'259/300 classes = 0.863]\n'
          f'   [TEST, COMPARISON ONLY: r1 K=200 loose 864,366 -> 827,457 = 95.7%, 184/200 = 0.920; '
          f'r2 K=300 tight (decide_k RAISE on junk_in_clean_class) -> 672,832 = 81.3%, 219/300 = 0.730]')
    cur_uid, cur_group = finer_out.uid, R['out_group']
    stop, reason, hard = stop_2d_loop('finer2d', finer_hist)
    if stop:
        print(f'\nLOOP B STOP: {reason}')
        break
    k_dec = decide_k(proj, 'finer2d', i + 2, finer_hist, K, class2d_job=finer_class,
                     rank_result=rk, box_px=BOX_2D, lane=LANE,
                     pool_note='This is the post-merge finer pass on the full merged+deduped pool; '
                               'the pool has not been enlarged since round 1.')
    i += 1

print('\nLOOP B K TRAJECTORY AS DECIDED: '
      + ' -> '.join(f'{h["K"]}({h["k_decision"]["source"]})' for h in finer_hist)
      + '   [ref, comparison only: one round at 300]')
CLEAN_JOB, CLEAN_GROUP = finer_out, cur_group
CLEAN_UID = CLEAN_JOB.uid
print(f'\nCLEAN POOL for 3D = {CLEAN_UID} ({CLEAN_GROUP}, {n_items(CLEAN_JOB, CLEAN_GROUP):,} particles)')

### STEP 6b (optional) — `reference_select_2D` cross-check

Run, **inspected, and NOT wired downstream** — exactly as in the reference (J594).

Its purpose is to show you the classes on which the two selections *disagree*: in the reference,
`reference_select_2D`'s default `sobel_max` rejected legitimate dumbbell / elongated side-view
classes that the class_ranker kept, and inspecting that disagreement is the entire basis for
preferring the loose ranker select.

**Honest caveat:** neither job contains orientation-aware logic. Rare views survive a loose cut only
because they *score low* — a side-effect of the threshold, not a criterion. Needs a reference volume
of the same molecule; skip if you have none.

In [ ]:
# STEP 6b (optional, diagnostic only)
REFSEL_VOLUME = None      # e.g. (job_uid, 'volume') of a refine of the SAME molecule; None = skip
if RUN_REFERENCE_SELECT_2D_CROSSCHECK and REFSEL_VOLUME:
    p = dict(REFSEL_PARAMS)
    p['window_inner_A'] = DIMS['refsel_window_inner_A']   # MUST be explicit: the live job fails
                                                          # without it even though MCP shows null
    rs = run_job('refsel_crosscheck', proj, ws, 'reference_select_2D',
                 {'particles': (finer_class.uid, 'particles'),
                  'templates': (finer_class.uid, 'class_averages'),
                  'volume': REFSEL_VOLUME},
                 p, 'reference_select_2D cross-check (DIAGNOSTIC ONLY, not wired downstream)',
                 gpus=GPUS_SINGLE)
    require_completed(rs)
    rg = resolve_out(rs, 'particles_selected', 'particles')
    print(f'reference_select_2D kept {n_items(rs, rg):,}   [ref 1,145,006 = 81.3%] vs the ranker '
          f'select {n_items(CLEAN_JOB, CLEAN_GROUP):,}   [ref 1,304,332 = 92.7%]')
    print('Now LOOK at the classes the two disagree on. Keep the elongated / side-view classes.')
    print(sig_block('which of the disputed classes are real views of the target'))
else:
    print('reference_select_2D cross-check skipped (no reference volume, or disabled).')

## STEP 7 — the PRE-3D GATE (hard gate)

Run on the job that **feeds the ab-initio**, not on one `class_2D`. Four checks: box/2D-mask vs the
measured particle density, whether the set is clean enough (against the extraction Nyquist), whether
there are enough particles (advisory, symmetry-aware), and whether the views are too narrow.
Verdicts: `HEALTHY` / `MINOR` / `MORE_2D_CLEANING` / `SERIOUS` / `ASK_HUMAN`. The last three
**raise** — this cell will not let you reconstruct from a set the gate would have stopped.

The gate was **vindicated** on the generalisation test: `HEALTHY`, no flags, on a set that went on
to 2.97 Å — and its box check *measured* the particle at 65.7–111.1 Å in a 245 Å box (2.21×), the
first time the data confirmed the derived box instead of arithmetic.

**[v2, finding #12] Per-branch tracing does not survive this topology, so the gate now also
intersects UIDs.** `trace_branches` walks class averages, and by the time LOOP B has run a *pooled*
`class_2D → select_2D` on the merged stack, per-picker provenance is structurally gone — it
reported **one** branch for the test's gate feed. `branch_survivorship()` intersects the feed
against each branch's final select and prints *share of feed* and *% of the branch surviving*.
On the test that answered the question directly (blob1 11.9 % at 35.5 %, blob2 19.9 % at 45.6 %,
topaz 68.2 % at 77.0 %) — the numbers that retired the second blob band and refuted "topaz was
under-cleaned".

In [ ]:
# STEP 7 — HARD GATE
ev, gate = pre_3d_gate(proj, CLEAN_JOB, CLEAN_GROUP, branches=BRANCHES)
print('\nGATE PASSED — proceeding to ab-initio.')

## STEP 8 — `homo_abinit` (K = 4, C1)

The mature low-SNR recipe, cross-validated in the reference against two same-specimen precedents:
`abinit_K 4`, `abinit_max_res 4` (**not** the coarse 8 of the abandoned early attempt),
`abinit_init_res 20`, `abinit_center False`, minisize 300 → 1000, `eps 0.005`.

Symmetry stays **C1**. If the symmetry is unknown, refine C1 first, confirm, and impose only if it
*improves* — and imposing symmetry needs a fresh ab-initio reseed, never locked C1 poses.

In [ ]:
# STEP 8
ab_params = dict(ABINIT_PARAMS)
ab_params['abinit_symmetry'] = SYMMETRY
ab_params['compute_use_ssd'] = COMPUTE_USE_SSD
abinit = run_job('abinit', proj, ws, 'homo_abinit',
                 {'particles': (CLEAN_JOB.uid, CLEAN_GROUP)},
                 ab_params, f'homo_abinit (K{ABINIT_PARAMS["abinit_K"]}, {SYMMETRY})',
                 gpus=GPUS_SINGLE, timeout_h=JOB_TIMEOUT_H)
require_completed(abinit)
print('\nlive output groups:', live_outputs(abinit))
print('[ref J596: class_0 351,748 | class_1 344,997 | class_2 217,136 | class_3 390,451 '
      '= a HARD partition of 1,304,332]')

## STEP 9 — DECISION: screen the ab-initio classes

> ### This is the cell that failed on the first generalisation test. Read it before you run it.
> The screen rejected the class that was the **largest** AND the **best on both count-robust
> metrics**, on a vision criterion it had **invented from the sample's name** — *transporter →
> channel → hollow*, so "accept ring + hollow channel, reject solid/filled core". On a
> micelle-embedded transporter that is **inverted**: the detergent belt *is* the torus and the
> ordered TM bundle is the *filled* density inside it. The whole 3D stage then ran on an ab-initio
> spherical-shell artifact and plateaued at **6.46 Å**, and the rejected class was additionally
> promoted to a junk-sink **decoy** that drained 103,807 genuine particles. Refined alone after the
> human's correction, that class gave **3.34 Å**, then **2.97 Å** un-binned.

Three defences, all [v2]:

1. **`PHYSICAL_SIGNATURE` (cell 1) is passed verbatim** into this call. It says what the target is
   made of, what a correct map shows, what the artifact looks like, and — the part that actually
   decides — **a feature that is not shared between them**. "Hollow vs filled" is shared; a
   detergent torus is hollow too. **Isotropy** is not: a genuine lumen is *directional*, an SGD
   shell is hollow from every angle in every projection.
2. **Vendored labelled controls** are attached (`controls/`): the reference project's own final map
   at 2.95 Å, the target *at ab-initio resolution*, and the artifact a screen once preferred. The
   model compares against a control instead of against its prior.
3. **METRIC/VISION DISAGREEMENT → refine BOTH** (`subC.md`, human directive 2026-08-23). If the
   metric-best and the vision-best differ, the notebook prints an unmissable banner and carries
   **both** into LOOP C. These refines cost minutes; an eyeball is not reliable; the FSC settles
   what the argument cannot.

**Primary decider remains the measurement.** `compare_volumes` scores every class at a **fixed
enclosed volume** on a **shared intensity window per view type** — in the reference that overturned
an eyeball ranking which was an artefact of per-panel normalisation flattering the higher-count
class. `homo_abinit` has **no gold-standard FSC**, so there is no number to rank on here; that is
why this is a real decision cell.

**And never seed a junk-sink decoy with a top-metric candidate** — check its metrics are consistent
with junk first.

In [ ]:
# STEP 9 — DECISION
rows, candidates, cmp_classes = screen_abinit_classes(proj, abinit)

## STEP 10 — LOOP C: the BRANCH-ALONE COMPARISON (keep **both** arms)

This is the heart of the recipe, and it is the one place where a "cleanup" would destroy the result.

* One `nonuniform_refine_new` per candidate class, refining that class **alone**, seeded with its
  own ab-initio volume.
* **Plus** one arm on the **merged** good classes (`remove_duplicate_particles` as the union tool —
  `particle_sets` has no union operation).

Reference: class_0 **alone** (351,748 particles) reported **3.64 Å**; the merged 0+1+3 pool
(1,087,189 particles) reported **6.54 Å**. A 3× smaller pool won, because merging visually similar
classes **decorrelated the half-maps** (the merged FSC fell 0.77 @ 7 Å → 0.02 @ 6 Å — not natural
decay). "Same conformation by eye" is not evidence that the classes are the same particles.

Both arms are kept. All arms use CryoSPARC's **default dynamic mask** so the comparison is
controlled — the focus mask comes later, and only in the local refine.

**[v2] One arm may exist because the screen disagreed with itself.** If STEP 9's count-robust
metrics and its vision read picked different classes, both are in `candidates` and both are refined
here — that is the `METRIC/VISION DISAGREEMENT` rule, and it is cheap insurance against the failure
that cost the generalisation test its whole first 3D stage (the metric-best class was rejected on an
inverted vision criterion; refined alone later it gave 3.34 Å against the artifact lineage's 6.46 Å
plateau).

*Un-validated extension, flagged honestly:* the reference only ever ran the class_0-alone arm and
the merged arm. Refining classes 1 and 3 alone is listed as **still open** in the reference's own
retrospective; this notebook runs them because they are cheap, but their numbers have no reference
to match.

In [ ]:
# STEP 10 — LOOP C
arms = []
for c in candidates:
    j = run_job(f'nu_alone_{c["label"]}', proj, ws, 'nonuniform_refine_new',
                {'particles': (abinit.uid, c['particles_group']),
                 'volume': (abinit.uid, c['volume_group'])},
                {'refine_symmetry': SYMMETRY,
                 'refine_res_init': NU_RES_INIT_BINNED,
                 'compute_use_ssd': COMPUTE_USE_SSD},
                f'NU-refine {c["label"]} branch ALONE ({c["n"]:,} ptcls)',
                gpus=GPUS_SINGLE)
    arms.append({'label': f'{c["label"]} ALONE', 'job': j, 'abinit_group': c['particles_group'],
                 'n': c['n']})

# ---- the MERGED arm (kept by design: this is what the branch-alone arms are held against) ----
if len(candidates) > 1:
    mg = run_job('merge_good_classes', proj, ws, 'remove_duplicate_particles',
                 {'particles': [(abinit.uid, c['particles_group']) for c in candidates]},
                 {},   # default min_dist_A; an ab-initio partition is hard, so ~0 duplicates expected
                 f'merge good abinit classes ({"+".join(c["label"] for c in candidates)})')
    require_completed(mg)
    mg_g = resolve_out(mg, 'particles_kept', 'particles')
    print(f'merged pool: {n_items(mg, mg_g):,}   [ref J1025: 1,087,189 kept / 7 rejected]')
    ref_vol = max(candidates, key=lambda c: c['n'])['volume_group']
    jm = run_job('nu_merged', proj, ws, 'nonuniform_refine_new',
                 {'particles': (mg.uid, mg_g), 'volume': (abinit.uid, ref_vol)},
                 {'refine_symmetry': SYMMETRY, 'refine_res_init': NU_RES_INIT_BINNED,
                  'compute_use_ssd': COMPUTE_USE_SSD},
                 'NU-refine MERGED pool (the arm the branch-alone arms are held against)',
                 gpus=GPUS_SINGLE)
    arms.append({'label': 'MERGED pool', 'job': jm, 'n': n_items(mg, mg_g)})

for a in arms:
    print(f'  arm {a["label"]:24s} {a["job"].uid}  ({a["n"]:,} particles)')
for a in arms:
    require_completed(a['job'], 'every arm must finish before STEP 11 can rank them')

## STEP 11 — DECISION: pick the winning arm (FSC-first)

Rank on **CryoSPARC's reported masked+corrected FSC** (`fsc_noisesub`). Arms closer than
**0.2 Å** are a **tie** — that is inside estimation noise and moves with mask tightness alone — and
are decided on count-robust real-space metrics instead. **Never** on cFAR: it is computed from the
assigned poses, so on an unconverged low-SNR map a low cFAR can mean alignment failure rather than
narrow orientation coverage. A verdict built on cFAR had to be retracted in the reference.

**[v2] The tie-break now has a second real-space axis: connectivity-sustained volume.**
`frac_in_largest` is measured at a fixed top-voxel occupancy, so two maps can tie there while
sustaining wildly different amounts of *connected* density. On the test's final adjudication that
number was decisive — 119k/68k Å³ against 23k/13k Å³ for a map with a 0.16 Å *better* reported FSC.
A map that cannot sustain its own protein volume at any readable level is the worse map.

In [ ]:
# STEP 11 — DECISION
best, ranked = pick_best_arm(proj, arms)
WINNER = best['job']
WINNER_RES = best['res_A']
print(f'\ncarrying forward: {WINNER.uid} at {WINNER_RES:.2f} A   [ref J1032 = 3.64 A, 351,748 ptcls]')

## STEP 12 — LOOP D: the un-bin gate, then the un-binned re-extract + refine

**The single largest lever of phase 2 in the reference: −0.58 Å.**

Un-bin only inside the regime where it pays: when the reported resolution is within ~**1.3×** of the
current binned Nyquist. The reference un-binned at 1.10× and gained; it correctly *deferred* at 2.0×,
where binning was demonstrably not the limiter (binning discards only information *above* the binned
Nyquist).

Then re-extract at the full box with `bin_size_pix` **omitted**, `update_location` /
`update_alignments2D` / `update_alignments3D` on, and remember the **`micrographs` input**. Seed the
un-binned NU-refine from the **binned** volume at `refine_res_init 10` — not from scratch.

Expect ~**1 %** particle loss to benign edge-of-frame exclusion (the reference measured 0.80 %,
verified as edge-driven and *not* the GPU-extract silent-truncation failure). The guard checks it.

**Written down before you run it:** the un-binned refine can report a *worse* number than the binned
one. That is a measurement correction (the binned box was flattering you), not a regression. The
reference saw exactly this once. Do not re-plan on it.

In [ ]:
# STEP 12 — LOOP D
if not unbin_gate(WINNER_RES, DIMS):
    raise RuntimeError('NOT in the un-bin regime yet. Go back and purify further in the cheap binned '
                     'box (more 2D rounds, a re-pick, or a different ab-initio class arm). '
                     'Un-binning here would be a waste of GPU. [v2] The generalisation test proved '
                     'this gate twice: it deferred at 2.11x, where un-binning ALONE cost +0.20 A, '
                     'and fired at 1.09x for -0.37 A. agentM overrode it once and was wrong — do '
                     'not override it here.')

WIN_P = resolve_out(WINNER, 'particles')
n_before = n_items(WINNER, WIN_P)
ux = run_job('unbin_extract', proj, ws, 'extract_micrographs_multi',
             {'micrographs': (ctf.uid, CTF_GROUP), 'particles': (WINNER.uid, WIN_P)},
             {'box_size_pix': DIMS['box_full_px'],       # bin_size_pix OMITTED == UNBINNED
              'update_location': True,
              'update_alignments2D': True,
              'update_alignments3D': True,
              'compute_num_gpus': 0},                    # CPU: GPU extract can silently truncate
             f'UNBINNED re-extract box {DIMS["box_full_px"]}')
require_completed(ux)
ux_g = resolve_out(ux, 'particles')
# RE-extract of already-centred particles -> the TIGHT tolerance, not the geometric first-extract
# prediction (finding #5). ref 0.80%, test 0.7-0.8%.
check_extract_count(ux, ux_g, n_before, first_extract=False)

unb = run_job('nu_unbinned', proj, ws, 'nonuniform_refine_new',
              {'particles': (ux.uid, ux_g), 'volume': (WINNER.uid, resolve_out(WINNER, 'volume'))},
              {'refine_symmetry': SYMMETRY,
               'refine_res_init': NU_RES_INIT_UNBINNED,   # seeded from the BINNED volume
               'compute_use_ssd': COMPUTE_USE_SSD},
              f'NU-refine UNBINNED box {DIMS["box_full_px"]}', gpus=GPUS_SINGLE)
require_completed(unb)
f_unb = reported_fsc(proj, unb.uid, 'unbinned')
GLOBAL_RES = (f_unb or {}).get('res_A')
print(f'\nun-bin: {WINNER_RES:.2f} A -> {GLOBAL_RES:.2f} A  (delta {GLOBAL_RES - WINNER_RES:+.2f} A)'
      f'   [ref 3.64 -> 3.06 A, delta -0.58; test 3.34 -> 2.97 A, delta -0.37]')
print(f'new (unbinned) Nyquist = {DIMS["nyquist_full_A"]:.3f} A. LOOP D EXIT: at bin 1 there is '
      f'nothing left to un-bin. If the reported resolution is still within '
      f'{UNBIN_TRIGGER_FACTOR}x of it ({UNBIN_TRIGGER_FACTOR * DIMS["nyquist_full_A"]:.2f} A) you '
      f'are sampling-limited and would need a larger box / finer pixel, not another re-extract.')
GLOBAL_MAP_JOB = unb

## STEP 13 — LOOP E: the texture-derived focused mask + focused local refinement

**Micelle-specific, and the reason `volume_tools` alone is not enough here.**

*No density threshold isolates the protein.* Swept at both 4.32 Å and 3.41 Å in the reference: at the
level enclosing ~63 kDa, 61 % of enclosed voxels lay beyond r = 30 Å and the region broke into 69
fragments. Protein and detergent **overlap in density magnitude** — they differ in **texture**
(local σ/mean: ordered protein 0.73–0.96, detergent 0.30–0.34, with the denominator floored at 10 %
of the map's p99.9 or the statistic tracks solvent zero-crossings and explodes).

*Find the radius from the radial profile of ordered voxels, not from a bound.* On the reference map:
the protein lobe peaks at r 25–30 Å, decays to a clean **minimum at 45–50 Å**, then a **separate
detergent bump** rises at 50–60 Å. r = 40 Å sits safely inside that minimum. **A mask that saturates
the radius bound means it is eating micelle — retune `--core-radius-A`, do not proceed.**

Then: `import_volumes` → `volume_tools` (`threshold` is **required** or the job fails "mask
max != 1"; `dilator 3`, `pad 5`) → `new_local_refine` with `refine_res_init ≈ 1.5 ×` the *current*
resolution. Do not copy an absolute 4 Å onto a worse map — that inverts the logic that made it work.

`pad` was swept 5 vs 15 in the reference: 2.95 vs 3.00 Å = a **tie**, question unresolved. `pad 5`
is the project default.

### [v2] Judge this loop on THREE axes, not on the masked FSC

On the generalisation test, a focused local refine **on top of an already-good un-binned map** was a
**negative** result — and it reported a *better* number: **2.81 Å vs 2.97 Å**. That 0.16 Å is inside
the < 0.2 Å tie rule, so the call went to real-space evidence, and the tight mask lost on every one
of them: `frac_in_largest` 0.427 vs 0.556, `mask_effect_A` 0.76 vs 0.49, and — decisively — a
**connectivity-knee sustained volume of 23k/13k Å³ against 119k/68k Å³**, i.e. 18–31 % of the
expected protein volume and **less than half of its own mask's footprint**, with 4–13× more
fragments per unit volume. A tighter mask can buy a better *masked* FSC by shrinking the comparison
window over noisier, less-uniformly-supported density.

`vis_res.assess()` returned **CONSISTENT** for that map. Legibility-at-a-point and volume coverage
are **different axes**; the standard check only tests the first. STEP 14 runs
`compare_connectivity()` on both maps for exactly this reason.

LOOP E's own stop condition was **vindicated**: a second round on a rebuilt, looser mask was worse,
and the loop stopped.

In [ ]:
# STEP 13a — build the mask OUTSIDE CryoSPARC (texture-derived), then sanity-check it
import subprocess

src_map = SJ.final_volume_map(proj, GLOBAL_MAP_JOB.uid, sharp=False)
mask_hard = WORK_DIR / f'focus_mask_hard_{GLOBAL_MAP_JOB.uid}.mrc'
cmd = [sys.executable, str(PROFILE_DIR / 'build_model_mask' / 'build_focus_mask.py'), str(src_map),
       '--out', str(mask_hard),
       '--core-radius-A', str(FOCUS_CORE_RADIUS_A),
       '--expect-kda', str(EXPECTED_CORE_MASS_KDA)]
print('$', ' '.join(cmd))
r = subprocess.run(cmd, capture_output=True, text=True)
print(r.stdout[-6000:])
if r.returncode != 0:
    print(r.stderr[-4000:])
    raise RuntimeError('build_focus_mask.py failed — do NOT fall back to a density threshold; '
                       'it does not work on a micelle-embedded particle.')
print(f'\nMASK SANITY (read the numbers above): ONE coherent body, texture ~0.7-0.9, implied mass '
      f'near {EXPECTED_CORE_MASS_KDA:g} kDa, and the radius NOT saturated at '
      f'{FOCUS_CORE_RADIUS_A:g} A. If it saturates the bound, or comes out far above the expected '
      f'mass, STOP and retune --core-radius-A from the radial profile.')

In [ ]:
# STEP 13b — import the mask, soften it, and run the focused local refine
imv = run_job('mask_import', proj, ws, 'import_volumes', {},
              {'volume_blob_path': str(mask_hard),     # ENV: this run's mask
               'volume_psize': DIMS['psize_A'],
               'volume_out_name': 'mask'},
              'import hard core mask')
imv_g = resolve_out(imv, 'imported_mask_1', 'imported_volume_1', 'mask')  # MCP name is stale here

vt = run_job('mask_soften', proj, ws, 'volume_tools',
             {'mask': (imv.uid, imv_g)},
             {'volume_in_type': 'mask', 'volume_out_type': 'mask',
              'threshold': FOCUS_MASK_THRESHOLD,       # REQUIRED or the job fails "mask max != 1"
              'dilator': FOCUS_MASK_DILATOR, 'pad': FOCUS_MASK_PAD},
             f'focus mask dil{FOCUS_MASK_DILATOR}/pad{FOCUS_MASK_PAD}')
vt_g = resolve_out(vt, 'mask', 'volume')                # MCP name is stale here too

local_res_init = round(LOCAL_RES_INIT_FACTOR * GLOBAL_RES, 1)
print(f'refine_res_init = {LOCAL_RES_INIT_FACTOR} x {GLOBAL_RES:.2f} A = {local_res_init} A   '
      f'[ref 1.5 x 3.06 = 4.59, used 4.5]')

loc = run_job('local_refine', proj, ws, 'new_local_refine',
              {'particles': (GLOBAL_MAP_JOB.uid, resolve_out(GLOBAL_MAP_JOB, 'particles')),
               'volume': (GLOBAL_MAP_JOB.uid, resolve_out(GLOBAL_MAP_JOB, 'volume')),
               'mask': (vt.uid, vt_g)},
              {'refine_symmetry': SYMMETRY,
               'refine_res_init': local_res_init,
               'init_r_extent': LOCAL_R_EXTENT,
               'init_s_extent': LOCAL_S_EXTENT,
               'use_alignment_prior': LOCAL_USE_ALIGNMENT_PRIOR,   # prior SIGMAS stay at default
               'compute_use_ssd': COMPUTE_USE_SSD},
              f'focused local refine pad{FOCUS_MASK_PAD:02d}', gpus=GPUS_SINGLE)

require_completed(loc)
f_loc = reported_fsc(proj, loc.uid, 'focused local refine')
LOCAL_RES = (f_loc or {}).get('res_A')
print(f'\nfocused local refine: {GLOBAL_RES:.2f} -> {LOCAL_RES:.2f} A '
      f'(delta {LOCAL_RES - GLOBAL_RES:+.2f} A)   [ref 3.06 -> 2.95, delta -0.11]')
print('NOTE: a delta this small is INSIDE FSC noise. The honest claim is "modestly better core '
      'map", carried by the real-space metrics (ref n_frag 41 -> 32, frac_in_largest 0.914 -> '
      '0.974) and the surface read, not by the 0.11 A. Out-of-mask density is degraded BY DESIGN.')
print('LOOP E: iterate (tighten the mask, re-run) only while a round improves ALL THREE of the '
      'reported FSC, n_frag/frac_in_largest, AND the connectivity-sustained volume (STEP 14). '
      '[v2] On the generalisation test this step REPORTED a better FSC (2.81 vs 2.97 A) while '
      'sustaining only 18-31% of the expected protein volume — the mask had shrunk the comparison '
      'window, not improved the map, and plain vis_res.assess() called it CONSISTENT. A local '
      'refine on top of an already-good un-binned map is not automatically an improvement.')

## STEP 14 — validate the resolution before reporting it (`vis_res` **+ volume coverage**)

The reported masked+corrected FSC is the **first** reference; `vis_res` is the **lie-detector**, not
a replacement and not a second FSC. Slices are read at native sampling for secondary structure; the
paired surface is the second look, because flat slices under-read a 3–4 Å map by ~1–1.5 Å.

If `assess()` returns `NEEDS_SURFACE_CHECK` it is refusing to guess — no resolution is reported
until the surface resolves it, and a verdict carrying `SURFACE_UNAVAILABLE` is explicitly
**unconfirmed**.

**[v2] Two additions, both from the test's final adjudication.**

* **Render size is not cosmetic.** `surface_judge`'s own default (900 px) gave **5.9 px/Å** on the
  test deliverable — below the ~10 px/Å legibility floor, where a side-chain knob and a bump are the
  same handful of pixels. `SURFACE_RENDER_SIZE` is now **1600** and the measured `px_per_A` is
  asserted against `SURFACE_MIN_PX_PER_A`. At 10.4 px/Å the same map read `CONSISTENT` (Δ 0.33).
* **Volume coverage is a different axis from legibility.** `assess()` returned `CONSISTENT` for a
  map (2.81 Å reported) that sustained only **23k/13k Å³** at the connectivity knee against the
  deliverable's **119k/68k Å³** — 18–31 % of the expected protein volume, less than half of its own
  mask footprint, 4–13× more fragments per unit volume. `connectivity_volume()` measures this
  numerically (no render), and `compare_connectivity()` runs it as an **A/B on every pair of
  refines you are choosing between**. It is what caught the run's one serious late error.

In [ ]:
# STEP 14 — DECISION
print('=== DELIVERABLE CANDIDATE: focused local refine ===')
v_local = vis_res_check(proj, loc.uid, mass_kDa=EXPECTED_CORE_MASS_KDA)
print('\n=== GLOBAL CONSENSUS MAP (the right parent for any further global work) ===')
v_global = vis_res_check(proj, GLOBAL_MAP_JOB.uid, mass_kDa=EXPECTED_CORE_MASS_KDA)

# ---- [v2] A/B THE TWO ON VOLUME COVERAGE, then decide (finding #14) ----------
# The local refine reports a masked FSC over a SMALLER window. If it wins by less than FSC_TIE_A,
# or wins at all while sustaining materially less connected volume, the honest deliverable is the
# global map. This is exactly the call the test got right only after a human asked for it.
print('\n=== WHICH MAP IS THE DELIVERABLE? ===')
pairs = [('local refine %s' % loc.uid, str(SJ.final_volume_map(proj, loc.uid, sharp=True))),
         ('global      %s' % GLOBAL_MAP_JOB.uid,
          str(SJ.final_volume_map(proj, GLOBAL_MAP_JOB.uid, sharp=True)))]
conn_rank = compare_connectivity(pairs)
d = (GLOBAL_RES - LOCAL_RES) if (GLOBAL_RES and LOCAL_RES) else None
print(f'\nreported FSC: local {LOCAL_RES:.2f} A vs global {GLOBAL_RES:.2f} A '
      f'(local is {abs(d):.2f} A {"better" if d and d > 0 else "worse"})'
      if d is not None else '\nreported FSC unavailable for one of the two')
if d is not None and abs(d) < FSC_TIE_A:
    print(f'  -> INSIDE the {FSC_TIE_A} A tie rule. Decide on real-space evidence, NOT on the FSC:')
    print('     * frac_in_largest / n_frag from compare_volumes')
    print('     * connectivity-sustained volume above (a map that does not sustain its own protein '
          'volume at any readable level is the worse map)')
    print('     * mask_effect_A (a big gap on a focused refine means the mask did the work)')
    if conn_rank:
        print(f'     -> volume coverage prefers: {conn_rank[0]["label"]}')
print('\n[TEST, for comparison: local J1530 2.81 A / 23k A^3 sustained / frac_in_largest 0.427 was '
      'REJECTED in favour of global J1525 2.97 A / 119k A^3 / 0.556 — and plain assess() had '
      'called J1530 CONSISTENT.]')

## STEP 15 (optional) — `orientation_diagnostics`

Kept only as a **diagnostic**, never as a gate, and carrying the reference's own retraction:

> cFAR is computed from the **assigned poses**. On a small low-SNR particle aligned only to ~6.5 Å
> those poses are unreliable, so a low cFAR can reflect **alignment failure** rather than genuinely
> narrow orientation coverage — and a high cFAR is not proof of isotropy either. Pose-derived
> statistics cannot adjudicate an unconverged map.

In the reference, cFAR 0.341 → 0.61 bought only 6.74 → 6.54 Å, and the `rebalance_3D` that cFAR
motivated made the map **worse** (6.54 → 6.75 Å) on 4 % fewer particles.

In [ ]:
# STEP 15 (optional)
RUN_ORIENT_DIAG = False
if RUN_ORIENT_DIAG:
    od = run_job('orient_diag', proj, ws, 'orientation_diagnostics',
                 {'volume': (GLOBAL_MAP_JOB.uid, resolve_out(GLOBAL_MAP_JOB, 'volume')),
                  'particles': (GLOBAL_MAP_JOB.uid, resolve_out(GLOBAL_MAP_JOB, 'particles'))},
                 {}, 'orientation_diagnostics (DIAGNOSTIC ONLY — cFAR is not a gate here)',
                 gpus=GPUS_SINGLE)
    print('Read cFAR as a DESCRIPTION, not a verdict. Do NOT run rebalance_3D on the strength of '
          'it alone: in the reference that cost 4% of the particles and 0.21 A.')
else:
    print('orientation_diagnostics skipped (optional).')

## STEP 16 — summary: the funnel and the resolution ladder, against the reference

Compare your numbers with the reference's. Large deviations in the **funnel** (especially the dedup
overlap and the round-1 keep fractions) are the earliest signal that something is off.

In [ ]:
# STEP 16 — summary
print('PARTICLE FUNNEL                    this run        reference (zll_260130)   TEST (P5/W23)')
print(f'  raw picks (2 branches)        {n_items(pk1, "particles") + n_tx:>14,}   12,597,780 (3 br)   9,003,955 (3 br)')
print(f'  after per-branch 2D loops     {n_sum:>14,}    2,407,011          1,114,303')
print(f'  after merge + dedup           {n_merge:>14,}    1,407,630 (41.5%)    864,366 (22.4%)')
print(f'  after finer high-K 2D+select  {n_items(CLEAN_JOB, CLEAN_GROUP):>14,}    1,304,332 (92.7%)    672,832')
print(f'  after taking ONE abinit class {best["n"]:>14,}      351,748            236,810')
print(f'  after unbinned re-extract     {n_items(ux, ux_g):>14,}      348,942            234,925')
print()
print('RESOLUTION LADDER (REPORTED masked+corrected FSC)  this run   reference   TEST')
print(f'  branch-alone NU-refine, binned                 {WINNER_RES:>8.2f} A   3.64 A      3.34 A')
print(f'  unbinned re-extract + NU-refine                {GLOBAL_RES:>8.2f} A   3.06 A      2.97 A (-0.37)')
print(f'  focused local refine (dil{FOCUS_MASK_DILATOR}/pad{FOCUS_MASK_PAD:02d})              {LOCAL_RES:>8.2f} A   2.95 A      2.81 A (REJECTED*)')
print('  * the test\'s local refine reported BETTER and was rejected on volume coverage — see STEP 14')
print()
print('class2D_K AS DECIDED AT RUN TIME  (never a ladder — K_DECISION_SPEC.md)')
print(f'{"loop":10s} {"rounds":>6s}  K trajectory (source/direction, trigger)')
for _h in [b1, b3, {'tag': 'finer2d', 'history': finer_hist}]:
    _traj = '  ->  '.join(
        f'{r["K"]} [{r["k_decision"]["source"]}/{r["k_decision"]["direction"]}'
        + ('' if r['k_decision'].get('trigger') in (None, 'none_observed', 'pool_origin_band')
           else ', ' + str(r['k_decision']['trigger'])) + ']'
        + ('' if r.get('n_sub', 1) == 1 else f' x{r["n_sub"]} subgroups')
        for r in _h['history'])
    print(f'  {_h["tag"]:8s} {len(_h["history"]):>6d}  {_traj}')
print('  reference (COMPARISON ONLY, never an input): blob1/blob2/topaz 100 -> 75 -> 50 over 3')
print('  rounds each; finer post-merge one round at K=300 on 1.41M. That 100->75->50 sequence has')
print('  0 occurrences in 74,623 history-bank lineage triples, and K=75 appears in 13 of 20,029')
print('  jobs and never in a descending chain — which is why it is not a schedule here.')
print('  TEST (COMPARISON ONLY): blob branches held K=300 for 5 rounds (one lowered to 200 at r5);')
print('  finer pass 200 -> 300 (a decide_k RAISE on junk_in_clean_class).')
_fb = [r for _h in [b1, b3, {'history': finer_hist}] for r in _h['history']
       if r['k_decision']['source'] in ('fallback', 'llm-refused')]
print(f'  K decisions taken by the deterministic FALLBACK or REFUSED by the rule: {len(_fb)} '
      f'(each one HELD K and printed why; the fallback never descends)')
print()
print(f'  topaz particle_threshold as DECIDED: {TOPAZ_DECISION.get("particle_threshold")} '
      f'(raw log-odds axis)  [ref: default 0 -> 190/mic; TEST: 1.843 -> 144.5/mic]')
print(f'  vis_res on the local refine : {v_local.get("status")}  flags={v_local.get("flags")}')
print(f'  vis_res on the global map   : {v_global.get("status")}  flags={v_global.get("flags")}')
for _v, _n in ((v_local, 'local'), (v_global, 'global')):
    _c = (_v or {}).get('connectivity')
    if _c:
        print(f'  volume coverage {_n:6s}      : {_c["rows"][0]["enclosed_A3"] / 1000:.0f}k A^3 = '
              f'{_c["fuller_frac_of_expected"]:.2f}x expected'
              + ('   <- FLAGGED' if _c.get('flagged') else ''))
print()
print('Report the REPORTED masked+corrected FSC, validated by vis_res AND corroborated on volume')
print('coverage. Never a hand-rolled mask-free FSC, never a sub-0.2 A difference, never cFAR.')
print(f'\nall job UIDs: {json.dumps(load_state(), indent=2)}')